# Cluster Composition Data

### Composition File

In [8]:
import os
import re
import argparse
import pandas as pd

# =========================
# CLI
# =========================
parser = argparse.ArgumentParser(
    description=("Collect cluster compositions and split outputs by Total Cells > 0, "
                 "handling integer/decimal parameter encodings in paths, and standardizing IDs.")
)
parser.add_argument("--base_dir", default="../../Results/output",
                    help="Root folder containing scan_* directories.")
parser.add_argument("--out_csv_base", default="../../Results/Data/Clusters/ClusterComposition",
                    help="Base path for outputs (no extension).")
parser.add_argument("--mode", choices=["all", "clusters", "both"], default="both",
                    help="What to save: 'all' (incl. zeros), 'clusters' (Total Cells > 0 only), or 'both'.")
args, _ = parser.parse_known_args()

base_dir = args.base_dir
out_base = args.out_csv_base
out_all_csv = f"{out_base}_All.csv"
out_clusters_csv = f"{out_base}_1.csv"
os.makedirs(os.path.dirname(out_all_csv), exist_ok=True)

# =========================
# Final, standardized column order (with new names)
# =========================
FINAL_COLUMNS = [
    "MCS", "Jlf", "mu", "PP", "rep",
    "Cluster UID",         # persistent across time (was 'Cluster ID' in source)
    "Cluster ID",          # per-frame rank (was 'ClusterID_frame' in source, or computed here)
    "Leader Cells", "Follower Cells", "Total Cells",
    "Centroid_X", "Centroid_Y",
    "Member_Leader_X", "Member_Leader_Y",
    "Member_Follower_X", "Member_Follower_Y"
]

# =========================
# Defaults for missing columns
# =========================
DEFAULTS = {
    "MCS": 0,
    "Jlf": 0.0,
    "mu": 0.0,
    "PP": 0.0,
    "rep": 0,
    "Cluster UID": 0,
    "Cluster ID": 0,
    "Leader Cells": 0,
    "Follower Cells": 0,
    "Total Cells": 0,
    "Centroid_X": 0.0,
    "Centroid_Y": 0.0,
    "Member_Leader_X": 0.0,
    "Member_Leader_Y": 0.0,
    "Member_Follower_X": 0.0,
    "Member_Follower_Y": 0.0,
}

def default_row(mcs, Jlf, mu, PP, rep):
    """One placeholder row with final standardized schema."""
    row = DEFAULTS.copy()
    row.update({
        "MCS": int(mcs),
        "Jlf": float(Jlf),
        "mu": float(mu),
        "PP": float(PP),
        "rep": int(rep),
    })
    return row

# =========================
# Helpers: tolerate -5 vs -5.0
# =========================
def canon_num(s: str) -> str:
    """Return canonical numeric string (e.g., '-5.0' -> '-5', '0.50' -> '0.5')."""
    try:
        return f"{float(s):g}"
    except Exception:
        return s

def find_position_folder(scan_path: str, Jlf: str, mu: str, PP: str) -> str:
    """
    Look inside scan_path for a folder named PositionData_* that numerically matches (Jlf, mu, PP),
    regardless of integer/decimal encoding. Returns matching folder or a literal fallback.
    """
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    for d in os.listdir(scan_path):
        if not d.startswith("PositionData_"):
            continue
        parts = d.split("_")
        if len(parts) >= 4:
            have = (canon_num(parts[1]), canon_num(parts[2]), canon_num(parts[3]))
            if have == want:
                return d
    return f"PositionData_{Jlf}_{mu}_{PP}"

def find_cluster_file(mcs_path: str, Jlf: str, mu: str, PP: str, mcs_val: int) -> str:
    """
    Look inside mcs_path for ClusterComposition_*_MCS_<mcs>.csv
    that numerically matches (Jlf, mu, PP). Returns full path or "" if not found.
    """
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    target_suffix = f"_MCS_{mcs_val}.csv"

    try:
        for fname in os.listdir(mcs_path):
            if not (fname.startswith("ClusterComposition_") and fname.endswith(target_suffix)):
                continue
            stem = fname[:-4] if fname.lower().endswith(".csv") else fname
            parts = stem.split("_")
            # Expected: ClusterComposition, Jlf, mu, PP, MCS, mcs
            if len(parts) >= 6:
                have = (canon_num(parts[1]), canon_num(parts[2]), canon_num(parts[3]))
                if have == want:
                    return os.path.join(mcs_path, fname)
    except FileNotFoundError:
        pass

    return ""

# =========================
# Regex for folder parsing
# =========================
folder_pattern = re.compile(r"scan_(\d+)_Jlf_([-.\d]+)_mu_([-.\d]+)_PP_([-.\d]+)_rep_(\d+)")
mcs_folder_pattern = re.compile(r"MCS_(\d+)")

# =========================
# Per-file loader: standardize columns and rename IDs
# =========================
def load_and_standardize(filepath: str, mcs_val: int, Jlf: str, mu: str, PP: str, rep: str) -> pd.DataFrame:
    """
    Read a single per-MCS ClusterComposition CSV and return a DataFrame with FINAL_COLUMNS.
    - Renames 'Cluster ID' -> 'Cluster UID'
    - Renames 'ClusterID_frame' -> 'Cluster ID'
    - If per-frame ID missing, computes it by sorting (Centroid_X, Centroid_Y) and assigning 1..N.
    - Adds/ensures MCS, Jlf, mu, PP, rep.
    """
    try:
        raw = pd.read_csv(filepath)
    except Exception as e:
        print(f"❌ Error reading {filepath}: {e}")
        return pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)])

    # If file is empty, return a default row
    if raw.empty:
        return pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)])

    df = raw.copy()

    # 1) Rename IDs to standardized names if present
    #    - persistent: 'Cluster ID' (source) -> 'Cluster UID' (final)
    #    - frame: 'ClusterID_frame' (source) -> 'Cluster ID' (final)
    rename_map = {}
    if "Cluster ID" in df.columns:
        rename_map["Cluster ID"] = "Cluster UID"
    if "ClusterID_frame" in df.columns:
        rename_map["ClusterID_frame"] = "Cluster ID"
    if rename_map:
        df = df.rename(columns=rename_map)

    # 2) Ensure per-frame 'Cluster ID' exists; if not, compute from centroids
    if "Cluster ID" not in df.columns:
        # If no centroid columns, just put zeros
        if "Centroid_X" in df.columns and "Centroid_Y" in df.columns:
            # assign per-frame rank left->right (then bottom->top)
            # We want 1..N within THIS FILE (which is a single MCS)
            df = df.sort_values(["Centroid_X", "Centroid_Y"], kind="mergesort").reset_index(drop=True)
            df["Cluster ID"] = range(1, len(df) + 1)
            # restore original order if needed (not necessary; we keep this order)
        else:
            df["Cluster ID"] = 0

    # 3) Ensure persistent 'Cluster UID' exists; if missing, fill 0 (older runs)
    if "Cluster UID" not in df.columns:
        df["Cluster UID"] = 0

    # 4) Add parameter columns
    df["MCS"] = int(mcs_val)
    df["Jlf"] = float(Jlf)
    df["mu"] = float(mu)
    df["PP"] = float(PP)
    df["rep"] = int(rep)

    # 5) Ensure all final columns exist; fill missing with defaults
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = DEFAULTS[col]

    # 6) Reorder columns
    df = df[FINAL_COLUMNS]

    return df

# =========================
# Collect rows
# =========================
all_rows = []

if not os.path.isdir(base_dir):
    print(f"❌ Base directory not found: {base_dir}")
else:
    for scan_folder in os.listdir(base_dir):
        scan_path = os.path.join(base_dir, scan_folder)
        if not os.path.isdir(scan_path):
            continue

        folder_match = folder_pattern.match(scan_folder)
        if not folder_match:
            continue

        scan_idx, Jlf, mu, PP, rep = folder_match.groups()

        # Tolerant PositionData folder
        position_data_folder = find_position_folder(scan_path, Jlf, mu, PP)
        position_path = os.path.join(scan_path, position_data_folder)

        if not os.path.isdir(position_path):
            print(f"❌ Missing PositionData folder (tolerant+fallback): {position_data_folder}")
            continue

        for mcs_folder in os.listdir(position_path):
            mcs_match = mcs_folder_pattern.match(mcs_folder)
            if not mcs_match:
                continue

            mcs_val = int(mcs_match.group(1))
            mcs_path = os.path.join(position_path, mcs_folder)
            if not os.path.isdir(mcs_path):
                continue

            tolerant_path = find_cluster_file(mcs_path, Jlf, mu, PP, mcs_val)
            if tolerant_path:
                filepath = tolerant_path
            else:
                filename_fallback = f"ClusterComposition_{Jlf}_{mu}_{PP}_MCS_{mcs_val}.csv"
                filepath = os.path.join(mcs_path, filename_fallback)

            if os.path.isfile(filepath):
                df_std = load_and_standardize(filepath, mcs_val, Jlf, mu, PP, rep)
                all_rows.append(df_std)
            else:
                print(f"⚠️ File missing: {filepath}")
                all_rows.append(pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)]))

# =========================
# Combine, sort, and save
# =========================
if not all_rows:
    print("\n❗No cluster composition files were processed.")
else:
    combined_df = pd.concat(all_rows, ignore_index=True)

    if combined_df.empty:
        print("\n❗Cluster files were collected, but all were empty.")
    else:
        # Sort: params → time → persistent UID → per-frame ID
        sort_cols = ["Jlf", "mu", "PP", "rep", "MCS", "Cluster UID", "Cluster ID"]
        combined_df.sort_values(by=sort_cols, inplace=True, kind="mergesort")

        # Save according to mode
        if args.mode in ("all", "both"):
            combined_df.to_csv(out_all_csv, index=False)
            print(f"\n✅ Saved ALL rows (incl. zero-cluster placeholders) to:\n{out_all_csv}")
            print(f"• Rows: {len(combined_df)} | "
                  f"Unique sims: {combined_df[['Jlf','mu','PP','rep']].drop_duplicates().shape[0]}")

        if args.mode in ("clusters", "both"):
            df_clusters_only = combined_df[combined_df["Total Cells"] > 0].copy()
            df_clusters_only.to_csv(out_clusters_csv, index=False)
            print(f"\n✅ Saved CLUSTERS-ONLY rows (Total Cells > 0) to:\n{out_clusters_csv}")
            print(f"• Rows: {len(df_clusters_only)} | "
                  f"Unique sims: {df_clusters_only[['Jlf','mu','PP','rep']].drop_duplicates().shape[0]}")

        if args.mode not in ("all", "clusters", "both"):
            print("\n⚠️ No output saved: unrecognized mode (should not happen).")

print("\n🎉 Done.")



✅ Saved ALL rows (incl. zero-cluster placeholders) to:
../../Results/Data/Clusters/ClusterComposition_All.csv
• Rows: 1141646 | Unique sims: 13310

✅ Saved CLUSTERS-ONLY rows (Total Cells > 0) to:
../../Results/Data/Clusters/ClusterComposition_1.csv
• Rows: 295738 | Unique sims: 4046

🎉 Done.


### Cluster Event File (Merging/Splitting/Dissolution)

In [9]:
import os
import re
import argparse
import pandas as pd

# =========================
# CLI
# =========================
parser = argparse.ArgumentParser(
    description=("Collect cluster event logs (merge/split/dissolve) across all scan_* "
                 "runs and save a single combined CSV keyed by (Jlf, mu, PP, rep).")
)
parser.add_argument("--base_dir", default="../../Results/output",
                    help="Root folder containing scan_* directories.")
parser.add_argument("--out_csv", default="../../Results/Data/Clusters/ClusterEvents_All.csv",
                    help="Full path to the combined output CSV.")
parser.add_argument("--include_empty", action="store_true",
                    help="If set, write a placeholder row for scans with no ClusterEvents file.")
args, _ = parser.parse_known_args()

base_dir = args.base_dir
out_csv  = args.out_csv
os.makedirs(os.path.dirname(out_csv), exist_ok=True)

# =========================
# Patterns
# =========================
# e.g. scan_0_Jlf_-5_mu_0_PP_0_rep_0
SCAN_RE = re.compile(r"^scan_(\d+)_Jlf_([-.\d]+)_mu_([-.\d]+)_PP_([-.\d]+)_rep_(\d+)$")
# Files like: ClusterEvents_-5.0_0.0_0.0.csv  OR ClusterEvents_-5_0_0.csv
EVENT_FILE_RE = re.compile(r"^ClusterEvents_([-.\d]+)_([-.\d]+)_([-.\d]+)\.csv$")

def canon_num(s: str) -> str:
    """Canonical numeric string (so '-5.0' == '-5')."""
    try:
        return f"{float(s):g}"
    except Exception:
        return s

def find_events_file(scan_path: str, Jlf: str, mu: str, PP: str) -> str:
    """
    In a scan_* folder, find ClusterEvents_* file whose numeric parts match (Jlf, mu, PP),
    regardless of integer/decimal formatting.
    """
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    try:
        for fname in os.listdir(scan_path):
            m = EVENT_FILE_RE.match(fname)
            if not m:
                continue
            have = (canon_num(m.group(1)), canon_num(m.group(2)), canon_num(m.group(3)))
            if have == want:
                return os.path.join(scan_path, fname)
    except FileNotFoundError:
        pass
    # Fallback: try exact string (may still succeed if folder uses same format)
    fallback = os.path.join(scan_path, f"ClusterEvents_{Jlf}_{mu}_{PP}.csv")
    return fallback if os.path.isfile(fallback) else ""

# Standard final column order (we keep Parents/Children/OverlapFractions as-is)
FINAL_COLUMNS = [
    "Jlf", "mu", "PP", "rep", "MCS", "Event",
    "Parents", "Children", "OverlapFractions",
    "LostToSingletons", "LostToMainTumor"
]

def default_empty_row(Jlf, mu, PP, rep):
    return {
        "Jlf": float(Jlf),
        "mu": float(mu),
        "PP": float(PP),
        "rep": int(rep),
        "MCS": -1,
        "Event": "none",
        "Parents": "[]",
        "Children": "[]",
        "OverlapFractions": "[]",
        "LostToSingletons": 0.0,
        "LostToMainTumor": 0.0
    }

rows = []

if not os.path.isdir(base_dir):
    print(f"❌ Base directory not found: {base_dir}")
else:
    for scan_folder in os.listdir(base_dir):
        scan_path = os.path.join(base_dir, scan_folder)
        if not os.path.isdir(scan_path):
            continue

        mm = SCAN_RE.match(scan_folder)
        if not mm:
            continue

        scan_idx, Jlf, mu, PP, rep = mm.groups()

        events_path = find_events_file(scan_path, Jlf, mu, PP)
        if not events_path or not os.path.isfile(events_path):
            if args.include_empty:
                rows.append(pd.DataFrame([default_empty_row(Jlf, mu, PP, rep)]))
            else:
                print(f"⚠️ No ClusterEvents file for {scan_folder}")
            continue

        try:
            df = pd.read_csv(events_path)
        except Exception as e:
            print(f"❌ Error reading {events_path}: {e}")
            if args.include_empty:
                rows.append(pd.DataFrame([default_empty_row(Jlf, mu, PP, rep)]))
            continue

        if df.empty:
            if args.include_empty:
                rows.append(pd.DataFrame([default_empty_row(Jlf, mu, PP, rep)]))
            continue

        # Tag parameters
        df = df.copy()
        df["Jlf"] = float(Jlf)
        df["mu"] = float(mu)
        df["PP"] = float(PP)
        df["rep"] = int(rep)

        # Ensure missing columns are present (older runs might omit some fields)
        for col in FINAL_COLUMNS:
            if col not in df.columns:
                # Provide sensible defaults
                if col in ("Parents", "Children", "OverlapFractions"):
                    df[col] = "[]"
                elif col in ("LostToSingletons", "LostToMainTumor"):
                    df[col] = 0.0
                elif col == "MCS":
                    df[col] = -1
                elif col == "Event":
                    df[col] = "unknown"

        df = df[FINAL_COLUMNS]
        rows.append(df)

# Save
if not rows:
    print("\n❗No ClusterEvents files were processed.")
else:
    out_df = pd.concat(rows, ignore_index=True)
    if out_df.empty:
        print("\n❗ClusterEvents were collected, but all were empty.")
    else:
        out_df.sort_values(
            by=["Jlf", "mu", "PP", "rep", "MCS", "Event"],
            inplace=True,
            kind="mergesort"
        )
        out_df.to_csv(out_csv, index=False)
        print(f"\n✅ Saved combined ClusterEvents to:\n{out_csv}")
        print(f"• Rows: {len(out_df)} | "
              f"Unique sims: {out_df[['Jlf','mu','PP','rep']].drop_duplicates().shape[0]}")



✅ Saved combined ClusterEvents to:
../../Results/Data/Clusters/ClusterEvents_All.csv
• Rows: 8751 | Unique sims: 2555


In [17]:
import os
import re
import argparse
import pandas as pd
import numpy as np
from ast import literal_eval
from itertools import zip_longest

# =========================
# CLI
# =========================
parser = argparse.ArgumentParser(
    description=("Collect cluster compositions and split outputs by Total Cells > 0, "
                 "handling integer/decimal parameter encodings in paths, "
                 "standardizing IDs, and annotating clusters with event info.")
)
parser.add_argument("--base_dir", default="../../Results/output",
                    help="Root folder containing scan_* directories.")
parser.add_argument("--out_csv_base", default="../../Results/Data/Clusters/ClusterComposition",
                    help="Base path for composition outputs (no extension).")
parser.add_argument("--mode", choices=["all", "clusters", "both"], default="both",
                    help="What to save: 'all' (incl. zeros), 'clusters' (Total Cells > 0 only), or 'both'.")
args, _ = parser.parse_known_args()

base_dir = args.base_dir
out_base = args.out_csv_base
out_all_csv = f"{out_base}_All.csv"
out_clusters_csv = f"{out_base}_1.csv"
out_all_enriched_csv = f"{out_base}_All_Enriched.csv"
out_clusters_enriched_csv = f"{out_base}_1_Enriched.csv"

# events output (parallel to compositions)
events_out_csv = os.path.join(os.path.dirname(out_all_csv), "ClusterEvents_All.csv")

os.makedirs(os.path.dirname(out_all_csv), exist_ok=True)

# =========================
# Final, standardized composition columns (with new names)
# =========================
FINAL_COLUMNS = [
    "MCS", "Jlf", "mu", "PP", "rep",
    "Cluster UID",         # persistent across time (was 'Cluster ID' in source)
    "Cluster ID",          # per-frame rank (was 'ClusterID_frame' in source, or computed here)
    "Leader Cells", "Follower Cells", "Total Cells",
    "Centroid_X", "Centroid_Y",
    "Member_Leader_X", "Member_Leader_Y",
    "Member_Follower_X", "Member_Follower_Y"
]

# =========================
# Defaults for missing composition columns
# =========================
DEFAULTS = {
    "MCS": 0,
    "Jlf": 0.0,
    "mu": 0.0,
    "PP": 0.0,
    "rep": 0,
    "Cluster UID": 0,
    "Cluster ID": 0,
    "Leader Cells": 0,
    "Follower Cells": 0,
    "Total Cells": 0,
    "Centroid_X": 0.0,
    "Centroid_Y": 0.0,
    "Member_Leader_X": 0.0,
    "Member_Leader_Y": 0.0,
    "Member_Follower_X": 0.0,
    "Member_Follower_Y": 0.0,
}

def default_row(mcs, Jlf, mu, PP, rep):
    """One placeholder row with final standardized schema."""
    row = DEFAULTS.copy()
    row.update({
        "MCS": int(mcs),
        "Jlf": float(Jlf),
        "mu": float(mu),
        "PP": float(PP),
        "rep": int(rep),
    })
    return row

# =========================
# Helpers: tolerant numeric matching (-5 vs -5.0)
# =========================
def canon_num(s: str) -> str:
    """Return canonical numeric string (e.g., '-5.0' -> '-5', '0.50' -> '0.5')."""
    try:
        return f"{float(s):g}"
    except Exception:
        return s

def find_position_folder(scan_path: str, Jlf: str, mu: str, PP: str) -> str:
    """Match PositionData_* regardless of integer/decimal encoding."""
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    for d in os.listdir(scan_path):
        if not d.startswith("PositionData_"):
            continue
        parts = d.split("_")
        if len(parts) >= 4:
            have = (canon_num(parts[1]), canon_num(parts[2]), canon_num(parts[3]))
            if have == want:
                return d
    return f"PositionData_{Jlf}_{mu}_{PP}"

def find_cluster_file(mcs_path: str, Jlf: str, mu: str, PP: str, mcs_val: int) -> str:
    """Match ClusterComposition_*_MCS_<mcs>.csv regardless of integer/decimal encoding."""
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    target_suffix = f"_MCS_{mcs_val}.csv"
    try:
        for fname in os.listdir(mcs_path):
            if not (fname.startswith("ClusterComposition_") and fname.endswith(target_suffix)):
                continue
            stem = fname[:-4] if fname.lower().endswith(".csv") else fname
            parts = stem.split("_")
            if len(parts) >= 6:
                have = (canon_num(parts[1]), canon_num(parts[2]), canon_num(parts[3]))
                if have == want:
                    return os.path.join(mcs_path, fname)
    except FileNotFoundError:
        pass
    return ""

def find_events_file(scan_path: str, Jlf: str, mu: str, PP: str) -> str:
    """Match ClusterEvents_* regardless of integer/decimal encoding (file at scan root)."""
    want = (canon_num(Jlf), canon_num(mu), canon_num(PP))
    try:
        for fname in os.listdir(scan_path):
            if not fname.startswith("ClusterEvents_") or not fname.endswith(".csv"):
                continue
            stem = fname[:-4] if fname.lower().endswith(".csv") else fname
            parts = stem.split("_")
            # Expected: ClusterEvents, Jlf, mu, PP
            if len(parts) >= 4:
                have = (canon_num(parts[1]), canon_num(parts[2]), canon_num(parts[3]))
                if have == want:
                    return os.path.join(scan_path, fname)
    except FileNotFoundError:
        pass
    return ""

# =========================
# Regex for folder parsing
# =========================
folder_pattern = re.compile(r"scan_(\d+)_Jlf_([-.\d]+)_mu_([-.\d]+)_PP_([-.\d]+)_rep_(\d+)")
mcs_folder_pattern = re.compile(r"MCS_(\d+)")

# =========================
# Per-file loader: standardize composition and rename IDs
# =========================
def load_and_standardize(filepath: str, mcs_val: int, Jlf: str, mu: str, PP: str, rep: str) -> pd.DataFrame:
    """
    Read a single per-MCS ClusterComposition CSV and return a DataFrame with FINAL_COLUMNS.
    - Renames 'Cluster ID' -> 'Cluster UID'
    - Renames 'ClusterID_frame' -> 'Cluster ID'
    - If per-frame ID missing, computes it by sorting (Centroid_X, Centroid_Y) and assigning 1..N.
    - Adds/ensures MCS, Jlf, mu, PP, rep.
    """
    try:
        raw = pd.read_csv(filepath)
    except Exception as e:
        print(f"❌ Error reading {filepath}: {e}")
        return pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)])

    if raw.empty:
        return pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)])

    df = raw.copy()

    # ID renames
    rename_map = {}
    if "Cluster ID" in df.columns:
        rename_map["Cluster ID"] = "Cluster UID"
    if "ClusterID_frame" in df.columns:
        rename_map["ClusterID_frame"] = "Cluster ID"
    if rename_map:
        df = df.rename(columns=rename_map)

    # Per-frame 'Cluster ID' (if missing) from centroid order
    if "Cluster ID" not in df.columns:
        if "Centroid_X" in df.columns and "Centroid_Y" in df.columns:
            df = df.sort_values(["Centroid_X", "Centroid_Y"], kind="mergesort").reset_index(drop=True)
            df["Cluster ID"] = range(1, len(df) + 1)
        else:
            df["Cluster ID"] = 0

    # Persistent 'Cluster UID' fallback
    if "Cluster UID" not in df.columns:
        df["Cluster UID"] = 0

    # Params
    df["MCS"] = int(mcs_val)
    df["Jlf"] = float(Jlf)
    df["mu"] = float(mu)
    df["PP"] = float(PP)
    df["rep"] = int(rep)

    # Ensure all final columns exist
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = DEFAULTS[col]

    # Reorder
    df = df[FINAL_COLUMNS]
    return df

# =========================
# Events collection & enrichment helpers
# =========================
def _safe_parse_listcol(df, col, default_factory=list):
    if col not in df.columns:
        df[col] = [default_factory() for _ in range(len(df))]
        return
    def _parse(x):
        if isinstance(x, str):
            try:
                return literal_eval(x)
            except Exception:
                return default_factory()
        return x if isinstance(x, (list, tuple)) else default_factory()
    df[col] = df[col].apply(_parse)

def _detect_dt(comp):
    sim_keys = ["Jlf","mu","PP","rep"]
    dt = (
        comp.groupby(sim_keys)["MCS"]
            .apply(lambda s: pd.Series(np.diff(np.unique(s))).mode().iloc[0] if s.nunique() > 1 else 10)
            .reset_index(name="dt")
    )
    return dt

def _explode_event_edges(events_df):
    E = events_df.copy()
    for col in ["Parents", "Children", "OverlapFractions"]:
        _safe_parse_listcol(E, col)
    for col in ["LostToSingletons", "LostToMainTumor"]:
        if col not in E.columns:
            E[col] = 0.0

    child_rows, parent_rows = [], []
    base_cols = ["Jlf","mu","PP","rep","MCS","Event"]

    for r in E.itertuples(index=False):
        base = {k: getattr(r, k) for k in base_cols}
        parents  = list(getattr(r, "Parents"))
        children = list(getattr(r, "Children"))
        overlaps = list(getattr(r, "OverlapFractions"))
        lost_single = float(getattr(r, "LostToSingletons", 0.0))
        lost_main   = float(getattr(r, "LostToMainTumor", 0.0))

        evt = base["Event"]
        if evt == "merge":
            for p, c, f in zip_longest(parents, children, overlaps, fillvalue=np.nan):
                if pd.isna(c):  # guard
                    continue
                child_rows.append({**base, "Parent": p, "Child": c, "OverlapFraction": f})
                parent_rows.append({**base, "Parent": p, "Child": c, "OverlapFraction": f,
                                    "LostToSingletons": 0.0, "LostToMainTumor": 0.0})
        elif evt == "split":
            parent_uid = parents[0] if parents else np.nan
            for c, f in zip_longest(children, overlaps, fillvalue=np.nan):
                if pd.isna(c):
                    continue
                child_rows.append({**base, "Parent": parent_uid, "Child": c, "OverlapFraction": f})
                parent_rows.append({**base, "Parent": parent_uid, "Child": c, "OverlapFraction": f,
                                    "LostToSingletons": 0.0, "LostToMainTumor": 0.0})
        elif evt in ("dissolve", "absorbed"):
            parent_uid = parents[0] if parents else np.nan
            parent_rows.append({**base, "Parent": parent_uid, "Child": np.nan, "OverlapFraction": np.nan,
                                "LostToSingletons": lost_single, "LostToMainTumor": lost_main})
        else:
            continue

    child_edges  = pd.DataFrame(child_rows)  if child_rows  else pd.DataFrame(columns=base_cols+["Parent","Child","OverlapFraction"])
    parent_edges = pd.DataFrame(parent_rows) if parent_rows else pd.DataFrame(columns=base_cols+["Parent","Child","OverlapFraction","LostToSingletons","LostToMainTumor"])
    return child_edges, parent_edges

def _build_child_table(child_edges):
    if child_edges.empty:
        return pd.DataFrame(columns=["Jlf","mu","PP","rep","MCS","Cluster UID",
                                     "EventHereType","EventHerePartners","EventHereOverlapFractions"])
    keys = ["Jlf","mu","PP","rep","MCS","Child"]
    agg = (child_edges
           .groupby(keys)
           .agg(EventHereType=("Event","first"),
                EventHerePartners=("Parent", lambda s: [int(x) for x in s if pd.notna(x)]),
                EventHereOverlapFractions=("OverlapFraction", lambda s: [float(x) for x in s if pd.notna(x)]))
           .reset_index())
    agg.rename(columns={"Child":"Cluster UID"}, inplace=True)
    return agg

def _build_parent_table(parent_edges, comp_dt):
    if parent_edges.empty:
        return pd.DataFrame(columns=["Jlf","mu","PP","rep","MCS","Cluster UID",
                                     "EventNextType","EventNextTargets","EventNextOverlapFractions",
                                     "EventNextLostToSingletons","EventNextLostToMainTumor"])
    sim_keys = ["Jlf","mu","PP","rep"]
    PE = parent_edges.merge(comp_dt, on=sim_keys, how="left").copy()
    PE["ParentMCS"] = PE["MCS"] - PE["dt"]

    keys = sim_keys + ["ParentMCS","Parent"]
    def _list_or_empty(s, cast):
        vals = [cast(x) for x in s if pd.notna(x)]
        return vals

    agg = (PE.groupby(keys)
             .agg(EventNextType=("Event", "first"),
                  EventNextTargets=("Child", lambda s: _list_or_empty(s, int)),
                  EventNextOverlapFractions=("OverlapFraction", lambda s: _list_or_empty(s, float)),
                  EventNextLostToSingletons=("LostToSingletons", "max"),
                  EventNextLostToMainTumor=("LostToMainTumor", "max"))
             .reset_index())
    agg.rename(columns={"ParentMCS":"MCS", "Parent":"Cluster UID"}, inplace=True)
    return agg

def annotate_composition_with_events(comp_df, events_df):
    """Return composition with event columns merged in (robust to column overlaps)."""
    out = comp_df.copy()
    keys = ["Jlf","mu","PP","rep","MCS","Cluster UID"]

    # No events: add empty columns and return
    if comp_df.empty or events_df.empty:
        for col, default in [
            ("EventHereType", np.nan),
            ("EventHerePartners", np.nan),
            ("EventHereOverlapFractions", np.nan),
            ("EventNextType", np.nan),
            ("EventNextTargets", np.nan),
            ("EventNextOverlapFractions", np.nan),
            ("EventNextLostToSingletons", 0.0),
            ("EventNextLostToMainTumor", 0.0),
        ]:
            if col not in out.columns:
                out[col] = default
        out["HasEventHere"] = out["EventHereType"].notna()
        out["HasEventNext"] = out["EventNextType"].notna()
        return out

    # Build edges/tables
    child_edges, parent_edges = _explode_event_edges(events_df)
    comp_dt = _detect_dt(comp_df)
    child_tab  = _build_child_table(child_edges)
    parent_tab = _build_parent_table(parent_edges, comp_dt)

    # Drop overlapping columns in 'out' before merging
    overlap_child = [c for c in ["EventHereType","EventHerePartners","EventHereOverlapFractions"] if c in out.columns]
    if overlap_child:
        out = out.drop(columns=overlap_child)
    overlap_parent = [c for c in ["EventNextType","EventNextTargets","EventNextOverlapFractions",
                                  "EventNextLostToSingletons","EventNextLostToMainTumor"] if c in out.columns]
    if overlap_parent:
        out = out.drop(columns=overlap_parent)

    # Merge in event annotations
    out = out.merge(child_tab, on=keys, how="left")
    out = out.merge(parent_tab, on=keys, how="left")

    # Ensure all event columns exist (if some sims had none)
    for col, default in [
        ("EventHereType", np.nan),
        ("EventHerePartners", np.nan),
        ("EventHereOverlapFractions", np.nan),
        ("EventNextType", np.nan),
        ("EventNextTargets", np.nan),
        ("EventNextOverlapFractions", np.nan),
        ("EventNextLostToSingletons", 0.0),
        ("EventNextLostToMainTumor", 0.0),
    ]:
        if col not in out.columns:
            out[col] = default

    # Boolean convenience flags
    out["HasEventHere"] = out["EventHereType"].notna()
    out["HasEventNext"] = out["EventNextType"].notna()
    return out

# =========================
# Collect compositions & events
# =========================
comp_rows = []
event_rows = []

if not os.path.isdir(base_dir):
    print(f"❌ Base directory not found: {base_dir}")
else:
    for scan_folder in os.listdir(base_dir):
        scan_path = os.path.join(base_dir, scan_folder)
        if not os.path.isdir(scan_path):
            continue

        folder_match = folder_pattern.match(scan_folder)
        if not folder_match:
            continue

        scan_idx, Jlf, mu, PP, rep = folder_match.groups()

        # -------- compositions --------
        position_data_folder = find_position_folder(scan_path, Jlf, mu, PP)
        position_path = os.path.join(scan_path, position_data_folder)

        if not os.path.isdir(position_path):
            print(f"❌ Missing PositionData folder (tolerant+fallback): {position_data_folder}")
        else:
            for mcs_folder in os.listdir(position_path):
                mcs_match = mcs_folder_pattern.match(mcs_folder)
                if not mcs_match:
                    continue
                mcs_val = int(mcs_match.group(1))
                mcs_path = os.path.join(position_path, mcs_folder)
                if not os.path.isdir(mcs_path):
                    continue

                tolerant_path = find_cluster_file(mcs_path, Jlf, mu, PP, mcs_val)
                filepath = tolerant_path if tolerant_path else os.path.join(mcs_path, f"ClusterComposition_{Jlf}_{mu}_{PP}_MCS_{mcs_val}.csv")

                if os.path.isfile(filepath):
                    df_std = load_and_standardize(filepath, mcs_val, Jlf, mu, PP, rep)
                    comp_rows.append(df_std)
                else:
                    print(f"⚠️ File missing: {filepath}")
                    comp_rows.append(pd.DataFrame([default_row(mcs_val, Jlf, mu, PP, rep)]))

        # -------- events (one file per scan) --------
        ev_path = find_events_file(scan_path, Jlf, mu, PP)
        if ev_path and os.path.isfile(ev_path):
            try:
                ev = pd.read_csv(ev_path)
                if not ev.empty:
                    ev = ev.copy()
                    ev["Jlf"] = float(Jlf)
                    ev["mu"]  = float(mu)
                    ev["PP"]  = float(PP)
                    ev["rep"] = int(rep)
                    # Ensure required columns exist (robust)
                    for c in ["MCS","Event","Parents","Children","OverlapFractions",
                              "LostToSingletons","LostToMainTumor"]:
                        if c not in ev.columns:
                            ev[c] = np.nan
                    event_rows.append(ev)
            except Exception as e:
                print(f"❌ Error reading events file {ev_path}: {e}")

# =========================
# Combine, sort, annotate, save
# =========================
if not comp_rows:
    print("\n❗No cluster composition files were processed.")
else:
    comp_df = pd.concat(comp_rows, ignore_index=True)

    if comp_df.empty:
        print("\n❗Cluster files were collected, but all were empty.")
    else:
        # Sort: params → time → persistent UID → per-frame ID
        sort_cols = ["Jlf", "mu", "PP", "rep", "MCS", "Cluster UID", "Cluster ID"]
        comp_df.sort_values(by=sort_cols, inplace=True, kind="mergesort")

        # Save base compositions (as before)
        if args.mode in ("all", "both"):
            comp_df.to_csv(out_all_csv, index=False)
            print(f"\n✅ Saved ALL rows (incl. zero-cluster placeholders) to:\n{out_all_csv}")
            print(f"• Rows: {len(comp_df)} | "
                  f"Unique sims: {comp_df[['Jlf','mu','PP','rep']].drop_duplicates().shape[0]}")

        if args.mode in ("clusters", "both"):
            df_clusters_only = comp_df[comp_df["Total Cells"] > 0].copy()
            df_clusters_only.to_csv(out_clusters_csv, index=False)
            print(f"\n✅ Saved CLUSTERS-ONLY rows (Total Cells > 0) to:\n{out_clusters_csv}")
            print(f"• Rows: {len(df_clusters_only)} | "
                  f"Unique sims: {df_clusters_only[['Jlf','mu','PP','rep']].drop_duplicates().shape[0]}")

        # Combine & save events
        events_df = pd.concat(event_rows, ignore_index=True) if event_rows else pd.DataFrame()
        if not events_df.empty:
            # make sure numeric types are consistent
            for c in ["Jlf","mu","PP"]:
                events_df[c] = pd.to_numeric(events_df[c], errors="coerce")
            events_df["rep"] = pd.to_numeric(events_df["rep"], errors="coerce")
            events_df["MCS"] = pd.to_numeric(events_df["MCS"], errors="coerce")
            events_df.to_csv(events_out_csv, index=False)
            print(f"\n✅ Saved collected ClusterEvents to:\n{events_out_csv} (rows: {len(events_df)})")
        else:
            print("\nℹ️ No ClusterEvents found; enriched outputs will still be written with empty event columns.")

        # Enrich compositions with event annotations
        comp_all_enriched = annotate_composition_with_events(comp_df, events_df)

        if args.mode in ("all", "both"):
            comp_all_enriched.to_csv(out_all_enriched_csv, index=False)
            print(f"✅ Saved ALL+EVENTS to:\n{out_all_enriched_csv}")

        if args.mode in ("clusters", "both"):
            comp_clust_enriched = comp_all_enriched[comp_all_enriched["Total Cells"] > 0].copy()
            comp_clust_enriched.to_csv(out_clusters_enriched_csv, index=False)
            print(f"✅ Saved CLUSTERS-ONLY+EVENTS to:\n{out_clusters_enriched_csv}")

print("\n🎉 Done.")



✅ Saved ALL rows (incl. zero-cluster placeholders) to:
../../Results/Data/Clusters/ClusterComposition_All.csv
• Rows: 1141646 | Unique sims: 13310

✅ Saved CLUSTERS-ONLY rows (Total Cells > 0) to:
../../Results/Data/Clusters/ClusterComposition_1.csv
• Rows: 295738 | Unique sims: 4046

✅ Saved collected ClusterEvents to:
../../Results/Data/Clusters/ClusterEvents_All.csv (rows: 8751)
✅ Saved ALL+EVENTS to:
../../Results/Data/Clusters/ClusterComposition_All_Enriched.csv
✅ Saved CLUSTERS-ONLY+EVENTS to:
../../Results/Data/Clusters/ClusterComposition_1_Enriched.csv

🎉 Done.


## Each Cluster Data

In [ ]:
import os
import math
import pandas as pd
import numpy as np

# -----------------------------
# Configuration
# -----------------------------
BEST_BY = "pp09"  # OR "clusters" OR "persistence"
PP = 0.9
CLUSTERS_FILE       = "../../Results/Data/Clusters/ClusterComposition_All.csv"
BEST_FILE_IN        = None  # optional override, e.g. "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"
PARA_SUMMARY_OUT    = "../../Results/Data/Clusters/clusters_per_parameter.csv"
BEST_FILE_OUT       = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"
DETAIL_OUT          = "../../Results/Data/Clusters/best_clusters_detailed.csv"
DETAIL_ALL_OUT      = "../../Results/Data/Clusters/cluster_detailed.csv"
# STABILITY_TIME_OUT  = "../../Results/Data/Clusters/cluster_stability_time.csv"

DOMINANCE_THRESHOLD = 60.0  # percent

# -----------------------------
# Helpers
# -----------------------------
def safe_pct(numer, denom):
    try:
        numer = float(numer); denom = float(denom)
        return 0.0 if denom <= 0 else 100.0 * numer / denom
    except Exception:
        return 0.0

def euclid_dxdy(x0, y0, x1, y1):
    try:
        return math.hypot(float(x1) - float(x0), float(y1) - float(y0))
    except Exception:
        return 0.0

def dominance_label(leader_pct, threshold=DOMINANCE_THRESHOLD):
    return "Leader-dominated" if leader_pct >= threshold else "Follower-dominated"

def select_best_pp(clusters_per_param, first_pos, mode):
    """Select best PP per (Jlf, mu) based on chosen mode."""
    tmp = clusters_per_param.merge(first_pos, on=["Jlf", "mu", "PP"], how="left")

    if mode == "clusters":
        sort_cols = ["Jlf", "mu", "NumClusters", "AvgPersistence", "first_index"]
        ascending = [ True,  True,  False,        False,            True]

    elif mode == "persistence":
        sort_cols = ["Jlf", "mu", "AvgPersistence", "NumClusters", "first_index"]
        ascending = [ True,  True,  False,           False,         True]

    elif mode == "pp09":
        tmp = tmp[tmp["PP"] == PP]
        sort_cols = ["Jlf", "mu", "first_index"]
        ascending = [True, True, True]

    else:
        raise ValueError(f"Unknown BEST_BY mode: {mode}")

    tmp = tmp.sort_values(by=sort_cols, ascending=ascending, kind="stable")

    return (
        tmp.groupby(["Jlf", "mu"], as_index=False)
           .first()[["Jlf", "mu", "PP", "NumClusters", "AvgPersistence"]]
    )

def size_cv(series: pd.Series):
    """Coefficient of variation of a size time series (std/mean)."""
    if series.size < 2:
        return np.nan
    m = series.mean()
    if m is None or m <= 0:
        return np.nan
    s = series.std()  # sample std (ddof=1)
    return s / m

def phenotypic_index(L, F):
    """PI = |L-F| / (L+F)."""
    try:
        L = float(L); F = float(F)
        T = L + F
        if T <= 0:
            return np.nan
        return abs(L - F) / T
    except Exception:
        return np.nan

def shannon_entropy(L, F):
    """H = -pL log2 pL - pF log2 pF, with 0*log 0 := 0."""
    try:
        L = float(L); F = float(F)
        T = L + F
        if T <= 0:
            return np.nan
        pL = L / T
        pF = F / T
        termL = -pL * np.log2(pL) if pL > 0 else 0.0
        termF = -pF * np.log2(pF) if pF > 0 else 0.0
        return termL + termF
    except Exception:
        return np.nan

def build_cluster_detail_rows_for_params(clusters_idx, param_keys):
    rows = []
    for rec in param_keys[["Jlf","mu","PP"]].drop_duplicates().itertuples(index=False):
        jlf_val, mu_val, pp_val = rec
        try:
            subset = clusters_idx.loc[(jlf_val, mu_val, pp_val)].reset_index()
        except KeyError:
            continue

        # Group by unique replicate–cluster
        for uid, g in subset.groupby("ClusterUID"):
            if g.empty:
                continue
            g = g.sort_values("MCS", kind="stable")

            # Renumbered ID (1..N within this parameter triple) and provenance
            renum_id = int(g["ClusterID_renum"].iloc[0])
            rep_src  = int(g["ClusterSourceRep"].iloc[0])
            orig_id  = int(g["ClusterSourceOrigID"].iloc[0])

            # Persistence and time range
            persistence = g["MCS"].nunique()
            start_time  = int(g["MCS"].min())
            end_time    = int(g["MCS"].max())
            time_span   = f"{start_time}-{end_time}"

            # Per-cluster size stats over its lifetime
            mean_size_over_time = g["Total Cells"].mean()
            std_size_over_time  = g["Total Cells"].std()  # sample std (ddof=1)
            cluster_size_cv     = size_cv(g["Total Cells"])

            # Start & end centroid and displacement/velocity
            start_x, start_y = g.iloc[0][["Centroid_X","Centroid_Y"]]
            end_x,   end_y   = g.iloc[-1][["Centroid_X","Centroid_Y"]]
            displacement = euclid_dxdy(start_x, start_y, end_x, end_y)
            velocity     = displacement / persistence if persistence > 0 else 0.0

            # Final snapshot composition and dominance
            last = g.iloc[-1]
            leader_cells   = pd.to_numeric(last.get("Leader Cells",   np.nan), errors="coerce")
            follower_cells = pd.to_numeric(last.get("Follower Cells", np.nan), errors="coerce")
            total_cells    = pd.to_numeric(last.get("Total Cells",    np.nan), errors="coerce")
            centroid_x     = pd.to_numeric(last.get("Centroid_X",     np.nan), errors="coerce")
            centroid_y     = pd.to_numeric(last.get("Centroid_Y",     np.nan), errors="coerce")

            leader_pct = safe_pct(leader_cells, total_cells)
            dom = dominance_label(leader_pct, DOMINANCE_THRESHOLD)

            # Final-snapshot PI & Entropy
            PI_final      = phenotypic_index(leader_cells, follower_cells)
            Entropy_final = shannon_entropy(leader_cells, follower_cells)

            rows.append({
                "Jlf": jlf_val,
                "mu": mu_val,
                "PP": pp_val,
                "ClusterUID": uid,
                "ClusterID_renum": renum_id,
                # "rep": rep_src,
                # "Cluster ID (orig)": orig_id,
                "Leader Cells": leader_cells,
                "Follower Cells": follower_cells,
                "Total Cells": total_cells,
                "Leader %": leader_pct,
                "Dominance": dom,
                "Centroid_X": centroid_x,
                "Centroid_Y": centroid_y,
                "PersistenceTime": persistence,
                "TimeRange": time_span,
                "Centroid_Displacement": displacement,
                "Velocity": velocity,
                "MeanSize_over_time": mean_size_over_time,
                "StdSize_over_time": std_size_over_time,
                "ClusterSizeCV_over_time": cluster_size_cv,
                "Phenotypic Index": PI_final,
                "Entropy": Entropy_final
            })
    return rows

# -----------------------------
# Load and preprocess data
# -----------------------------
if not os.path.exists(CLUSTERS_FILE):
    raise FileNotFoundError(f"Cannot find required file: {CLUSTERS_FILE}")

df = pd.read_csv(CLUSTERS_FILE, low_memory=False)

required_cols = [
    "MCS", "Jlf", "mu", "PP", "rep", "Cluster ID",
    "Leader Cells", "Follower Cells", "Total Cells",
    "Centroid_X", "Centroid_Y"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {CLUSTERS_FILE}: {missing}")

# Only “real” clusters
clusters_df = df[df["Total Cells"] > 0].copy()
# Coerce dtypes (incl. rep!)
for col in ["Jlf","mu","PP","rep","MCS","Cluster ID",
            "Leader Cells","Follower Cells","Total Cells",
            "Centroid_X","Centroid_Y"]:
    clusters_df[col] = pd.to_numeric(clusters_df[col], errors="coerce")
clusters_df = clusters_df.dropna(subset=["Jlf","mu","PP","rep","Cluster ID","MCS"])

# -----------------------------
# Make replicate–cluster unique + renumber within (Jlf, mu, PP)
# -----------------------------
# Stable unique ID for a cluster instance within a replicate:
# e.g., "5_1" for rep=5, Cluster ID=1
clusters_df["ClusterUID"] = (
    clusters_df["rep"].astype(int).astype(str) + "_" + clusters_df["Cluster ID"].astype(int).astype(str)
)

def _renumber_within_param(g: pd.DataFrame) -> pd.DataFrame:
    # Unique replicate–clusters for this (Jlf, mu, PP), sorted by rep, then original Cluster ID
    keys = (
        g[["rep","Cluster ID","ClusterUID"]]
        .drop_duplicates()
        .sort_values(["rep","Cluster ID"], kind="stable")
        .reset_index(drop=True)
    )
    mapping = {uid: i+1 for i, uid in enumerate(keys["ClusterUID"])}  # 1..N
    g = g.copy()
    g["ClusterID_renum"] = g["ClusterUID"].map(mapping).astype(int)
    # Keep provenance
    g["ClusterSourceRep"] = g["rep"].astype(int)
    g["ClusterSourceOrigID"] = g["Cluster ID"].astype(int)
    return g

clusters_df = (
    clusters_df
    .groupby(["Jlf","mu","PP"], group_keys=False)
    .apply(_renumber_within_param)
)

# -----------------------------
# Stability across time (CV_t) and summaries (per rep, then summarized)
# -----------------------------
cv_time = (
    clusters_df.groupby(["Jlf","mu","PP","rep","MCS"])["Total Cells"]
    .agg(mean_size_t="mean", std_size_t="std", n_clusters_t="count")
    .reset_index()
)
cv_time["CV_t"] = np.where(
    (cv_time["n_clusters_t"] >= 2) & (cv_time["mean_size_t"] > 0),
    cv_time["std_size_t"] / cv_time["mean_size_t"],
    np.nan
)
# if you enable STABILITY_TIME_OUT above, uncomment to save:
# cv_time.to_csv(STABILITY_TIME_OUT, index=False)

cv_rep = (
    cv_time.groupby(["Jlf","mu","PP","rep"], dropna=False)["CV_t"]
    .agg(MeanCV_over_time="mean", MedianCV_over_time="median", Timepoints="count")
    .reset_index()
)
cv_param = (
    cv_rep.groupby(["Jlf","mu","PP"], dropna=False)
    .agg(MeanCV_over_time=("MeanCV_over_time","mean"),
         MedianCV_over_time=("MedianCV_over_time","median"),
         Reps=("rep","nunique"))
    .reset_index()
)

# -----------------------------
# Parameter-level summary using unique replicate–clusters
# -----------------------------
# Persistence per unique replicate–cluster
persistence_per_cluster = (
    clusters_df.groupby(["Jlf","mu","PP","ClusterUID"], as_index=False)["MCS"]
    .nunique().rename(columns={"MCS":"PersistenceTime"})
)

# NumClusters = number of unique replicate–clusters for that (Jlf,mu,PP)
clusters_per_param = (
    persistence_per_cluster
    .groupby(["Jlf","mu","PP"], as_index=False)
    .agg(NumClusters=("ClusterUID","nunique"),
         AvgPersistence=("PersistenceTime","mean"))
)

# Merge CV summaries
clusters_per_param = clusters_per_param.merge(cv_param, on=["Jlf","mu","PP"], how="left")

# First appearance index for tie-breaking and sorted param table
first_pos = (
    clusters_df.reset_index()
    .groupby(["Jlf","mu","PP"])["index"].min()
    .reset_index()
    .rename(columns={"index":"first_index"})
)
clusters_per_param_sorted = (
    clusters_per_param
    .merge(first_pos, on=["Jlf","mu","PP"], how="left")
    .sort_values(by="first_index", kind="stable")
    .drop(columns=["first_index"])
)
# Ensure output directory exists
os.makedirs(os.path.dirname(PARA_SUMMARY_OUT) or ".", exist_ok=True)
clusters_per_param_sorted.to_csv(PARA_SUMMARY_OUT, index=False)

# -----------------------------
# Select best PP per (Jlf, mu)
# -----------------------------
if BEST_FILE_IN and os.path.exists(BEST_FILE_IN):
    best_df = pd.read_csv(BEST_FILE_IN, low_memory=False)
    # Ensure required cols exist; if not, enrich from our computed table
    needed = {"NumClusters","AvgPersistence"}
    if not needed.issubset(best_df.columns):
        best_df = best_df.merge(clusters_per_param[["Jlf","mu","PP","NumClusters","AvgPersistence"]],
                                on=["Jlf","mu","PP"], how="left")
else:
    best_df = select_best_pp(clusters_per_param, first_pos, BEST_BY)
    os.makedirs(os.path.dirname(BEST_FILE_OUT) or ".", exist_ok=True)
    best_df.to_csv(BEST_FILE_OUT, index=False)

for col in ["Jlf","mu","PP","NumClusters","AvgPersistence"]:
    if col in best_df.columns:
        best_df[col] = pd.to_numeric(best_df[col], errors="coerce")

# -----------------------------
# Build detailed lists
# -----------------------------
# Index by parameters for quick subset; ClusterUID handled in groupby below
clusters_idx = clusters_df.set_index(["Jlf","mu","PP"]).sort_index()

# (A) Detailed for BEST ONLY
rows_best = build_cluster_detail_rows_for_params(
    clusters_idx=clusters_idx,
    param_keys=best_df[["Jlf","mu","PP"]]
)
detailed = pd.DataFrame(rows_best)
if not detailed.empty:
    detailed = detailed.sort_values(
        by=["Jlf","mu","PP","ClusterID_renum","PersistenceTime"],
        ascending=[True, True, True, True, False],
        kind="stable"
    )
os.makedirs(os.path.dirname(DETAIL_OUT) or ".", exist_ok=True)
detailed.to_csv(DETAIL_OUT, index=False)

# (B) Detailed for ALL (Jlf, mu, PP)
all_param_keys = clusters_df[["Jlf","mu","PP"]].drop_duplicates()
rows_all = build_cluster_detail_rows_for_params(
    clusters_idx=clusters_idx,
    param_keys=all_param_keys
)
detailed_all = pd.DataFrame(rows_all)
if not detailed_all.empty:
    detailed_all = detailed_all.sort_values(
        by=["Jlf","mu","PP","ClusterID_renum","PersistenceTime"],
        ascending=[True, True, True, True, False],
        kind="stable"
    )
os.makedirs(os.path.dirname(DETAIL_ALL_OUT) or ".", exist_ok=True)
detailed_all.to_csv(DETAIL_ALL_OUT, index=False)

# -----------------------------
# Save parameter table + best file info
# -----------------------------
used_best = BEST_FILE_IN if (BEST_FILE_IN and os.path.exists(BEST_FILE_IN)) else BEST_FILE_OUT
print(f"[BEST_BY: {BEST_BY}]")
if 'STABILITY_TIME_OUT' in globals() and STABILITY_TIME_OUT:
    print(f"Saved per-time stability: {STABILITY_TIME_OUT}")
else:
    print("Per-time stability file: (not saved)")
print(f"Saved parameter summary: {PARA_SUMMARY_OUT}")
print(f"Saved best (Jlf, mu, PP): {used_best}")
print(f"Saved detailed clusters (best only): {DETAIL_OUT}")
print(f"Saved detailed clusters (ALL params): {DETAIL_ALL_OUT}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_98613/301126205.py:243: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_renumber_within_param)


[BEST_BY: pp09]
Per-time stability file: (not saved)
Saved parameter summary: ../../Results/Data/Clusters/clusters_per_parameter.csv
Saved best (Jlf, mu, PP): ../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv
Saved detailed clusters (best only): ../../Results/Data/Clusters/best_clusters_detailed.csv
Saved detailed clusters (ALL params): ../../Results/Data/Clusters/cluster_detailed.csv


# Cluster Analysis

### Cluster Rate

The **cluster rate** is defined as the discrete difference of the cluster count over time. It quantifies the net change in the number of clusters between two consecutive time steps:

$$
\text{Cluster Rate}_{t} = \Delta C_t = C_{t} - C_{t-1}
$$

Or, in approximate derivative form:

$$
\frac{dC}{dt} \approx \frac{C(t+\Delta t) - C(t)}{\Delta t}
$$

This metric can be:
- **Positive** (formation of new clusters),
- **Negative** (merging or disappearance),
- **Zero** (stable cluster count).

---

### Cluster Formation Rate

The **cluster formation rate** captures only the positive changes (i.e., formation of new clusters):

$$
\text{CFR}(t) = \max(0, \Delta C_t) = \max\left(0, \frac{C(t+\Delta t) - C(t)}{\Delta t}\right)
$$

This reflects the number of clusters that newly emerged between  $t-1$  and  $t $.


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# -------------------------------
# STEP 1: Load the simulation data
# -------------------------------
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path, low_memory=False)

# -------------------------------------------
# STEP 2: Compute Cluster Rate and CFR per replicate
# -------------------------------------------
group_cols = ['Jlf', 'mu', 'PP', 'rep']  # Unique simulation setup
cluster_rate_records = []

# Loop through each replicate of each parameter combination
for group_key, group_df in df.groupby(group_cols):
    # Count number of unique clusters at each time step (MCS)
    cluster_count_per_mcs = group_df.groupby('MCS')['Cluster ID'].nunique().sort_index()
    
    # Compute Cluster Rate: ΔCₜ = Cₜ - Cₜ₋₁
    delta_ct = cluster_count_per_mcs.diff().fillna(0).astype(int)
    
    # Compute CFR: max(0, ΔCₜ)
    cfr_t = delta_ct.clip(lower=0)
    
    # Store results
    for mcs, rate in delta_ct.items():
        cluster_rate_records.append({
            'Jlf': group_key[0],
            'mu': group_key[1],
            'PP': group_key[2],
            'rep': group_key[3],
            'MCS': mcs,
            'Cluster_Rate': rate,
            'CFR': cfr_t.loc[mcs]
        })

# Convert to DataFrame
cluster_rate_df = pd.DataFrame(cluster_rate_records)

# Save per-replicate Cluster Rate and CFR
output_csv_path = "../../Results/Cluster/ClusterRate_CFR.csv"
cluster_rate_df.to_csv(output_csv_path, index=False)

# ----------------------------------------------------
# STEP 3: Average Cluster Rate and CFR across replicates
# ----------------------------------------------------
avg_cluster_rate_df = (
    cluster_rate_df
    .groupby(['Jlf', 'mu', 'PP', 'MCS'])[['Cluster_Rate', 'CFR']]
    .mean()
    .reset_index()
)

# Save averaged results
avg_output_csv_path = "../../Results/Cluster/Average_ClusterRate_CFR.csv"
avg_cluster_rate_df.to_csv(avg_output_csv_path, index=False)

# ----------------------------------------------------
# STEP 4: Select specific parameter set to visualize
# ----------------------------------------------------
selected_params = (2.0, 27.0, 0.4)  # Choose one (Jlf, mu, PP) combo

replicate_df = cluster_rate_df[
    (cluster_rate_df['Jlf'] == selected_params[0]) &
    (cluster_rate_df['mu'] == selected_params[1]) &
    (cluster_rate_df['PP'] == selected_params[2])
]

avg_df = avg_cluster_rate_df[
    (avg_cluster_rate_df['Jlf'] == selected_params[0]) &
    (avg_cluster_rate_df['mu'] == selected_params[1]) &
    (avg_cluster_rate_df['PP'] == selected_params[2])
]

# ----------------------------------------------------
# STEP 5: Plot all replicates (each replicate in its own subplot)
# ----------------------------------------------------
rep_ids = replicate_df['rep'].unique()
num_reps = len(rep_ids)
cols = min(5, num_reps)  
rows = math.ceil(num_reps / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), sharex=True, sharey=True)
axes = axes.flatten() if num_reps > 1 else [axes]

for i, rep_id in enumerate(rep_ids):
    rep_data = replicate_df[replicate_df['rep'] == rep_id]
    axes[i].plot(rep_data['MCS'], rep_data['Cluster_Rate'], label='Cluster Rate', marker='o', markersize=3)
    axes[i].plot(rep_data['MCS'], rep_data['CFR'], label='CFR', marker='s', markersize=3)
    axes[i].set_title(f"Replicate {rep_id}")
    axes[i].grid(True)
    axes[i].legend(fontsize="small")


for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Cluster Rate and CFR per Replicate\n(Jlf={selected_params[0]}, mu={selected_params[1]}, PP={selected_params[2]})', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save replicate subplot grid
rep_plot_path = "../../Results/Cluster/ClusterRate_CFR.png"
plt.savefig(rep_plot_path)
plt.close()

# ----------------------------------------------------
# STEP 6: Plot average Cluster Rate and CFR across replicates
# ----------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(avg_df['MCS'], avg_df['Cluster_Rate'], label='Avg Cluster Rate (ΔCₜ)', marker='o', markersize=4)
plt.plot(avg_df['MCS'], avg_df['CFR'], label='Avg CFR', marker='s', markersize=4)
plt.title(f'Average Cluster Dynamics\nJlf={selected_params[0]}, mu={selected_params[1]}, PP={selected_params[2]}')
plt.xlabel('MCS')
plt.ylabel('Rate')
plt.grid(True)
plt.legend()
plt.tight_layout()

# Save average plot
avg_plot_path = "../../Results/Cluster/Average_ClusterRate_CFR.png"
plt.savefig(avg_plot_path)
plt.close()

# ----------------------------------------------------
# STEP 7: Final output confirmation
# ----------------------------------------------------
print("✔ Cluster Rate (per replicate) saved to:", output_csv_path)
print("✔ Average Cluster Rate (across replicates) saved to:", avg_output_csv_path)
print("✔ Replicate plots saved to:", rep_plot_path)
print("✔ Average plot saved to:", avg_plot_path)


✔ Cluster Rate (per replicate) saved to: ../../Results/Cluster/ClusterRate_CFR.csv
✔ Average Cluster Rate (across replicates) saved to: ../../Results/Cluster/Average_ClusterRate_CFR.csv
✔ Replicate plots saved to: ../../Results/Cluster/ClusterRate_CFR.png
✔ Average plot saved to: ../../Results/Cluster/Average_ClusterRate_CFR.png


### Cluster Stability and Deformation

To quantify the stability of cluster sizes over time, we use the **Coefficient of Variation (CV)** of average cluster size. It measures the variation in cluster size over time. Let $S_i(t)$  be the size (number of cells) of cluster  $i$  at time  $t $, and $\overline{S}_t$  be the mean size of clusters at time $t$ .

- **Variance of cluster sizes over time:**

$$
\sigma_S^2(t) = \frac{1}{N_t} \sum_{i=1}^{N_t} \left(S_i(t) - \overline{S}_t\right)^2
$$

- **Cluster Size Coefficient of Variation (CV):**

$$
\text{CV}_{\overline{S}_t} = \frac{\sigma_S(t)}{\overline{S}_t}
$$

A high CV suggests instability in cluster size reflecting greater heterogeneity, fragmentation, or active invasion., while a low CV indicates stable, uniform cluster structures.
A **CV close to 0** indicates uniform, stable clusters, typically reflecting cohesive tumor growth. A **CV between 0.1 and 0.3** suggests mild to moderate structural heterogeneity, often due to asynchronous growth or early invasion. A **CV above 0.4** indicates significant variability in cluster sizes, which may correspond to fragmentation, leader–follower dynamics, or sustained invasive behavior. Therefore, rising CV over time can signal a structural transition from uniform expansion to multimodal or unstable tumor morphology.



In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

# ========== File Paths ==========
input_file = "../../Results/Data/Clusters/ClusterComposition.csv"
cv_per_cluster_path = "../../Results/Cluster/ClusterSizeCV_perCluster.csv"
cv_per_rep_path = "../../Results/Cluster/ClusterSizeCV.csv"
cv_avg_path = "../../Results/Cluster/ClusterSizeCV_Avg.csv"
cv_plot_path = "../../Results/Cluster/ClusterCV.png"
cv_avg_plot_path = "../../Results/Cluster/Avg_ClusterCV.png"

# Ensure output directory exists
os.makedirs("../../Results/Cluster", exist_ok=True)

# ========== Load Data ==========
df = pd.read_csv(input_file, low_memory=False)

# ========== 1. Compute CV per Cluster ID ==========
group_cols_cluster = ['Jlf', 'mu', 'PP', 'rep', 'Cluster ID']
cv_cluster_results = []

for (jlf, mu, pp, rep, cluster_id), cluster_df in df.groupby(group_cols_cluster):
    total_cells_over_time = cluster_df.sort_values('MCS')['Total Cells'].values
    mean_size = np.mean(total_cells_over_time)
    std_size = np.std(total_cells_over_time)
    cv = std_size / mean_size if mean_size > 0 else np.inf

    cv_cluster_results.append({
        'Jlf': jlf,
        'mu': mu,
        'PP': pp,
        'rep': rep,
        'Cluster ID': cluster_id,
        'CV': cv if np.isfinite(cv) else '-',
        'Mean Size': mean_size,
        'Std Dev Size': std_size,
        'Time Points': len(total_cells_over_time)
    })

cv_cluster_df = pd.DataFrame(cv_cluster_results)
cv_cluster_df.to_csv(cv_per_cluster_path, index=False)
print(f"✔ Saved CV per Cluster to {cv_per_cluster_path}")

# ========== 2. Compute CV per Replicate per MCS ==========
group_cols_mcs = ['Jlf', 'mu', 'PP', 'rep', 'MCS']
cv_results = []

for (jlf, mu, pp, rep, mcs), group_df in df.groupby(group_cols_mcs):
    cluster_sizes = group_df['Total Cells'].values
    cluster_sizes = cluster_sizes[cluster_sizes > 0]

    if len(cluster_sizes) == 0 or np.mean(cluster_sizes) == 0:
        mean_size = std_size = count = 0
        cv = np.inf
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size
        count = len(cluster_sizes)

    cv_results.append({
        'Jlf': jlf,
        'mu': mu,
        'PP': pp,
        'rep': rep,
        'MCS': mcs,
        'Cluster Count': count,
        'Mean Size': mean_size,
        'Std Dev': std_size,
        'CV': cv if np.isfinite(cv) else 0
    })

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(cv_per_rep_path, index=False)
print(f"✔ Saved CV per MCS per Replicate to {cv_per_rep_path}")

# ========== 3. Compute Average CV across Replicates ==========
avg_cv_df = cv_df.groupby(['Jlf', 'mu', 'PP', 'MCS']).agg({
    'Cluster Count': 'mean',
    'Mean Size': 'mean',
    'Std Dev': 'mean',
    'CV': 'mean'
}).reset_index()

avg_cv_df.rename(columns={
    'Cluster Count': 'Avg Cluster Count',
    'Mean Size': 'Avg Mean Size',
    'Std Dev': 'Avg Std Dev',
    'CV': 'Avg CV'
}, inplace=True)

avg_cv_df.to_csv(cv_avg_path, index=False)
print(f"✔ Saved Averaged CV across Replicates to {cv_avg_path}")

# ========== 4. Plotting for a Parameter Combination ==========
plot_jlf = 3
plot_mu = 27
plot_pp = 0.4

# --- Replicate Subplots ---
subset_reps = cv_df[
    (cv_df['Jlf'] == plot_jlf) &
    (cv_df['mu'] == plot_mu) &
    (cv_df['PP'] == plot_pp)
]
replicates = sorted(subset_reps['rep'].unique())
num_reps = len(replicates)

if num_reps > 0:
    n_cols = min(4, num_reps)
    n_rows = math.ceil(num_reps / n_cols)
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharey=True)
    axs = axs.flatten()

    for i, rep in enumerate(replicates):
        rep_data = subset_reps[subset_reps['rep'] == rep].sort_values("MCS")
        axs[i].plot(rep_data["MCS"], rep_data["CV"], label=f"Rep {rep}")
        axs[i].set_title(f"Rep {rep}")
        axs[i].set_xlabel("MCS")
        axs[i].set_ylabel("CV of Cluster Size")
        axs[i].grid(True)

    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    plt.suptitle(f"CV per Replicate for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(cv_plot_path)
    plt.close()
    print(f"✔ Saved CV replicate subplot to {cv_plot_path}")
else:
    print(f"⚠ No replicates found for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")

# --- Average Line Plot ---
avg_subset = avg_cv_df[
    (avg_cv_df['Jlf'] == plot_jlf) &
    (avg_cv_df['mu'] == plot_mu) &
    (avg_cv_df['PP'] == plot_pp)
].sort_values("MCS")

if not avg_subset.empty:
    plt.figure(figsize=(7, 5))
    plt.plot(avg_subset["MCS"], avg_subset["Avg CV"], color='red', linewidth=2)
    plt.title(f"Averaged CV Across Replicates\nJlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")
    plt.xlabel("MCS")
    plt.ylabel("Avg CV of Cluster Size")
    plt.grid(True)
    plt.savefig(cv_avg_plot_path)
    plt.close()
    print(f"✔ Saved average CV plot to {cv_avg_plot_path}")
else:
    print(f"⚠ No averaged data found for Jlf={plot_jlf}, mu={plot_mu}, PP={plot_pp}")


✔ Saved CV per Cluster to ../../Results/Cluster/ClusterSizeCV_perCluster.csv
✔ Saved CV per MCS per Replicate to ../../Results/Cluster/ClusterSizeCV.csv
✔ Saved Averaged CV across Replicates to ../../Results/Cluster/ClusterSizeCV_Avg.csv
✔ Saved CV replicate subplot to ../../Results/Cluster/ClusterCV.png
✔ Saved average CV plot to ../../Results/Cluster/Avg_ClusterCV.png


## Phenotypic Composition Analysis

### Leader Cell Percentage

At each time step, the proportion of leader cells is defined as the **leader cell percentage** of a cluster at time  $t $:

$$
p_L(t) = \frac{L_{i}^{(t)}}{L_{i}^{(t)} + F_{i}^{(t)}} \times 100
$$

Clusters or time windows can be classified based on $ p_L(t) $:

- **Leader-dominated**:  $p_L(t) > \theta $
- **Follower-dominated**:  $p_L(t) \leq \theta$ 

where  $\theta$  is a defined threshold (e.g.,  $\theta = 60\% $).

---

### Composition-Based Stability

Let $C^{\text{LD}}_t$  and  $C^{\text{FD}}_t$  represent the number of leader- and follower-dominated clusters at time  $t $. We compute separate CV values for each:

$$
\text{CV}^{\text{LD}} = \frac{\sigma^{\text{LD}}(t)}{\overline{S}^{\text{LD}}_t}, \quad
\text{CV}^{\text{FD}} = \frac{\sigma^{\text{FD}}(t)}{\overline{S}^{\text{FD}}_t}
$$

Comparing  $\text{CV}^{\text{LD}}$  and $\text{CV}^{\text{FD}}$  helps assess which type exhibits more stable sizes.

---

### Phenotypic Index and Entropy

For a cluster  $i$  at time  $t$ , let  $L_i(t)$  be the number of leader cells and  $F_i(t) $ the number of follower cells.

- **Phenotypic Index:**

$$
PI_i(t) = \frac{|L_{i}^{(t)} - F_{i}^{(t)}|}{L_{i}^{(t)} + F_{i}^{(t)}} = \frac{|L_{i}^{(t)} - F_{i}^{(t)}|}{T^t_i}
$$

- **Shannon Entropy of Composition:**

$$
p_L = \frac{L_{i}^{(t)} }{T^t_i}, \quad p_F = \frac{F_{i}^{(t)} }{T^t_i}
$$

$$
H_i(t) = -p_L \log_2(p_L) - p_F \log_2(p_F)
$$

Entropy quantifies phenotypic diversity, with higher values indicating more balanced compositions.
The Phenotypic Index ($PI_i(t)$) measures the imbalance between leader and follower cells within a cluster, ranging from 0 (perfectly balanced) to 1 (entirely one type). A high $PI$ suggests strong phenotypic skew, which may signal specialization or dominance. In contrast, the Entropy $H_i(t)$ captures the diversity in phenotypic makeup, peaking at 1 when leaders and followers are equally represented ($p_L = p_F = 0.5$) and dropping to 0 when one phenotype dominates completely. Thus, together, these metrics provide insight into the heterogeneity and cooperative balance of tumor cell clusters, which can influence collective behavior and invasive potential.

In [ ]:
import pandas as pd
import numpy as np

file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path, low_memory=False)

df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')


L = df["Leader Cells"]        # Number of leader cells
F = df["Follower Cells"]      # Number of follower cells
T = L + F                     # Total cells = Leader + Follower

# ------------------------------------------------------
#  Leader Percentage
# Formula: p_L(t) = (L / (L + F)) * 100
# ------------------------------------------------------
threshold = 60  # Classification threshold in percent
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100

# ------------------------------------------------------
#  Dominance Classification
# If leader percentage > threshold => Leader-dominated
# Else => Follower-dominated
# ------------------------------------------------------
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # Handle cases where total cells = 0


# ------------------------------------------------------
# Phenotypic Index
# Formula: PI_i(t) = |L - F| / (L + F)
# Measures degree of phenotypic imbalance
# ------------------------------------------------------
df["Phenotypic_Index"] = np.abs(L - F) / T.replace(0, np.nan)

# ------------------------------------------------------
# Shannon Entropy
# Formula: H = -p_L * log2(p_L) - p_F * log2(p_F)
# Measures phenotypic diversity (max = 1 when L=F, min = 0 when one type dominates)
# ------------------------------------------------------
p_L = L / T.replace(0, np.nan)
p_F = F / T.replace(0, np.nan)
df["Entropy"] = -(
    p_L * np.log2(p_L.replace(0, np.nan)) +
    p_F * np.log2(p_F.replace(0, np.nan))
)

result_df = df[[
    "Jlf", "mu", "PP", "rep", "MCS", "Cluster ID",
    "Leader_Percentage", "Dominance", "Phenotypic_Index", "Entropy"
]]

# Save to CSV
output_path =  "../../Results/Cluster/PhenotypicComposition.csv"
result_df.to_csv(output_path, index=False)

output_path


'../../Results/Cluster/PhenotypicComposition.csv'

In [42]:
import pandas as pd
import numpy as np

# Load original composition file (ensure it includes Total Cells and Dominance)
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path)

# Ensure numeric types
df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')

# Recompute dominance if not already in the file
L = df["Leader Cells"]
F = df["Follower Cells"]
T = L + F
threshold = 60
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # Skip zero-sized clusters

# --------------------------------------------------
# Compute CV of size over time for LD and FD clusters
# Grouping: each cluster over time in a given sim run
# --------------------------------------------------
group_cols = ["Jlf", "mu", "PP", "rep", "Cluster ID", "Dominance"]
cv_records = []

for group_key, cluster_df in df.groupby(group_cols):
    jlf, mu, pp, rep, cluster_id, dom_type = group_key
    cluster_sizes = cluster_df["Total Cells"].dropna().values

    if len(cluster_sizes) == 0 or np.mean(cluster_sizes) == 0:
        cv = np.inf
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size

    cv_records.append({
        "Jlf": jlf,
        "mu": mu,
        "PP": pp,
        "rep": rep,
        "Dominance": dom_type,
        "CV": cv if np.isfinite(cv) else 0,
        "Mean Size": mean_size if len(cluster_sizes) > 0 else 0,
        "Std Size": std_size if len(cluster_sizes) > 0 else 0,
        "Cluster Count": len(cluster_sizes),
    })

# Convert to DataFrame
cv_df = pd.DataFrame(cv_records)

# --------------------------------------------------
# Aggregate CVs across clusters per dominance group
# Result: one LD and one FD CV per sim run
# --------------------------------------------------
stability_summary = cv_df.groupby(["Jlf", "mu", "PP", "rep", "Dominance"]).agg({
    "CV": "mean",
    "Mean Size": "mean",
    "Std Size": "mean"
}).reset_index()

# Optional pivot for easier comparison
pivot_df = stability_summary.pivot(
    index=["Jlf", "mu", "PP", "rep"],
    columns="Dominance",
    values="CV"
).reset_index()

# Save to file
output_path = "../../Results/Cluster/ClusterStability_by_Dominance.csv"
stability_summary.to_csv(output_path, index=False)

print(f"Saved dominance-based cluster stability to: {output_path}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_80865/2118834574.py:6: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Saved dominance-based cluster stability to: ../../Results/Cluster/ClusterStability_by_Dominance.csv


In [44]:
import pandas as pd
import numpy as np

# Load composition data
file_path = "../../Results/Data/Clusters/ClusterComposition.csv"
df = pd.read_csv(file_path)

# Convert to numeric
df["Leader Cells"] = pd.to_numeric(df["Leader Cells"], errors='coerce')
df["Follower Cells"] = pd.to_numeric(df["Follower Cells"], errors='coerce')
df["Total Cells"] = pd.to_numeric(df["Total Cells"], errors='coerce')

# Recalculate dominance
L = df["Leader Cells"]
F = df["Follower Cells"]
T = L + F
threshold = 60
df["Leader_Percentage"] = (L / T.replace(0, np.nan)) * 100
df["Dominance"] = np.where(df["Leader_Percentage"] > threshold, "Leader-dominated", "Follower-dominated")
df.loc[T == 0, "Dominance"] = np.nan  # mark undefined dominance

# -------------------------------
# Compute CV per cluster
# -------------------------------
group_cols = ["Jlf", "mu", "PP", "rep", "Cluster ID", "Dominance"]
cv_records = []

for group_key, cluster_df in df.groupby(group_cols):
    jlf, mu, pp, rep, cluster_id, dom_type = group_key
    cluster_sizes = cluster_df["Total Cells"].dropna().values
    time_points = len(cluster_sizes)

    if time_points == 0 or np.mean(cluster_sizes) == 0:
        mean_size = 0
        std_size = 0
        cv = np.nan
    else:
        mean_size = np.mean(cluster_sizes)
        std_size = np.std(cluster_sizes)
        cv = std_size / mean_size

    cv_records.append({
        "Jlf": jlf,
        "mu": mu,
        "PP": pp,
        "rep": rep,
        "Cluster ID": cluster_id,
        "Dominance": dom_type,
        "CV": cv if np.isfinite(cv) else np.nan,
        "Mean Size": mean_size,
        "Std Size": std_size,
        "Time Points": time_points
    })

# Create DataFrame
cv_df = pd.DataFrame(cv_records)

# -------------------------------
# Aggregate: mean CV by dominance
# -------------------------------
stability_summary = cv_df.groupby(["Jlf", "mu", "PP", "rep", "Dominance"]).agg({
    "CV": "mean",
    "Mean Size": "mean",
    "Std Size": "mean",
    "Time Points": "mean",   # Optional: shows avg lifespan of LD vs FD clusters
    "Cluster ID": "count"    # Number of clusters in each dominance category
}).rename(columns={"Cluster ID": "Num Clusters"}).reset_index()

# Optional: Pivot for easy comparison (LD vs FD per run)
pivot_df = stability_summary.pivot(
    index=["Jlf", "mu", "PP", "rep"],
    columns="Dominance",
    values="CV"
).reset_index()

# Save outputs
cv_per_cluster_path = "../../Results/Cluster/ClusterStability_perCluster.csv"
summary_path = "../../Results/Cluster/ClusterStability_by_Dominance.csv"

cv_df.to_csv(cv_per_cluster_path, index=False)
stability_summary.to_csv(summary_path, index=False)

print(f"Saved per-cluster CVs to: {cv_per_cluster_path}")
print(f"Saved dominance-wise CV summary to: {summary_path}")


/var/folders/jr/3tw1qcls4879795w5933hkj00000gn/T/ipykernel_80865/3144707752.py:6: DtypeWarning: Columns (11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Saved per-cluster CVs to: ../../Results/Cluster/ClusterStability_perCluster.csv
Saved dominance-wise CV summary to: ../../Results/Cluster/ClusterStability_by_Dominance.csv


## Merging/Splitting

In [34]:
import pandas as pd
import numpy as np

# --- Tunable thresholds ---
D_THRESH = 5.0     # max centroid distance to count as same
S_DIFF_MAX = 0.30  # max relative size change
DOWNSAMPLE_STEP = 1  # set to >1 to skip frames (e.g., 2 keeps every other MCS)

# Paths
clusters_path = "../../Results/Data/Clusters/ClusterComposition.csv"
best_path = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"  # optional

# Load
df = pd.read_csv(clusters_path, low_memory=False)
df = df[df["Total Cells"] > 0].copy()

num_cols = ["MCS","Jlf","mu","PP","rep","Cluster ID","Total Cells","Centroid_X","Centroid_Y"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=num_cols)

# Optional: focus only on selected parameter combos to speed up
try:
    best_df = pd.read_csv(best_path)
    for c in ["Jlf","mu","PP"]:
        best_df[c] = pd.to_numeric(best_df[c], errors="coerce")
    df = df.merge(best_df[["Jlf","mu","PP"]].drop_duplicates(), on=["Jlf","mu","PP"], how="inner")
except Exception:
    pass

events = []

for (jlf, mu, pp, rep), g in df.groupby(["Jlf","mu","PP","rep"]):
    g = g.sort_values(["MCS","Cluster ID"])
    times = sorted(g["MCS"].unique())
    if DOWNSAMPLE_STEP > 1:
        times = times[::DOWNSAMPLE_STEP]
    time_groups = {t: x for t, x in g.groupby("MCS")}
    
    for i in range(len(times)-1):
        t, t_next = times[i], times[i+1]
        if t not in time_groups or t_next not in time_groups:
            continue
        
        prev_df = time_groups[t][["Cluster ID","Total Cells","Centroid_X","Centroid_Y"]].reset_index(drop=True)
        next_df = time_groups[t_next][["Cluster ID","Total Cells","Centroid_X","Centroid_Y"]].reset_index(drop=True)
        if prev_df.empty or next_df.empty:
            continue
        
        # Vectorized pairing between prev and next
        P = prev_df.shape[0]; N = next_df.shape[0]
        p_ids = prev_df["Cluster ID"].to_numpy(dtype=int)
        n_ids = next_df["Cluster ID"].to_numpy(dtype=int)
        p_xy  = prev_df[["Centroid_X","Centroid_Y"]].to_numpy(float)
        n_xy  = next_df[["Centroid_X","Centroid_Y"]].to_numpy(float)
        p_sz  = prev_df["Total Cells"].to_numpy(float).reshape(P,1)
        n_sz  = next_df["Total Cells"].to_numpy(float).reshape(1,N)
        
        diff = p_xy[:,None,:] - n_xy[None,:,:]           # (P,N,2)
        dist = np.sqrt((diff**2).sum(axis=2))            # (P,N)
        denom = np.maximum(p_sz, n_sz)
        size_diff = np.abs(n_sz - p_sz) / np.maximum(denom, 1.0)
        
        match_mask = (dist <= D_THRESH) & (size_diff <= S_DIFF_MAX)  # (P,N)
        
        # Splits: row has >=2 matches
        row_counts = match_mask.sum(axis=1)
        split_rows = np.where(row_counts >= 2)[0]
        for r in split_rows:
            js = np.where(match_mask[r])[0]
            events.append({
                "type": "split",
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": rep,
                "time_from": t, "time_to": t_next,
                "from_cluster": int(p_ids[r]),
                "to_clusters": [int(n_ids[j]) for j in js],
                "avg_dist": float(dist[r, js].mean()),
                "avg_size_diff_frac": float(size_diff[r, js].mean()),
                "D_THRESH": D_THRESH, "S_DIFF_MAX": S_DIFF_MAX,
                "DOWNSAMPLE_STEP": DOWNSAMPLE_STEP
            })
        
        # Merges: column has >=2 matches
        col_counts = match_mask.sum(axis=0)
        merge_cols = np.where(col_counts >= 2)[0]
        for c in merge_cols:
            is_ = np.where(match_mask[:, c])[0]
            events.append({
                "type": "merge",
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": rep,
                "time_from": t, "time_to": t_next,
                "from_clusters": [int(p_ids[i]) for i in is_],
                "to_cluster": int(n_ids[c]),
                "avg_dist": float(dist[is_, c].mean()),
                "avg_size_diff_frac": float(size_diff[is_, c].mean()),
                "D_THRESH": D_THRESH, "S_DIFF_MAX": S_DIFF_MAX,
                "DOWNSAMPLE_STEP": DOWNSAMPLE_STEP
            })

events_df = pd.DataFrame(events)
events_df.to_csv("../../Results/Data/Clusters/cluster_merge_split_centroid.csv", index=False)
events_df.head()


,type,Jlf,mu,PP,rep,time_from,time_to,from_clusters,to_cluster,avg_dist,avg_size_diff_frac,D_THRESH,S_DIFF_MAX,DOWNSAMPLE_STEP,from_cluster,to_clusters
0,merge,0,27,0.0,9,650,660,"[3, 5]",4.0,3.015953,0.1000,5.0,0.3,1,NaN,NaN
1,merge,3,27,0.2,2,670,680,"[1, 4]",4.0,2.856476,0.0625,5.0,0.3,1,NaN,NaN
2,split,3,27,0.2,6,640,650,NaN,NaN,2.686067,0.0000,5.0,0.3,1,9.0,"[9, 11]"
3,split,3,27,0.2,6,650,660,NaN,NaN,1.994674,0.0000,5.0,0.3,1,11.0,"[9, 11]"
4,merge,3,27,0.2,6,650,660,"[9, 11]",9.0,2.406674,0.0000,5.0,0.3,1,NaN,NaN


In [16]:
import os
import math
import ast
import pandas as pd
import numpy as np

# =========================
# CONFIG (tune as needed)
# =========================
CLUSTERS_FILE = "../../Results/Data/Clusters/ClusterComposition.csv"
FINAL_MCS     = 700             # simulation final time
MAX_GAP       = 50              # how far ahead (in MCS) we’ll search for a successor
RADIUS_PX     = 12.0            # centroid distance threshold to flag a merge
SIZE_BUMP_FR  = 0.10            # target total size must increase by at least +10%
USE_MEMBER_POINTS = False       # set True to add an extra membership-overlap style check

# If you enable member checks, we count source member points that fall near the target
MEMBER_NEAR_RADIUS = 8.0
MIN_MEMBER_NEAR_FR = 0.25       # at least 25% of source members near target centroid


# =========================
# HELPERS
# =========================
def _safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def _euclid(x0, y0, x1, y1):
    try:
        return math.hypot(_safe_float(x1) - _safe_float(x0),
                          _safe_float(y1) - _safe_float(y0))
    except Exception:
        return np.inf

def _parse_member_list(s):
    """Parse a column like '[181.23, 182.1]' into a list of floats."""
    try:
        arr = ast.literal_eval(str(s))
        if isinstance(arr, (list, tuple)):
            return [ _safe_float(v, default=np.nan) for v in arr ]
    except Exception:
        pass
    return []

def _count_members_near_target(row_src, row_tgt, radius=MEMBER_NEAR_RADIUS):
    """Approximate overlap: count how many source member points lie within radius of target centroid."""
    cx = _safe_float(row_tgt.get("Centroid_X"))
    cy = _safe_float(row_tgt.get("Centroid_Y"))

    Lx = _parse_member_list(row_src.get("Member_Leader_X"))
    Ly = _parse_member_list(row_src.get("Member_Leader_Y"))
    Fx = _parse_member_list(row_src.get("Member_Follower_X"))
    Fy = _parse_member_list(row_src.get("Member_Follower_Y"))

    pts = list(zip(Lx, Ly)) + list(zip(Fx, Fy))
    if not pts:
        return 0, 0

    near = 0
    for (x, y) in pts:
        if np.isnan(x) or np.isnan(y):
            continue
        if math.hypot(x - cx, y - cy) <= radius:
            near += 1
    return near, len(pts)


# =========================
# CORE: detect merges
# =========================
def detect_merges(
    clusters_csv=CLUSTERS_FILE,
    final_mcs=FINAL_MCS,
    max_gap=MAX_GAP,
    radius_px=RADIUS_PX,
    size_bump_fr=SIZE_BUMP_FR,
    use_member_points=USE_MEMBER_POINTS,
    member_near_radius=MEMBER_NEAR_RADIUS,
    min_member_near_fr=MIN_MEMBER_NEAR_FR
):
    """
    Returns a DataFrame of termination events for clusters that did not reach final_mcs,
    with a heuristic classification 'LikelyMerged' when a plausible successor exists
    at the next available time (within max_gap MCS), near in space and larger in size.
    """

    if not os.path.exists(clusters_csv):
        raise FileNotFoundError(f"Cannot find file: {clusters_csv}")

    df = pd.read_csv(clusters_csv, low_memory=False)

    # Use only real clusters
    df = df[df["Total Cells"] > 0].copy()

    # Ensure numeric
    for c in ["Jlf","mu","PP","rep","MCS","Cluster ID","Total Cells",
              "Centroid_X","Centroid_Y","Leader Cells","Follower Cells"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Jlf","mu","PP","rep","Cluster ID","MCS","Total Cells","Centroid_X","Centroid_Y"])

    # Work per (Jlf, mu, PP, rep)
    group_keys = ["Jlf","mu","PP","rep"]
    results = []

    for (jlf, mu, pp, rep), g in df.groupby(group_keys, sort=False):
        g = g.sort_values(["Cluster ID","MCS"], kind="stable").reset_index(drop=True)

        # Quick index by time for successor search
        times = sorted(g["MCS"].unique())
        # For faster access: dict time -> rows at that time
        by_time = { t: sub.copy() for t, sub in g.groupby("MCS") }

        # For each cluster id in this replicate, detect its termination
        for cid, gc in g.groupby("Cluster ID", sort=False):
            gc = gc.sort_values("MCS", kind="stable")
            mcs_end   = int(gc["MCS"].max())
            row_final = gc[gc["MCS"] == mcs_end].iloc[-1]

            # If it reaches final_mcs, skip (it persists)
            if mcs_end >= final_mcs:
                continue

            # Find the next available time after mcs_end within max_gap
            next_times = [t for t in times if (t > mcs_end and (t - mcs_end) <= max_gap)]
            if not next_times:
                # No successor window to check
                results.append({
                    "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                    "SourceClusterID": int(cid),
                    "EndTime": mcs_end,
                    "SuccessorTime": np.nan,
                    "LikelyMerged": False,
                    "Reason": "No next time within gap",
                    "TargetClusterID": np.nan,
                    "CentroidDistance": np.nan,
                    "SizeChange": np.nan,
                    "NearMemberFrac": np.nan
                })
                continue

            # Check the very next time available (strictest)
            t_next = min(next_times)
            cand_df = by_time.get(t_next, pd.DataFrame())
            if cand_df.empty:
                results.append({
                    "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                    "SourceClusterID": int(cid),
                    "EndTime": mcs_end,
                    "SuccessorTime": np.nan,
                    "LikelyMerged": False,
                    "Reason": "No clusters at next time",
                    "TargetClusterID": np.nan,
                    "CentroidDistance": np.nan,
                    "SizeChange": np.nan,
                    "NearMemberFrac": np.nan
                })
                continue

            # Find nearest target by centroid
            sx, sy = row_final["Centroid_X"], row_final["Centroid_Y"]
            stot   = row_final["Total Cells"]

            cand_df = cand_df.copy()
            cand_df["dist"] = np.hypot(cand_df["Centroid_X"] - sx,
                                       cand_df["Centroid_Y"] - sy)
            cand_df = cand_df.sort_values("dist", kind="stable")
            tgt_row = cand_df.iloc[0]
            dist    = float(tgt_row["dist"])
            ttot    = float(tgt_row["Total Cells"])
            size_change = (ttot - float(stot)) / max(float(stot), 1e-9)

            near_frac = np.nan
            member_ok = True
            if use_member_points:
                near_cnt, total_cnt = _count_members_near_target(row_final, tgt_row, radius=member_near_radius)
                near_frac = (near_cnt / total_cnt) if total_cnt > 0 else np.nan
                member_ok = (not np.isnan(near_frac)) and (near_frac >= min_member_near_fr)

            ok_dist  = dist <= radius_px
            ok_size  = size_change >= size_bump_fr
            likely   = ok_dist and ok_size and member_ok

            reason_bits = []
            if not ok_dist:  reason_bits.append("far")
            if not ok_size:  reason_bits.append("no_size_bump")
            if use_member_points and not member_ok: reason_bits.append("low_member_near")
            reason_str = "ok" if likely else "|".join(reason_bits) if reason_bits else "unclear"

            results.append({
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                "SourceClusterID": int(cid),
                "EndTime": mcs_end,
                "SuccessorTime": int(t_next),
                "LikelyMerged": bool(likely),
                "Reason": reason_str,
                "TargetClusterID": int(tgt_row["Cluster ID"]),
                "CentroidDistance": dist,
                "SizeChange": size_change,
                "NearMemberFrac": near_frac
            })

    out = pd.DataFrame(results).sort_values(
        ["Jlf","mu","PP","rep","EndTime"], kind="stable"
    ).reset_index(drop=True)
    return out


merges = detect_merges(
    clusters_csv="../../Results/Data/Clusters/ClusterComposition.csv",
    final_mcs=700,
    max_gap=50,
    radius_px=12.0,
    size_bump_fr=0.10,
    use_member_points=False  # switch to True for the extra membership check
)

# Quick summary:
total_ended_early = (merges["EndTime"].notna()).sum()
likely_merged = merges["LikelyMerged"].sum()
print(f"Terminated before 700: {total_ended_early}")
print(f"Likely merged: {likely_merged}")

# Save if you want:
merges.to_csv("../../Results/Data/Clusters/ClusterMerge_Detections.csv", index=False)


Terminated before 700: 1214
Likely merged: 70


In [18]:
import os, math, ast
import pandas as pd
import numpy as np

# ==== CONFIG ====
CLUSTERS_FILE = "../../Results/Data/Clusters/ClusterComposition.csv"
BEST_FILE     = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"

FINAL_MCS     = 700      # simulation end
MAX_GAP       = 50       # search window (MCS) after a cluster ends
RADIUS_PX     = 12.0     # centroid distance threshold to call a merge
SIZE_BUMP_FR  = 0.10     # target must be at least +10% larger than source
USE_MEMBER_POINTS = False
MEMBER_NEAR_RADIUS = 8.0
MIN_MEMBER_NEAR_FR = 0.25

# ==== helpers ====
def _safe_float(x, default=np.nan):
    try: return float(x)
    except: return default

def _euclid(x0,y0,x1,y1):
    try: return math.hypot(_safe_float(x1)-_safe_float(x0), _safe_float(y1)-_safe_float(y0))
    except: return np.inf

def _parse_list(s):
    try:
        arr = ast.literal_eval(str(s))
        return arr if isinstance(arr,(list,tuple)) else []
    except: return []

def _count_members_near_target(row_src, row_tgt, radius=MEMBER_NEAR_RADIUS):
    cx, cy = _safe_float(row_tgt.get("Centroid_X")), _safe_float(row_tgt.get("Centroid_Y"))
    Lx, Ly = _parse_list(row_src.get("Member_Leader_X")), _parse_list(row_src.get("Member_Leader_Y"))
    Fx, Fy = _parse_list(row_src.get("Member_Follower_X")), _parse_list(row_src.get("Member_Follower_Y"))
    pts = list(zip(Lx,Ly)) + list(zip(Fx,Fy))
    if not pts: return 0,0
    near = sum(1 for (x,y) in pts if not (np.isnan(x) or np.isnan(y)) and math.hypot(x-cx,y-cy) <= radius)
    return near, len(pts)

def detect_merges_restricted_to_best(
    clusters_csv=CLUSTERS_FILE,
    best_csv=BEST_FILE,
    final_mcs=FINAL_MCS,
    max_gap=MAX_GAP,
    radius_px=RADIUS_PX,
    size_bump_fr=SIZE_BUMP_FR,
    use_member_points=USE_MEMBER_POINTS,
    member_near_radius=MEMBER_NEAR_RADIUS,
    min_member_near_fr=MIN_MEMBER_NEAR_FR
):
    # --- load best combos ---
    if not os.path.exists(best_csv):
        raise FileNotFoundError(f"Best file not found: {best_csv}")
    best = pd.read_csv(best_csv)
    for c in ["Jlf","mu","PP"]:
        best[c] = pd.to_numeric(best[c], errors="coerce")
    best = best.dropna(subset=["Jlf","mu","PP"]).drop_duplicates(subset=["Jlf","mu","PP"])
    best_keys = set(map(tuple, best[["Jlf","mu","PP"]].to_numpy()))

    # --- load clusters and filter to best combos ---
    if not os.path.exists(clusters_csv):
        raise FileNotFoundError(f"Clusters file not found: {clusters_csv}")
    df = pd.read_csv(clusters_csv, low_memory=False)
    df = df[df["Total Cells"] > 0].copy()
    for c in ["Jlf","mu","PP","rep","MCS","Cluster ID","Total Cells","Centroid_X","Centroid_Y"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Jlf","mu","PP","rep","MCS","Cluster ID","Total Cells","Centroid_X","Centroid_Y"])

    df = df[df[["Jlf","mu","PP"]].apply(tuple, axis=1).isin(best_keys)]
    if df.empty:
        return pd.DataFrame(columns=[
            "Jlf","mu","PP","rep","SourceClusterID","EndTime","SuccessorTime",
            "LikelyMerged","Reason","TargetClusterID","CentroidDistance","SizeChange","NearMemberFrac"
        ])

    # --- detect merges per (Jlf, mu, PP, rep) ---
    results = []
    for (jlf, mu, pp, rep), g in df.groupby(["Jlf","mu","PP","rep"], sort=False):
        g = g.sort_values(["Cluster ID","MCS"], kind="stable").reset_index(drop=True)
        times = sorted(g["MCS"].unique())
        by_time = { t: sub.copy() for t, sub in g.groupby("MCS") }

        for cid, gc in g.groupby("Cluster ID", sort=False):
            gc = gc.sort_values("MCS", kind="stable")
            mcs_end = int(gc["MCS"].max())
            row_final = gc[gc["MCS"] == mcs_end].iloc[-1]

            if mcs_end >= final_mcs:
                continue

            # next time within gap
            next_times = [t for t in times if (t > mcs_end and (t - mcs_end) <= max_gap)]
            if not next_times:
                results.append({
                    "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                    "SourceClusterID": int(cid),
                    "EndTime": mcs_end, "SuccessorTime": np.nan,
                    "LikelyMerged": False, "Reason": "No next time within gap",
                    "TargetClusterID": np.nan, "CentroidDistance": np.nan,
                    "SizeChange": np.nan, "NearMemberFrac": np.nan
                })
                continue

            t_next = min(next_times)
            cand_df = by_time.get(t_next, pd.DataFrame())
            if cand_df.empty:
                results.append({
                    "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                    "SourceClusterID": int(cid),
                    "EndTime": mcs_end, "SuccessorTime": np.nan,
                    "LikelyMerged": False, "Reason": "No clusters at next time",
                    "TargetClusterID": np.nan, "CentroidDistance": np.nan,
                    "SizeChange": np.nan, "NearMemberFrac": np.nan
                })
                continue

            sx, sy = row_final["Centroid_X"], row_final["Centroid_Y"]
            stot   = float(row_final["Total Cells"])

            cand_df = cand_df.copy()
            cand_df["dist"] = np.hypot(cand_df["Centroid_X"] - sx,
                                       cand_df["Centroid_Y"] - sy)
            cand_df = cand_df.sort_values("dist", kind="stable")
            tgt_row = cand_df.iloc[0]
            dist    = float(tgt_row["dist"])
            ttot    = float(tgt_row["Total Cells"])
            size_change = (ttot - stot) / max(stot, 1e-9)

            near_frac = np.nan
            member_ok = True
            if use_member_points:
                near_cnt, total_cnt = _count_members_near_target(row_final, tgt_row, radius=member_near_radius)
                near_frac = (near_cnt / total_cnt) if total_cnt > 0 else np.nan
                member_ok = (not np.isnan(near_frac)) and (near_frac >= min_member_near_fr)

            ok_dist = dist <= radius_px
            ok_size = size_change >= size_bump_fr
            likely  = ok_dist and ok_size and member_ok
            reason  = "ok" if likely else "|".join(
                [lbl for lbl, flag in [("far", not ok_dist),
                                       ("no_size_bump", not ok_size),
                                       ("low_member_near", use_member_points and not member_ok)]
                 if flag]) or "unclear"

            results.append({
                "Jlf": jlf, "mu": mu, "PP": pp, "rep": int(rep),
                "SourceClusterID": int(cid),
                "EndTime": mcs_end, "SuccessorTime": int(t_next),
                "LikelyMerged": bool(likely), "Reason": reason,
                "TargetClusterID": int(tgt_row["Cluster ID"]),
                "CentroidDistance": dist, "SizeChange": size_change,
                "NearMemberFrac": near_frac
            })

    out = pd.DataFrame(results).sort_values(["Jlf","mu","PP","rep","EndTime"], kind="stable").reset_index(drop=True)
    return out

merges_best = detect_merges_restricted_to_best(
    clusters_csv="../../Results/Data/Clusters/ClusterComposition.csv",
    best_csv="../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv",
    final_mcs=700,
    max_gap=50,
    radius_px=12.0,
    size_bump_fr=0.10,
    use_member_points=False
)

print("Terminated before 700:", merges_best.shape[0])
print("Likely merged:", merges_best["LikelyMerged"].sum())
merges_best.head()

merges_best.to_csv("../../Results/Data/Clusters/ClusterMerge.csv", index=False)


Terminated before 700: 114
Likely merged: 6


# Cluster Plots

### Phase Plots

In [1]:
from __future__ import annotations

import os
import glob
import math
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Iterable, Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle, ConnectionPatch
import matplotlib.cm as cm
import matplotlib.colors as mcolors


# ============================================================
# Configuration & Types
# ============================================================

PLOT_TYPES = {"bar", "box", "violin"}

@dataclass(frozen=True)
class PhaseDiagramConfig:
    # --- I/O ---
    csv_path: str
    save_path: Optional[str] = None  # file or directory; supports "{pp}" pattern
    # --- plotting mode ---
    plot_type: str = "bar"           # "bar" | "box" | "violin"
    y_col: str = "Total Cells"
    y_col_color: Optional[str] = None
    show_dominance: bool = True
    # --- labels ---
    y_label: Optional[str] = None
    color_label: Optional[str] = None
    show_ylabel_all: bool = False
    # --- grid axes (phase diagram) ---
    jlf_min: float = -5.0
    jlf_max: float = 5.0
    jlf_step: float = 1.0
    mu_vals: Optional[Iterable[float]] = None  # defaults to [0,3,...,30] pruned by data
    # --- PP selection ---
    pp_fixed: Optional[float] = None
    generate_each_pp: bool = True
    # --- fig/layout ---
    figsize: Tuple[float, float] = (24, 40)     # portrait because μ is x, Jlf is y
    panel_pad: Tuple[float, float] = (0.025, 0.04)  # fractional inset padding
    title_fontsize: int = 13
    tick_labelsize: int = 10
    no_data_fontsize: int = 10
    # --- colors ---
    leader_color: str = "tab:blue"
    follower_color: str = "tab:green"
    neutral_cmap: str = "viridis"
    add_colorbar: bool = True
    # --- scaling ---
    same_scale: bool = True

    # -----------------------------
    # Inset zoom settings
    # -----------------------------
    add_zoom_inset: bool = True
    inset_jlf: float = 2.0
    inset_mu: float = 30.0
    inset_title: str = "Zoomed in"
    inset_border_color: str = "#c62828"  # red highlight
    inset_border_lw: float = 3.0
    # Placement: if no empty panel exists, use this fallback (host axes coords)
    inset_fallback_box: Tuple[float, float, float, float] = (0.70, 0.58, 0.28, 0.30)
    # Size controls if an empty panel is available to host inset
    inset_scale: float = 2.2               # how much larger than a single small panel
    inset_max_size: Tuple[float, float] = (0.42, 0.42)  # cap (w,h) in host-axes coords
    # Dual subplot layout inside the inset
    inset_dual: bool = True                # if True: show [leaders/followers | total colored]
    inset_gap_frac: float = 0.04           # gap between the two subplots as fraction of inset width
    inset_cmap: str = "plasma"             # cmap for persistence-colored bars
    inset_colorbar: bool = False            # tiny colorbar inside the inset (right chart)
    inset_tick_labelsize: int = 16
    inset_axis_labelsize: int = 16
    inset_title_fontsize: int = 16

    # Column autodetection (for the inset)
    persist_col_candidates: Tuple[str, ...] = (
        "PersistenceTime", "Persistence Time", "persistence_time"
    )
    leader_ct_candidates: Tuple[str, ...] = (
        "Leader Count", "Leader Cells", "n_leader", "n_leaders", "LeaderCount"
    )
    follower_ct_candidates: Tuple[str, ...] = (
        "Follower Count", "Follower Cells", "n_follower", "n_followers", "FollowerCount"
    )
    leader_pct_candidates: Tuple[str, ...] = (
        "Leader %", "Leader Fraction", "Leader_Fraction", "LeaderFrac", "LeaderPct", "Leader_Percent"
    )

    def normalized(self) -> "PhaseDiagramConfig":
        """Return a copy with normalized/derived defaults filled."""
        pt = (self.plot_type or "bar").strip().lower()
        if pt not in PLOT_TYPES:
            raise ValueError(f"plot_type must be one of {sorted(PLOT_TYPES)} (got {self.plot_type!r})")
        return replace(
            self,
            plot_type=pt,
            y_label=self.y_label or self.y_col,
            color_label=self.color_label or (self.y_col_color if self.y_col_color else None),
        )


# ============================================================
# Plotter
# ============================================================

class PhaseDiagramPlotter:
    def __init__(self, cfg: PhaseDiagramConfig):
        self.cfg = cfg.normalized()

        # Will be set in load/prepare
        self.df_all: pd.DataFrame = pd.DataFrame()
        self.unique_pp: np.ndarray = np.array([])
        self.pp_list: List[float] = []

        # Color state
        self.base_cmap = mpl.colormaps.get_cmap(self.cfg.neutral_cmap)
        self.neutral_color = mcolors.to_hex(self.base_cmap(0.6))
        self.color_norm = None
        self.color_cmap = None

    # -----------------------------
    # Public API
    # -----------------------------
    def render_all(self) -> List[str]:
        self._load_and_validate()
        self._select_pp_list()
        outputs = []

        for pp_val in self.pp_list:
            df_pp = self._slice_pp(self.df_all, pp_val)
            df_pp = self._ensure_cluster_renumbering(df_pp)
            save_path = self._resolve_save_path(pp_val)
            out = self._plot_one_pp(df_pp, pp_val, save_path)
            outputs.append(out)

        if outputs:
            out_dir = os.path.dirname(outputs[-1]) or "."
            self._write_simple_index_html(out_dir)

        return outputs

    # -----------------------------
    # Data handling
    # -----------------------------
    def _load_and_validate(self) -> None:
        df = pd.read_csv(self.cfg.csv_path)
        needed = {"Jlf", "mu", "PP", self.cfg.y_col, "Dominance"}
        if self.cfg.y_col_color:
            needed.add(self.cfg.y_col_color)
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"Input file missing columns: {sorted(missing)}")

        for c in ["Jlf", "mu", "PP", self.cfg.y_col]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        if self.cfg.y_col_color:
            df[self.cfg.y_col_color] = pd.to_numeric(df[self.cfg.y_col_color], errors="coerce")

        df = df.dropna(subset=["Jlf", "mu", "PP", self.cfg.y_col]).copy()
        df["Dominance"] = df["Dominance"].astype(str).str.strip()

        self.df_all = df
        self.unique_pp = np.sort(df["PP"].unique())

    def _select_pp_list(self) -> None:
        if self.cfg.pp_fixed is not None:
            if not np.any(np.isclose(self.unique_pp, self.cfg.pp_fixed)):
                raise ValueError(f"PP={self.cfg.pp_fixed} not found. Found: {self.unique_pp}")
            self.pp_list = [self.cfg.pp_fixed]
            return

        if self.cfg.generate_each_pp:
            self.pp_list = list(self.unique_pp)
        else:
            if len(self.unique_pp) == 1:
                self.pp_list = [float(self.unique_pp[0])]
            else:
                raise ValueError(
                    f"Multiple PP values present: {self.unique_pp}. "
                    f"Pass pp_fixed or set generate_each_pp=True."
                )

    @staticmethod
    def _slice_pp(df: pd.DataFrame, pp_val: float) -> pd.DataFrame:
        return df[np.isclose(df["PP"], pp_val)].copy()

    # -----------------------------
    # Plot orchestration
    # -----------------------------
    def _plot_one_pp(self, df: pd.DataFrame, pp_val: float, final_save_path: str) -> str:
        cfg = self.cfg
        jlf_grid = np.arange(cfg.jlf_min, cfg.jlf_max + 1e-9, cfg.jlf_step)
        mu_vals = self._resolve_mu_values(df)

        # Host figure/axes
        fig = plt.figure(figsize=cfg.figsize)
        host = self._add_host_axes(fig, jlf_grid, mu_vals)  # swapped axes inside

        # Legend (dominance)
        if cfg.show_dominance:
            self._add_dominance_legend(fig)

        # Global y scale (if requested)
        ymin_glob, ymax_glob, max_bars = self._compute_global_scales(df)

        # Color mapping for secondary metric (if using neutral color mode)
        self._configure_secondary_color(df)

        # Panel geometry (normalized to host)
        cell_w = 1.0 / len(mu_vals)   # μ on x-axis
        cell_h = 1.0 / len(jlf_grid)  # Jlf on y-axis
        pad_x, pad_y = cfg.panel_pad
        panel_w = max(0.0, cell_w - pad_x)
        panel_h = max(0.0, cell_h - pad_y)

        # Index maps (μ -> x index, Jlf -> y index)
        mu_to_ix  = {float(v): i for i, v in enumerate(mu_vals)}
        jlf_to_iy = {float(v): i for i, v in enumerate(jlf_grid)}

        # Track panel positions & emptiness to place inset smartly
        panel_pos: Dict[Tuple[float, float], Tuple[float, float, float, float]] = {}
        panel_empty: Dict[Tuple[float, float], bool] = {}

        colorbar_added = False

        # Iterate the grid (μ across x, Jlf across y)
        for mu_val in mu_vals:
            for jlf_val in jlf_grid:
                ix = mu_to_ix[float(mu_val)]
                iy = jlf_to_iy[float(jlf_val)]
                is_first_col = (ix == 0)

                left   = ix * cell_w + pad_x * 0.5
                bottom = iy * cell_h + pad_y * 0.5
                panel_pos[(mu_val, jlf_val)] = (left, bottom, panel_w, panel_h)

                ax_in  = host.inset_axes([left, bottom, panel_w, panel_h], transform=host.transAxes)

                sub = df[(np.isclose(df["Jlf"], jlf_val)) & (np.isclose(df["mu"], mu_val))]
                is_empty = sub.empty
                panel_empty[(mu_val, jlf_val)] = is_empty
                if is_empty:
                    ax_in.set_axis_off()
                    host.text(mu_val, jlf_val, "No Cluster", ha="center", va="center",
                              fontsize=cfg.no_data_fontsize, color="gray")
                    continue

                # Panel draw
                colorbar_added = self._draw_panel(
                    ax=ax_in,
                    sub=sub,
                    add_colorbar=(cfg.add_colorbar and not cfg.show_dominance and not colorbar_added),
                    host=host,
                ) | colorbar_added

                # Common cosmetics
                if cfg.same_scale:
                    ax_in.set_ylim(ymin_glob, ymax_glob)
                    if cfg.plot_type == "bar":
                        ax_in.set_xlim(0.5, max_bars + 0.5)
                        if max_bars == 1:
                            ax_in.set_xticks([1])
                        else:
                            ticks = np.linspace(1, max_bars, num=min(max_bars, 4), dtype=int)
                            ax_in.set_xticks(np.unique(ticks))
                    elif cfg.plot_type in {"box", "violin"}:
                        if cfg.show_dominance:
                            ax_in.set_xlim(0.5, 2.5)
                            ax_in.set_xticks([1, 2])
                            if cfg.plot_type == "box":
                                ax_in.set_xticklabels(["Leader", "Follower"], fontsize=8)
                        else:
                            ax_in.set_xlim(0.5, 1.5)
                            ax_in.set_xticks([1])
                            if cfg.plot_type == "box":
                                ax_in.set_xticklabels(["Number of Clusters"], fontsize=8)

                ax_in.set_title(f"λ={mu_val:g}, Jlf={jlf_val:g}",
                                fontsize=cfg.title_fontsize, fontweight="bold", pad=2)
                self._bold_axis_labels(ax_in, size=11)

                for spine in ax_in.spines.values():
                    spine.set_linewidth(0.99)

                # y-label placement logic unchanged (based on first column)
                if self.cfg.show_ylabel_all or is_first_col:
                    ax_in.set_ylabel(self.cfg.y_label, fontsize=11, fontweight="bold")
                else:
                    ax_in.set_ylabel("") 
                                
                ax_in.tick_params(axis="both", labelsize=10)
                for label in ax_in.get_xticklabels() + ax_in.get_yticklabels():
                    label.set_fontweight("bold")

        # semantic legend for box/violin
        self._add_semantic_legend(fig)

        # Add the zoom inset (NEW dual view)
        if cfg.add_zoom_inset:
            try:
                self._add_zoom_inset(
                    df=df,
                    host=host,
                    panel_pos=panel_pos,
                    panel_empty=panel_empty,
                    cell_w=cell_w,
                    cell_h=cell_h,
                    ymin_glob=ymin_glob,
                    ymax_glob=ymax_glob,
                    max_bars=max_bars,
                )
            except Exception as e:
                print(f"[WARN] Could not add zoom inset: {e}")

        # Save
        out_dir = os.path.dirname(final_save_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        plt.savefig(final_save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        return final_save_path

    # -----------------------------
    # Helpers: data -> panels
    # -----------------------------
    def _ensure_cluster_renumbering(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.cfg.plot_type != "bar":
            return df

        if "ClusterID_renum" in df.columns:
            df["ClusterID_renum"] = pd.to_numeric(df["ClusterID_renum"], errors="coerce")
            return df

        if "ClusterUID" in df.columns:
            def _renumber(g: pd.DataFrame) -> pd.DataFrame:
                u = g[["ClusterUID"]].drop_duplicates().reset_index(drop=True)
                mapping = {uid: i + 1 for i, uid in enumerate(u["ClusterUID"])}
                g = g.copy()
                g["ClusterID_renum"] = g["ClusterUID"].map(mapping)
                return g
            return df.groupby(["Jlf", "mu", "PP"], group_keys=False).apply(_renumber)

        # Fallback if no UID
        def _fallback(g: pd.DataFrame) -> pd.DataFrame:
            u = g.drop_duplicates(subset=[self.cfg.y_col, "Dominance"]).reset_index(drop=True)
            u["_rid"] = np.arange(1, len(u) + 1, dtype=int)
            return g.merge(u, on=[self.cfg.y_col, "Dominance"], how="left") \
                    .rename(columns={"_rid": "ClusterID_renum"})
        return df.groupby(["Jlf", "mu", "PP"], group_keys=False).apply(_fallback)

    def _resolve_mu_values(self, df: pd.DataFrame) -> List[float]:
        if self.cfg.mu_vals is None:
            mu_list = list(range(0, 31, 3))  # 0,3,...,30
        else:
            mu_list = list(self.cfg.mu_vals)

        valid = []
        for mu in mu_list:
            if not df[np.isclose(df["mu"], mu)].empty:
                valid.append(float(mu))
        if not valid:
            raise ValueError("No valid μ values found in the dataset for the requested grid.")
        return valid

    def _compute_global_scales(self, df: pd.DataFrame) -> Tuple[float, float, int]:
        ymin_glob = float(df[self.cfg.y_col].min())
        ymax_glob = float(df[self.cfg.y_col].max())
        span = ymax_glob - ymin_glob
        if span <= 0:
            ymin_glob -= 0.5
            ymax_glob += 0.5
        else:
            pad = 0.05 * span
            ymin_glob -= pad
            ymax_glob += pad

        max_bars = 1
        if self.cfg.plot_type == "bar":
            if "ClusterID_renum" in df.columns:
                grp = df.dropna(subset=["ClusterID_renum"]).groupby(["Jlf", "mu"])["ClusterID_renum"].nunique()
                if not grp.empty:
                    max_bars = int(max(1, grp.max()))
        return ymin_glob, ymax_glob, max_bars

    def _configure_secondary_color(self, df: pd.DataFrame) -> None:
        if self.cfg.y_col_color and not self.cfg.show_dominance:
            ymin_c = float(df[self.cfg.y_col_color].min())
            ymax_c = float(df[self.cfg.y_col_color].max())
            self.color_norm = plt.Normalize(vmin=ymin_c, vmax=ymax_c)
            self.color_cmap = self.base_cmap
        else:
            self.color_norm = None
            self.color_cmap = None
            
    def _sort_and_enumerate_clusters(self, sub: pd.DataFrame) -> pd.DataFrame:
        cfg = self.cfg
        s = sub.copy()
        if "ClusterUID" in s.columns:
            s = s.sort_values([cfg.y_col, "ClusterUID"], ascending=[True, True], kind="stable").reset_index(drop=True)
        else:
            # Fallback: stable tie-breaker using a renumbering on (y_col, Dominance)
            u = s.drop_duplicates(subset=[cfg.y_col, "Dominance"]).reset_index(drop=True)
            u["_rid"] = np.arange(1, len(u) + 1, dtype=int)
            s = s.merge(u, on=[cfg.y_col, "Dominance"], how="left")
            s = s.sort_values([cfg.y_col, "_rid"], ascending=[True, True], kind="stable").reset_index(drop=True)
            s = s.drop(columns=["_rid"], errors="ignore")
        s["ClusterNum"] = np.arange(1, len(s) + 1, dtype=int)
        return s

    # -----------------------------
    # Helpers: drawing
    # -----------------------------
    def _bold_axis_labels(self, ax: plt.Axes, size: int | None = None) -> None:
        if size is not None:
            ax.xaxis.label.set_size(size)
            ax.yaxis.label.set_size(size)
        ax.xaxis.label.set_weight("bold")
        ax.yaxis.label.set_weight("bold")

    def _estimate_step(self, values: List[float], default_step: float) -> float:
        vals = np.array(sorted(values), dtype=float)
        if vals.size >= 2:
            diffs = np.diff(vals)
            diffs = diffs[diffs > 0]
            if diffs.size:
                return float(np.min(diffs))
        return default_step

    def _add_host_axes(self, fig: plt.Figure, jlf_grid: np.ndarray, mu_vals: List[float]) -> plt.Axes:
        """Create the big host axes with SWAPPED axes:
           x-axis  -> μ (Migration Coefficient)
           y-axis  -> Jlf (Contact Energy)
        """
        cfg = self.cfg

        host_rect = [0.11, 0.12, 0.72, 0.78]
        host = fig.add_axes(host_rect)
        host.set_facecolor("white")

        x_step = self._estimate_step(mu_vals, default_step=3.0)  # μ step
        y_step = cfg.jlf_step                                     # Jlf step

        x_margin = 0.5 * x_step
        y_margin = 0.5 * y_step
        host.set_xlim(min(mu_vals) - x_margin, max(mu_vals) + x_margin)
        host.set_ylim(cfg.jlf_min - y_margin, cfg.jlf_max + y_margin)

        for s in host.spines.values():
            s.set_visible(True)

        host.set_xticks(mu_vals)
        host.set_yticks(jlf_grid)

        axis_tick_fontsize = 20
        host.set_xticklabels([f"{v:g}" for v in mu_vals],
                             fontsize=axis_tick_fontsize, fontweight="bold")
        host.set_yticklabels([f"{v:g}" for v in jlf_grid],
                             fontsize=axis_tick_fontsize, fontweight="bold")
        host.tick_params(length=8)

        for x in mu_vals:
            host.axvline(x, color="lightgray", linewidth=0.6, linestyle="--", zorder=0)
        for y in jlf_grid:
            host.axhline(y, color="lightgray", linewidth=0.6, linestyle="--", zorder=0)

        # x-axis arrow/label for Migration Coefficient
        host.annotate(
            "",
            xy=(max(mu_vals) + 0.65 * x_step, cfg.jlf_min - 0.85 * y_step),
            xytext=(min(mu_vals) - 0.9 * x_step,  cfg.jlf_min - 0.85 * y_step),
            arrowprops=dict(arrowstyle="->", lw=2.2, color="black")
        )
        host.text((min(mu_vals) + max(mu_vals)) / 2.0, cfg.jlf_min - 0.85 * y_step,
                  r"Migration Coefficient", fontsize=28, fontweight="bold", ha="center", va="center")

        # y-axis arrow/label for Contact Energy
        host.annotate(
            "",
            xy=(min(mu_vals) - 0.9 * x_step, cfg.jlf_max + 0.65 * y_step),
            xytext=(min(mu_vals) - 0.9 * x_step, cfg.jlf_min - 0.9 * y_step),
            arrowprops=dict(arrowstyle="->", lw=2.2, color="black")
        )
        host.text(min(mu_vals) - 0.95 * x_step, (cfg.jlf_min + cfg.jlf_max) / 2.0,
                  r"Contact Energy", fontsize=28, fontweight="bold",
                  ha="center", va="center", rotation=90)

        return host

    def _add_dominance_legend(self, fig: plt.Figure) -> None:
        legend_font = font_manager.FontProperties(weight="bold", size=26)
        leader_proxy = Line2D([0], [0], color=self.cfg.leader_color, lw=6, label="Leader-dominated")
        follower_proxy = Line2D([0], [0], color=self.cfg.follower_color, lw=6, label="Follower-dominated")
        fig.legend(handles=[leader_proxy, follower_proxy],
                   loc="upper center", bbox_to_anchor=(0.5, 0.94),
                   ncol=2, prop=legend_font, frameon=False)

    def _add_semantic_legend(self, fig: plt.Figure) -> None:
        cfg = self.cfg
        if cfg.plot_type not in {"box", "violin"}:
            return

        sem_face = self.neutral_color if not cfg.show_dominance else "#9e9e9e"
        sem_edge = "black"
        handles, labels = [], []

        if cfg.plot_type == "violin":
            handles.append(Patch(facecolor=sem_face, edgecolor=sem_edge, alpha=0.6))
            labels.append("width ∝ density; height spans values")
            handles.append(Line2D([0], [0], linestyle="-", linewidth=2, color=sem_edge))
            labels.append("Median")
        elif cfg.plot_type == "box":
            handles.append(Patch(facecolor=sem_face, edgecolor=sem_edge, alpha=0.5))
            labels.append("IQR (Q1–Q3)")
            handles.append(Line2D([0], [0], linestyle="--", color=sem_edge))
            labels.append("Whiskers (range)")
            handles.append(Line2D([0], [0], linestyle="-", linewidth=2, color=sem_edge))
            labels.append("Median")
            handles.append(Line2D([0], [0], marker="o", linestyle="None", color=sem_edge))
            labels.append("Outliers")

        fig.legend(
            handles=handles,
            labels=labels,
            loc="upper right",
            bbox_to_anchor=(0.85, 0.95),
            ncol=2,
            frameon=True,
            fontsize=18,
        )

    def _draw_panel(self, ax: plt.Axes, sub: pd.DataFrame, add_colorbar: bool, host: plt.Axes) -> bool:
        cfg = self.cfg
        # ---- BAR ----
        if cfg.plot_type == "bar":
            sub = sub.dropna(subset=["ClusterID_renum"]).copy()
            sub = sub.sort_values(cfg.y_col, ascending=True, kind="stable").reset_index(drop=True)
            sub["ClusterID_renum"] = np.arange(1, len(sub) + 1)

            if cfg.show_dominance:
                L = sub[sub["Dominance"] == "Leader-dominated"]
                F = sub[sub["Dominance"] == "Follower-dominated"]
                if not L.empty: ax.bar(L["ClusterID_renum"], L[cfg.y_col], color=cfg.leader_color,  alpha=0.95)
                if not F.empty: ax.bar(F["ClusterID_renum"], F[cfg.y_col], color=cfg.follower_color, alpha=0.95)
            else:
                if cfg.y_col_color and (self.color_norm is not None):
                    bar_colors = self.color_cmap(self.color_norm(sub[cfg.y_col_color]))
                else:
                    bar_colors = self.neutral_color
                ax.bar(sub["ClusterID_renum"], sub[cfg.y_col], color=bar_colors, alpha=0.95)

                if add_colorbar and (self.color_norm is not None):
                    self._add_colorbar(host, cfg.color_label)
                    return True  # colorbar added

            ax.set_xlabel("Number of Clusters", fontsize=12)
            ax.tick_params(axis="both", labelsize=7)
            return False

        # ---- BOX ----
        if cfg.plot_type == "box":
            if cfg.show_dominance:
                L = sub[sub["Dominance"] == "Leader-dominated"][cfg.y_col].to_numpy()
                F = sub[sub["Dominance"] == "Follower-dominated"][cfg.y_col].to_numpy()
                data, ticks, colors = [], [], []
                if L.size: data.append(L); ticks.append("L"); colors.append(cfg.leader_color)
                if F.size: data.append(F); ticks.append("F"); colors.append(cfg.follower_color)
                if data:
                    bp = ax.boxplot(data, tick_labels=ticks, patch_artist=True)
                    for box, c in zip(bp["boxes"], colors):
                        box.set_facecolor(c); box.set_edgecolor("black")
                    for whisk in bp["whiskers"]: whisk.set_color("black")
                    for cap in bp["caps"]: cap.set_color("black")
                    for med in bp["medians"]: med.set_color("black")
            else:
                vals = sub[cfg.y_col].to_numpy()
                if vals.size:
                    bp = ax.boxplot([vals], tick_labels=["Number of Clusters"], patch_artist=True)
                    col = (self.color_cmap(self.color_norm(np.nanmean(sub[self.cfg.y_col_color])))
                           if (self.cfg.y_col_color and (self.color_norm is not None))
                           else self.neutral_color)
                    bp["boxes"][0].set_facecolor(col); bp["boxes"][0].set_edgecolor("black")
                    for whisk in bp["whiskers"]: whisk.set_color("black")
                    for cap in bp["caps"]: cap.set_color("black")
                    for med in bp["medians"]: med.set_color("black")

                    if add_colorbar and (self.color_norm is not None):
                        self._add_colorbar(host, cfg.color_label)
                        return True

            ax.tick_params(axis="both", labelsize=7)
            return False

        # ---- VIOLIN ----
        if cfg.plot_type == "violin":
            if cfg.show_dominance:
                L = sub[sub["Dominance"] == "Leader-dominated"][cfg.y_col].to_numpy()
                F = sub[sub["Dominance"] == "Follower-dominated"][cfg.y_col].to_numpy()
                data, positions, xticks = [], [], []
                pos = 1
                if L.size: data.append(L); positions.append(pos); xticks.append(("L", cfg.leader_color)); pos += 1
                if F.size: data.append(F); positions.append(pos); xticks.append(("F", cfg.follower_color))
                if data:
                    vp = ax.violinplot(data, positions=positions, showmedians=True, widths=0.9)
                    for body, (_, col) in zip(vp["bodies"], xticks):
                        body.set_facecolor(col); body.set_edgecolor("black"); body.set_alpha(0.9)
                    ax.set_xticks(positions)
                    ax.set_xticklabels([name for (name, _) in xticks], fontsize=8)
                    for part in ("cmins","cmaxes","cbars","cmedians"):
                        if part in vp: vp[part].set_color("black")
            else:
                vals = sub[cfg.y_col].to_numpy()
                if vals.size:
                    vp = ax.violinplot([vals], positions=[1], showmedians=True, widths=0.9)
                    col = (self.color_cmap(self.color_norm(np.nanmean(sub[self.cfg.y_col_color])))
                           if (self.cfg.y_col_color and (self.color_norm is not None))
                           else self.neutral_color)
                    for body in vp["bodies"]:
                        body.set_facecolor(col); body.set_edgecolor("black"); body.set_alpha(0.9)
                    ax.set_xticks([1]); ax.set_xticklabels(["Number of Clusters"], fontsize=8)
                    for part in ("cmins","cmaxes","cbars","cmedians"):
                        if part in vp: vp[part].set_color("black")

                    if add_colorbar and (self.color_norm is not None):
                        self._add_colorbar(host, cfg.color_label)
                        return True

            ax.tick_params(axis="both", labelsize=7)
            return False

        return False

    def _add_colorbar(self, host_ax: plt.Axes, label: Optional[str]) -> None:
        sm = cm.ScalarMappable(cmap=self.color_cmap, norm=self.color_norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=host_ax, orientation="vertical", fraction=0.02, pad=0.04)
        if label:
            cbar.set_label(label, fontsize=22, fontweight="bold")
        cbar.ax.tick_params(labelsize=18)
        for t in cbar.ax.get_yticklabels():
            t.set_fontweight("bold")

    # -----------------------------
    # Zoom inset helpers (NEW)
    # -----------------------------
    @staticmethod
    def _find_col(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
        """Case/space/underscore-insensitive match of candidate column names."""
        norm_map = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}
        for cand in candidates:
            k = cand.lower().replace(" ", "").replace("_", "")
            if k in norm_map:
                return norm_map[k]
        return None

    def _resolve_leader_follower_persistence(
        self, sub: pd.DataFrame
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, str]:
        """
        Returns (leaders, followers, total_size, persistence_vals, persistence_colname).
        Derives counts from leader % when needed.
        """
        cfg = self.cfg
        y = pd.to_numeric(sub[cfg.y_col], errors="coerce").fillna(0).to_numpy()

        leader_ct = self._find_col(sub, cfg.leader_ct_candidates)
        follower_ct = self._find_col(sub, cfg.follower_ct_candidates)
        leader_pct = self._find_col(sub, cfg.leader_pct_candidates)

        if leader_ct and follower_ct:
            L = pd.to_numeric(sub[leader_ct], errors="coerce").fillna(0).to_numpy()
            F = pd.to_numeric(sub[follower_ct], errors="coerce").fillna(0).to_numpy()
        else:
            if leader_pct is None:
                raise ValueError("Inset needs leader/follower counts or a leader percentage column.")
            pct = pd.to_numeric(sub[leader_pct], errors="coerce").fillna(0).to_numpy()
            pct = np.where(pct <= 1.0, pct * 100.0, pct)  # allow fraction inputs
            L = np.round(y * (pct / 100.0))
            F = y - L

        L = np.clip(L, 0, y)
        F = np.clip(F, 0, y - L)

        persist_col = self._find_col(sub, cfg.persist_col_candidates)
        if persist_col is None:
            raise ValueError(f"Inset needs a persistence column; tried {cfg.persist_col_candidates}")
        P = pd.to_numeric(sub[persist_col], errors="coerce").to_numpy()

        return L, F, y, P, persist_col

    def _add_zoom_inset(
        self,
        df: pd.DataFrame,
        host: plt.Axes,
        panel_pos: Dict[Tuple[float, float], Tuple[float, float, float, float]],
        panel_empty: Dict[Tuple[float, float], bool],
        cell_w: float,
        cell_h: float,
        ymin_glob: float,
        ymax_glob: float,
        max_bars: int,
    ) -> None:
        cfg = self.cfg
        target = (cfg.inset_mu, cfg.inset_jlf)
        if target not in panel_pos:
            raise ValueError(f"Target inset (μ={cfg.inset_mu}, Jlf={cfg.inset_jlf}) not in grid.")

        # Source (small) panel rect (in host axes coords)
        src_left, src_bot, src_w, src_h = panel_pos[target]

        # Choose a destination box: prefer any empty cell, else fallback
        dest_box = None
        for (mu_val, jlf_val), empty in panel_empty.items():
            if empty:
                left, bottom, w, h = panel_pos[(mu_val, jlf_val)]
                scale = cfg.inset_scale
                max_w, max_h = cfg.inset_max_size
                dest_w = min(w * scale, max_w)
                dest_h = min(h * scale, max_h)
                # center over that empty cell
                dest_left = max(0.0, min(1.0 - dest_w, left + 0.5 * (w - dest_w)))
                dest_bot  = max(0.0, min(1.0 - dest_h, bottom + 0.5 * (h - dest_h)))
                offset_x, offset_y = 0.04, 0.04   # host-axes units
                dest_left = min(1.0 - dest_w, max(0.0, dest_left + offset_x))
                dest_bot  = min(1.0 - dest_h, max(0.0, dest_bot  + offset_y))

                dest_box = (dest_left, dest_bot, dest_w, dest_h)
                break

        if dest_box is None:
            # Fallback position (in host axes coords)
            dest_box = cfg.inset_fallback_box

        dest_left, dest_bot, dest_w, dest_h = dest_box

        # Create the outer inset container
        ax_inset = host.inset_axes([dest_left, dest_bot, dest_w, dest_h], transform=host.transAxes)
        ax_inset.set_facecolor("white")
        # ax_inset.set_xticks([]); ax_inset.set_yticks([])

        
        sub_zoom = df[(np.isclose(df["Jlf"], cfg.inset_jlf)) & (np.isclose(df["mu"], cfg.inset_mu))]
        if sub_zoom.empty:
            ax_inset.text(0.5, 0.5, "No Data", ha="center", va="center", color="gray", transform=ax_inset.transAxes)
        else:
            if cfg.inset_dual:
                # --- NEW: sort & enumerate for consistent bar order ---
                sub_sorted = self._sort_and_enumerate_clusters(sub_zoom)

                inner_gap = 0.04  # fraction of inset width for padding
                left_w_frac  = (1.0 - inner_gap) * 0.5
                right_w_frac = (1.0 - inner_gap) * 0.5
                left_left = 0.0
                right_left = left_w_frac + inner_gap

                axL = ax_inset.inset_axes([left_left, 0.0, left_w_frac, 1.0])
                axR = ax_inset.inset_axes([right_left, 0.0, right_w_frac, 1.0])

                # --- Left: Stacked Leaders/Followers (sorted order) ---
                L, F, Y, P, persist_col = self._resolve_leader_follower_persistence(sub_sorted)
                x = sub_sorted["ClusterNum"].to_numpy()
                axL.bar(x, F, color=self.cfg.follower_color, alpha=0.95, label="Followers")
                axL.bar(x, L, bottom=F, color=self.cfg.leader_color, alpha=0.95, label="Leaders")
                axL.set_xlim(0.5, len(x) + 0.5)
                axL.set_xlabel("Clusters", fontsize=cfg.inset_axis_labelsize, fontweight="bold")
                axL.set_ylabel("Size",     fontsize=cfg.inset_axis_labelsize, fontweight="bold")
                axL.tick_params(axis="both", labelsize=cfg.inset_tick_labelsize)
                for t in axL.get_xticklabels() + axL.get_yticklabels():
                    t.set_fontweight("bold")
                axL.legend(frameon=False, fontsize=cfg.inset_tick_labelsize-1, loc="upper left")

                cmap = mpl.colormaps.get_cmap(cfg.inset_cmap)
                norm = plt.Normalize(vmin=np.nanmin(P), vmax=np.nanmax(P))
                colors = cmap(norm(P))
                axR.bar(x, Y, color=colors, alpha=0.95)
                axR.set_xlim(0.5, len(x) + 0.5)
                axR.set_xlabel("Clusters", fontsize=cfg.inset_axis_labelsize, fontweight="bold")
                axR.set_ylabel("Size",     fontsize=cfg.inset_axis_labelsize, fontweight="bold")
                axR.tick_params(axis="both", labelsize=cfg.inset_tick_labelsize)
                for t in axR.get_xticklabels() + axR.get_yticklabels():
                    t.set_fontweight("bold")

                ax_inset.set_title(f"{cfg.inset_title}: λ={cfg.inset_mu:g}, Jlf={cfg.inset_jlf:g}",
                                fontsize=cfg.inset_title_fontsize, fontweight="bold", pad=2)
            else:
                # Single-panel fallback unchanged...
                _ = self._draw_panel(ax=ax_inset, sub=sub_zoom, add_colorbar=False, host=host)
                if self.cfg.same_scale:
                    ax_inset.set_ylim(ymin_glob, ymax_glob)
                    if self.cfg.plot_type == "bar":
                        ax_inset.set_xlim(0.5, max_bars + 0.5)
                self._bold_axis_labels(ax_inset, size=cfg.inset_axis_labelsize)
                ax_inset.tick_params(axis="both", labelsize=cfg.inset_tick_labelsize)
                for t in ax_inset.get_xticklabels() + ax_inset.get_yticklabels():
                    t.set_fontweight("bold")

                # ax_inset.set_title(
                #     f"{cfg.inset_title}: λ={cfg.inset_mu:g}, Jlf={cfg.inset_jlf:g}",
                #     fontsize=cfg.inset_title_fontsize, fontweight="bold", pad=2
                # )        

        
        rect = Rectangle(
            (src_left, src_bot), src_w, src_h,
            transform=host.transAxes, fill=False,
            ec=cfg.inset_border_color, lw=cfg.inset_border_lw
        )
        host.add_patch(rect)
        
        con = ConnectionPatch(
            xyA=(src_left + src_w, src_bot + src_h), coordsA=host.transAxes,
            xyB=(dest_left, dest_bot),                 coordsB=host.transAxes,
            arrowstyle="->",
            mutation_scale=16,
            shrinkA=2, shrinkB=2,
            lw=2.0,
            color=self.cfg.inset_border_color,
        )
        con.set_linestyle((0, (2.0, 3.0)))  
        host.add_artist(con)


    # -----------------------------
    # Helpers: saving, paths, html
    # -----------------------------
    def _resolve_save_path(self, pp_val: float) -> str:
        cfg = self.cfg
        y_clean = cfg.y_col.replace(" ", "")
        default_dir = "../../Results/Data/Clusters/phaseplots"
        if cfg.save_path is None:
            out_dir = default_dir
            os.makedirs(out_dir, exist_ok=True)
            return os.path.join(out_dir, f"phase_{cfg.plot_type}_{y_clean}_PP{pp_val:g}.png")

        if os.path.isdir(cfg.save_path) or str(cfg.save_path).endswith(os.sep):
            out_dir = cfg.save_path
            os.makedirs(out_dir, exist_ok=True)
            return os.path.join(out_dir, f"phase_{cfg.plot_type}_{y_clean}_PP{pp_val:g}.png")

        root, ext = os.path.splitext(cfg.save_path)
        if "{pp}" in cfg.save_path:
            out_path = cfg.save_path.format(pp=f"{pp_val:g}")
            out_dir = os.path.dirname(out_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)
            return out_path

        if len(self.pp_list) > 1:
            out_dir = os.path.dirname(cfg.save_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)
            return f"{root}_PP{pp_val:g}{ext}"

        out_dir = os.path.dirname(cfg.save_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        return cfg.save_path

    @staticmethod
    def _write_simple_index_html(folder: str) -> None:
        pngs = sorted(glob.glob(os.path.join(folder, "*.png")))
        if not pngs:
            return
        with open(os.path.join(folder, "index.html"), "w") as f:
            f.write("<html><body><h1>Phase Plots</h1>\n")
            for p in pngs:
                name = os.path.basename(p)
                f.write(f"<div style='margin:10px 0'><h3>{name}</h3>"
                        f"<img src='{name}' style='max-width:100%;'></div>\n")
            f.write("</body></html>")


# ============================================================
# Convenience Functional Wrapper (API parity)
# ============================================================

def plot_phase_diagram_arrowed(
    csv_path: str,
    plot_type: str = "bar",
    y_col: str = "Total Cells",
    y_col_color: Optional[str] = None,
    y_label: Optional[str] = None,
    color_label: Optional[str] = None,
    show_ylabel_all: bool = False,
    pp_fixed: Optional[float] = None,
    jlf_min: float = -5.0,
    jlf_max: float = 5.0,
    jlf_step: float = 1.0,
    mu_vals: Optional[Iterable[float]] = None,
    figsize: Tuple[float, float] = (30, 32),
    panel_frac: Tuple[float, float] = (0.12, 0.12),  
    panel_pad: Tuple[float, float] = (0.025, 0.04),
    title_fontsize: int = 13,
    tick_labelsize: int = 10,
    no_data_fontsize: int = 10,
    leader_color: str = "tab:blue",
    follower_color: str = "tab:orange",
    save_path: Optional[str] = None,
    same_scale: bool = True,
    generate_each_pp: bool = True,
    show_dominance: bool = True,
    neutral_cmap: str = "viridis",
    add_colorbar: bool = True,
    # ---- inset tuning (NEW) ----
    add_zoom_inset: bool = True,
    inset_mu: float = 30.0,
    inset_jlf: float = 2.0,
    inset_dual: bool = True,
    inset_scale: float = 2.2,
    inset_max_size: Tuple[float, float] = (0.42, 0.42),
    inset_fallback_box: Tuple[float, float, float, float] = (0.70, 0.58, 0.28, 0.30),
) -> List[str]:
    cfg = PhaseDiagramConfig(
        csv_path=csv_path,
        save_path=save_path,
        plot_type=plot_type,
        y_col=y_col,
        y_col_color=y_col_color,
        show_dominance=show_dominance,
        y_label=y_label,
        color_label=color_label,
        show_ylabel_all=show_ylabel_all,
        jlf_min=jlf_min,
        jlf_max=jlf_max,
        jlf_step=jlf_step,
        mu_vals=mu_vals,
        pp_fixed=pp_fixed,
        generate_each_pp=generate_each_pp,
        figsize=figsize,
        panel_pad=panel_pad,
        title_fontsize=title_fontsize,
        tick_labelsize=tick_labelsize,
        no_data_fontsize=no_data_fontsize,
        leader_color=leader_color,
        follower_color=follower_color,
        neutral_cmap=neutral_cmap,
        add_colorbar=add_colorbar,
        same_scale=same_scale,
        # inset config
        add_zoom_inset=add_zoom_inset,
        inset_mu=inset_mu,
        inset_jlf=inset_jlf,
        inset_dual=inset_dual,
        inset_scale=inset_scale,
        inset_max_size=inset_max_size,
        inset_fallback_box=inset_fallback_box,
    )
    plotter = PhaseDiagramPlotter(cfg)
    return plotter.render_all()


# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    main_y  = "Total Cells"            # e.g., "Leader %", "Centroid Displacement", "Velocity"
    color_y = "PersistenceTime"        # e.g., "ClusterSizeCV_over_time", "Phenotypic Index", "Entropy"

    out_dir = f"../../Results/Plots/cluster_plots/PhasePlots/phaseplots_{main_y.replace(' ', '')}/"

    outputs = plot_phase_diagram_arrowed(
        csv_path="../../Results/Data/Clusters/cluster_detailed.csv",
        plot_type="bar",                         # "bar" | "box" | "violin"
        y_col=main_y,
        y_col_color=color_y,
        y_label="Cluster Size",
        color_label="Persistence Time",
        show_ylabel_all=True,
        pp_fixed=0.9,
        generate_each_pp=True,
        same_scale=True,
        show_dominance=False,                    
        neutral_cmap="plasma",
        add_colorbar=True,
        save_path=out_dir,
        # inset knobs you can tweak:
        add_zoom_inset=True,
        inset_mu=30.0, inset_jlf=2.0,
        inset_dual=False,                # two-panel inset
        inset_scale=3.0,                 # bigger inset
        inset_max_size=(0.48, 0.48),     # allow it to grow larger
        inset_fallback_box=(0.80, 0.75, 0.32, 0.36),
    )

    print("Saved:")
    for p in outputs:
        print("  -", p)


Saved:
  - ../../Results/Plots/cluster_plots/PhasePlots/phaseplots_TotalCells/phase_bar_TotalCells_PP0.9.png


### Bubble Plots

In [25]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from IPython.display import display, clear_output
import ipywidgets as widgets

# -----------------------------
# Settings
# -----------------------------
CSV_PATH = "../../Results/Data/Clusters/cluster_detailed.csv"
SAVE_DIR = "../../Results/Data/Clusters/BubblePlots"  
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Column helpers
# -----------------------------
def _normalize_colname(c: str) -> str:
    c = str(c).strip()
    c = c.replace("-", "_").replace(" ", "_")
    return "".join(ch.lower() for ch in c)

def _build_norm_map(columns):
    return {orig: _normalize_colname(orig) for orig in columns}

def _find_col(columns, candidates):
    norm_map = _build_norm_map(columns)
    cand_norm = [_normalize_colname(c) for c in candidates]
    for orig, norm in norm_map.items():
        if norm in cand_norm:
            return orig
    for orig, norm in norm_map.items():
        if any(c in norm for c in cand_norm):
            return orig
    return None

# -----------------------------
# Load data
# -----------------------------
assert os.path.exists(CSV_PATH), f"CSV not found: {CSV_PATH}"
df = pd.read_csv(CSV_PATH)

col_jlf = _find_col(df.columns, ["Jlf", "J_lf", "JLF", "j_lf"])
col_mu  = _find_col(df.columns, ["mu", "Mu"])
col_size = _find_col(df.columns, ["Total Cells", "Total_Cells", "cluster size", "cells", "n_cells", "size"])
col_persist = _find_col(df.columns, ["Persistence time", "Persistence_time", "persistence", "persist_time", "persistenceTime"])
col_pp = _find_col(df.columns, ["PP", "pp", "proliferative_probability", "proliferation", "proliferation_probability"])

missing = []
if not col_jlf: missing.append("Jlf")
if not col_mu: missing.append("mu")
if not col_size: missing.append("Total Cells (cluster size)")
if not col_persist: missing.append("Persistence time")

if missing:
    raise ValueError(f"Missing required columns: {', '.join(missing)}")

cols = {
    "Jlf": col_jlf,
    "mu": col_mu,
    "Total Cells": col_size,
    "Persistence time": col_persist,
    "PP": col_pp,
}

df = df.dropna(subset=[cols["Jlf"], cols["mu"], cols["Total Cells"], cols["Persistence time"]]).copy()

# -----------------------------
# Plot helpers
# -----------------------------
def _scale_sizes(series: pd.Series, min_size=50, max_size=1500) -> np.ndarray:
    s = series.astype(float)
    if len(s) == 0:
        return np.array([])
    if math.isclose(float(s.max()), float(s.min())):
        return np.full_like(s.astype(float), (min_size + max_size) / 2.0, dtype=float)
    s_scaled = (s - s.min()) / (s.max() - s.min())
    return min_size + s_scaled * (max_size - min_size)

def make_bubble_plot(filtered_df: pd.DataFrame, cmap: str = "plasma", title_suffix: str = ""):
    fig, ax = plt.subplots(figsize=(9, 6.5))
    sizes = _scale_sizes(filtered_df[cols["Total Cells"]])
    sc = ax.scatter(
        filtered_df[cols["Jlf"]].astype(float),
        filtered_df[cols["mu"]].astype(float),
        s=sizes,
        c=filtered_df[cols["Persistence time"]].astype(float),
        cmap=cmap,
        alpha=0.8,
        edgecolors="k",
        linewidths=0.3,
        marker="o",
    )

    title = f"Bubble Plot: Jlf vs mu | Size = {'Cluster size' if cols['Total Cells']=='Total Cells' else cols['Total Cells']}; Color = {cols['Persistence time']}"
    if title_suffix:
        title = f"{title} | {title_suffix}"
    ax.set_title(title, fontsize=16, fontweight="bold")   
    
    ax.set_xlabel("Contact Energy", fontsize=14, fontweight="bold")
    ax.set_ylabel("Migration Coefficient", fontsize=14, fontweight="bold")
    

    ax.tick_params(axis="both", labelsize=12)  # axis tick values
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

    cbar = plt.colorbar(sc)
    cbar.set_label(cols["Persistence time"] or "Persistence time", fontsize=14, fontweight="bold")
    cbar.ax.tick_params(labelsize=12)
    for label in cbar.ax.get_yticklabels():
        label.set_fontweight("bold")

    #  -----------------------------
    # Bubble size legend
    # -----------------------------
    quantiles = [0.0, 0.5, 0.8, 0.9, 0.95, 1.0]
    example_sizes = np.quantile(filtered_df[cols["Total Cells"]], quantiles)
    example_sizes = np.unique(example_sizes)  
    example_sizes = [float(s) for s in example_sizes]
    scaled_sizes = _scale_sizes(pd.Series(example_sizes),
                                min_size=50, max_size=1000)
    
    handles = [plt.scatter([], [], s=s, color="gray", alpha=0.6, edgecolors="k") 
               for s in scaled_sizes]
    labels = [f"{int(val)} cells" for val in example_sizes]
    legend = ax.legend(handles, labels, title="Cluster Size",
                    scatterpoints=1, frameon=True, fontsize=11,
                    title_fontsize=12, loc="center left",
                    bbox_to_anchor=(0.05, 0.3))  
    ax.add_artist(legend)
    ax.grid(True, linestyle=":")
    plt.tight_layout()
    return fig, ax

# -----------------------------
# Widgets
# -----------------------------
if cols["PP"] and cols["PP"] in df.columns:
    unique_pp = df[cols["PP"]].dropna().unique().tolist()
    try:
        unique_pp = sorted(unique_pp, key=lambda x: float(x))
    except Exception:
        unique_pp = sorted(map(str, unique_pp))
    pp_options = [("All", "ALL")] + [(str(v), v) for v in unique_pp]
else:
    pp_options = [("All", "ALL")]

cmap_options = ["plasma", "viridis", "inferno", "magma", "cividis", "turbo"]

pp_dd = widgets.Dropdown(options=pp_options, value="ALL", description="PP:", layout=widgets.Layout(width="220px"))
cmap_dd = widgets.Dropdown(options=cmap_options, value="plasma", description="Colormap:", layout=widgets.Layout(width="220px"))
out = widgets.Output()

def render(*_):
    with out:
        clear_output(wait=True)
        if pp_dd.value == "ALL" or not cols["PP"] or cols["PP"] not in df.columns:
            filtered = df
            suffix = f"(PP = ALL)"
            if col_size == "Total Cells":
                fname = f"bubble_ALL_ClusterSize.png"
            else:
                fname = f"bubble_ALL_{col_size}.png"
        else:
            mask = df[cols["PP"]] == pp_dd.value
            filtered = df[mask]
            suffix = f"(PP = {pp_dd.value})"
            if col_size == "Total Cells":
                fname = fname = f"bubble_PP{pp_dd.value}__ClusterSize.png"
            else:
                fname = fname = f"bubble_PP{pp_dd.value}_{col_size}.png"
            

        if filtered.empty:
            print("No rows match the current filter.")
            return

        fig, ax = make_bubble_plot(filtered, cmap=cmap_dd.value, title_suffix=suffix)
        display(fig)

        fname = (
            fname.replace(" ", "_")
                 .replace("(", "")
                 .replace(")", "")
                 .replace("__", "_")
        )
        out_path = os.path.join(SAVE_DIR, fname)
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        print(f"Plot saved to {out_path}")

        plt.close(fig)

pp_dd.observe(render, names="value")
cmap_dd.observe(render, names="value")

display(widgets.HBox([pp_dd, cmap_dd]))
render()
display(out)


Output()

In [ ]:
import glob
out_dir = "../../Results/Data/Clusters/BubblePlots"  
pngs = sorted(glob.glob(os.path.join(out_dir, "*.png")))
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "index.html"), "w") as f:
    f.write("<html><body><h1>Phase Plots</h1>\n")
    for p in pngs:
        name = os.path.basename(p)
        f.write(f"<div style='margin:10px 0'><h3>{name}</h3><img src='{name}' style='max-width:100%;'></div>\n")
    f.write("</body></html>")
print(f"Wrote index.html in {out_dir}")

Wrote index.html in ../../Results/Data/Clusters/BubblePlots


# Cluster TraJectory

### Tracks

In [ ]:
import os
import math
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import matplotlib
# matplotlib.use("Agg") 
import gc

# =============================
# CONFIG (edit paths as needed)
# =============================
COMPOSITION_CSV = "../../Results/Data/Clusters/ClusterComposition_1_Enriched.csv"
PARAM_LIST_CSV  = "../../Results/Data/Clusters/best_clusters_per_jlf_mu.csv"

OUT_DIR         = Path("../../Results/Data/Clusters/Trajectory")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# per-panel sizing; figure size scales with number of reps
BASE_PANEL_SIZE = (6.0, 5.2)   # (width, height) inches per subplot panel
MAX_NCOLS       = 3            # max panels per row
CMAP_NAME       = "viridis"    # for MCS dots

LINE_WIDTH      = 1.8
MARKER_SIZE     = 22
SHOW_EQUAL_ASPECT = False      # True if x/y must be metric-equal
LABEL_FRAME_ORDER = False      # If True and "Cluster ID" exists, label each point with its per-frame rank

# --- Shared axis scale across all panels in a figure (full trajectory visible) ---
MIN_X_SPAN = 50.0  # minimum width of x-axis window in data units
MIN_Y_SPAN = 5.0   # minimum height of y-axis window in data units
PAD_FRAC   = 0.02  # 2% padding added around data when data span exceeds minimum

# =============================
# Utilities
# =============================
def color_cycle(n):
    cmaps = ["tab10", "tab20", "tab20b", "tab20c", "Set3", "Accent", "Dark2", "Paired"]
    cols, idx = [], 0
    while len(cols) < n:
        cmap = plt.get_cmap(cmaps[idx % len(cmaps)])
        steps = 20 if "tab20" in cmap.name else 10
        for i in range(steps):
            if len(cols) >= n:
                break
            cols.append(cmap(i / max(1, steps - 1)))
        idx += 1
    return cols[:n]

def safe_name(x):
    s = f"{float(x):g}"
    return s.replace(".", "p")

def read_param_list_or_fallback(comp_df: pd.DataFrame, path: str) -> pd.DataFrame:
    try:
        pl = pd.read_csv(path)
        for c in ["Jlf","mu","PP"]:
            if c not in pl.columns:
                raise ValueError("param list missing required columns")
        pl = pl.dropna(subset=["Jlf","mu","PP"]).copy()
        for c in ["Jlf","mu","PP"]:
            pl[c] = pd.to_numeric(pl[c], errors="coerce")
        pl = pl.dropna(subset=["Jlf","mu","PP"]).copy()
        if len(pl) > 0:
            return pl[["Jlf","mu","PP"]].drop_duplicates().reset_index(drop=True)
    except Exception:
        pass

    # fallback
    tmp = comp_df[["Jlf","mu","PP"]].dropna().drop_duplicates().reset_index(drop=True)
    for c in ["Jlf","mu","PP"]:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
    tmp = tmp.dropna().reset_index(drop=True)
    return tmp

# =============================
# Track builder (using Cluster UID)
# =============================
def build_tracks_df_from_uid(df, jlf, mu, pp, rep):
    sub = df[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp) & (df["rep"] == rep)].copy()
    if sub.empty:
        return pd.DataFrame(columns=[
            "MCS","TrackID","centroid_x_fix","centroid_y_fix","Cluster ID",
            "HasEventHere","EventHereType","HasEventNext","EventNextType"
        ])

    # Required columns check
    need = ["MCS","Cluster UID","Centroid_X","Centroid_Y"]
    missing = [c for c in need if c not in sub.columns]
    if missing:
        raise ValueError(f"Missing required columns in composition data: {missing}")

    # Sort by lifetime (UID, then MCS)
    sub = sub.sort_values(["Cluster UID","MCS"]).reset_index(drop=True)

    # Rename to tracking schema
    out = sub.rename(columns={
        "Cluster UID": "TrackID",
        "Centroid_X": "centroid_x_fix",
        "Centroid_Y": "centroid_y_fix",
    })

    keep_cols = ["MCS","TrackID","centroid_x_fix","centroid_y_fix"]
    # pass-through per-frame ID if present (for optional labeling)
    if "Cluster ID" in out.columns:
        keep_cols.append("Cluster ID")
    # pass-through event columns if present
    for evcol in ["HasEventHere","EventHereType","HasEventNext","EventNextType"]:
        if evcol in out.columns:
            keep_cols.append(evcol)

    out = out[keep_cols].copy()
    # coerce numeric
    for c in ["MCS","TrackID","centroid_x_fix","centroid_y_fix"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # Normalize event columns presence
    if "HasEventHere" not in out.columns:
        out["HasEventHere"] = False
    if "HasEventNext" not in out.columns:
        out["HasEventNext"] = False
    if "EventHereType" not in out.columns:
        out["EventHereType"] = np.nan
    if "EventNextType" not in out.columns:
        out["EventNextType"] = np.nan

    return out

# =============================
# Plotting: one figure per (Jlf, mu, PP), one panel per rep
# =============================
def plot_param_all_reps(df, jlf, mu, pp, out_dir=OUT_DIR, idx=None):
    # find reps available for this param triple
    reps = sorted(df[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp)]["rep"].dropna().unique())
    if not reps:
        return None

    # Build per-rep tracks
    rep_tracks = {}
    for rep in reps:
        trdf = build_tracks_df_from_uid(df, jlf, mu, pp, rep)
        rep_tracks[rep] = trdf

    # If nothing to plot, exit early
    xs = []
    for t in rep_tracks.values():
        if not t.empty:
            xs.append(t[["centroid_x_fix","centroid_y_fix"]])
    if not xs:
        return None

    # Color normalization over all MCS values across all reps for this param triple
    all_mcs = pd.concat([t[["MCS"]] for t in rep_tracks.values() if not t.empty], axis=0, ignore_index=True)
    mcs_min, mcs_max = float(all_mcs["MCS"].min()), float(all_mcs["MCS"].max())
    norm = Normalize(vmin=mcs_min, vmax=mcs_max)
    sm = ScalarMappable(norm=norm, cmap=plt.get_cmap(CMAP_NAME))

    # ----- SAME SCALE & FULL TRAJECTORY (per-figure) -----
    all_xy = pd.concat(xs, axis=0, ignore_index=True)
    x_min, x_max = float(all_xy["centroid_x_fix"].min()), float(all_xy["centroid_x_fix"].max())
    y_min, y_max = float(all_xy["centroid_y_fix"].min()), float(all_xy["centroid_y_fix"].max())
    data_x_span = max(x_max - x_min, 0.0)
    data_y_span = max(y_max - y_min, 0.0)
    eff_x_span = max(MIN_X_SPAN, data_x_span * (1.0 + PAD_FRAC))
    eff_y_span = max(MIN_Y_SPAN, data_y_span * (1.0 + PAD_FRAC))
    x_center = 0.5 * (x_min + x_max) if data_x_span > 0 else x_min
    y_center = 0.5 * (y_min + y_max) if data_y_span > 0 else y_min
    xlim = (x_center - eff_x_span / 2.0, x_center + eff_x_span / 2.0)
    ylim = (y_center - eff_y_span / 2.0, y_center + eff_y_span / 2.0)

    # subplot grid
    n_rep = len(reps)
    ncols = min(MAX_NCOLS, n_rep)
    nrows = int(math.ceil(n_rep / ncols))
    figsize = (BASE_PANEL_SIZE[0] * ncols, BASE_PANEL_SIZE[1] * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True, squeeze=False)
    flat_axes = axes.ravel()

    # For a clean legend across panels, we’ll add basic handles (tracks + event markers) per panel
    scatter_for_cbar = None

    for ax, rep in zip(flat_axes, reps):
        trdf = rep_tracks[rep]
        if trdf.empty:
            ax.set_title(f"rep={rep} (no tracks)")
            ax.axis("off")
            continue

        # Each TrackID gets its own polyline + points colored by MCS
        tids = trdf["TrackID"].dropna().unique()
        cols = color_cycle(len(tids))
        track_handles = []
        for tid, col in zip(tids, cols):
            g = trdf[trdf["TrackID"] == tid].sort_values("MCS")

            # line per track
            ln, = ax.plot(g["centroid_x_fix"], g["centroid_y_fix"],
                          linewidth=LINE_WIDTH, color=col, label=f"Cluster {int(tid)}")
            track_handles.append(ln)

            # dots colored by MCS (shared norm/cmap)
            sc = ax.scatter(g["centroid_x_fix"], g["centroid_y_fix"],
                            s=MARKER_SIZE, c=g["MCS"], cmap=CMAP_NAME, norm=norm)
            scatter_for_cbar = sc

            # start/end markers
            ax.plot(g["centroid_x_fix"].iloc[0], g["centroid_y_fix"].iloc[0],
                    marker="o", markersize=7, markerfacecolor="white", markeredgecolor=col)
            ax.plot(g["centroid_x_fix"].iloc[-1], g["centroid_y_fix"].iloc[-1],
                    marker="x", markersize=8, markeredgecolor=col)

            # Optional: label per-frame rank if available
            if LABEL_FRAME_ORDER and "Cluster ID" in g.columns:
                for _, r in g.iterrows():
                    cid = r.get("Cluster ID")
                    if pd.notna(cid):
                        ax.text(r["centroid_x_fix"], r["centroid_y_fix"],
                                f"{int(cid)}", fontsize=7, ha="center", va="center",
                                color="black", alpha=0.7)

        # ----- Event annotations (stars) -----
        ev_handles = []
        legend_labels = []

        # Event "at this MCS"
        if "HasEventHere" in trdf.columns and trdf["HasEventHere"].any():
            ev_here = trdf[trdf["HasEventHere"] == True]
            ax.scatter(ev_here["centroid_x_fix"], ev_here["centroid_y_fix"],
                    marker="*", s=180, facecolors="none", edgecolors="k", linewidths=1.5)
            ev_handles.append(Line2D([0], [0], marker="*", linestyle="None",
                                    markerfacecolor="none", markeredgecolor="k", markersize=12))

            # Collect event type(s) for legend
            if "EventHereType" in ev_here.columns:
                types = sorted(set(str(t).upper() for t in ev_here["EventHereType"] if pd.notna(t)))
                if types:
                    legend_labels.append(f"Event here (★, {', '.join(types)})")
                else:
                    legend_labels.append("Event here (★)")
            else:
                legend_labels.append("Event here (★)")

            # Optional: annotate compact type labels on the plot
            if "EventHereType" in ev_here.columns:
                for _, r in ev_here.iterrows():
                    et = str(r.get("EventHereType", "")).strip()
                    if et and et.lower() != "nan":
                        short = {"MERGE": "M", "SPLIT": "S", "DISSOLVE": "D"}.get(et.upper(), et[:1].upper())
                        ax.text(r["centroid_x_fix"], r["centroid_y_fix"],
                                short, fontsize=8, fontweight="bold", ha="center", va="center", color="k")

        # Event "at next MCS"
        if "HasEventNext" in trdf.columns and trdf["HasEventNext"].any():
            ev_next = trdf[trdf["HasEventNext"] == True]
            ax.scatter(ev_next["centroid_x_fix"], ev_next["centroid_y_fix"],
                    marker="*", s=120, facecolors="none", edgecolors="gray", linewidths=1.2)
            ev_handles.append(Line2D([0], [0], marker="*", linestyle="None",
                                    markerfacecolor="none", markeredgecolor="gray", markersize=10))

            # Collect event type(s) for legend
            if "EventNextType" in ev_next.columns:
                types = sorted(set(str(t).upper() for t in ev_next["EventNextType"] if pd.notna(t)))
                if types:
                    legend_labels.append(f"Event next (★, {', '.join(types)})")
                else:
                    legend_labels.append("Event next (★)")
            else:
                legend_labels.append("Event next (★)")

            # Optional: annotate compact type labels on the plot
            if "EventNextType" in ev_next.columns:
                for _, r in ev_next.iterrows():
                    et = str(r.get("EventNextType", "")).strip()
                    if et and et.lower() != "nan":
                        short = {"MERGE": "M", "SPLIT": "S", "DISSOLVE": "D"}.get(et.upper(), et[:1].upper())
                        ax.text(r["centroid_x_fix"], r["centroid_y_fix"],
                                short, fontsize=8, ha="center", va="bottom", color="gray")


        # Titles, axes, legends
        ax.set_title(f"rep={int(rep)}")
        ax.set_xlabel("Centroid_X")
        ax.set_ylabel("Centroid_Y")
        if SHOW_EQUAL_ASPECT:
            ax.set_aspect("equal", "box")
        ax.grid(True, alpha=0.3)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        # Build a combined legend: tracks + events
        handles = track_handles + ev_handles
        labels  = [h.get_label() for h in track_handles] + legend_labels
        if handles:
            ax.legend(handles=handles, labels=labels, loc="upper left",
                      bbox_to_anchor=(1.02, 1.0), frameon=False, borderaxespad=0.0, ncol=1)

    # turn off unused axes
    for k in range(len(reps), len(flat_axes)):
        flat_axes[k].axis("off")

    # Super title
    fig.suptitle(f"Cluster Trajectories — Jlf={jlf}, μ={mu}, PP={pp}", fontsize=16, y=1.02)
    fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.04, hspace=0.10, wspace=0.10)

    # Shared colorbar for MCS
    cbar = fig.colorbar(scatter_for_cbar if scatter_for_cbar is not None else ScalarMappable(norm=norm, cmap=CMAP_NAME),
                        ax=axes, orientation="vertical", fraction=0.03, pad=0.02)
    cbar.set_label("MCS")

    
    prefix = f"{idx:04d}_" if idx is not None else ""
    out_path = out_dir / f"{prefix}tracks_J{safe_name(jlf)}_mu{safe_name(mu)}_PP{safe_name(pp)}.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return str(out_path)

# =============================
# MAIN
# =============================
def main():
    # Load composition data (enriched)
    comp = pd.read_csv(COMPOSITION_CSV)

    # Ensure required columns
    need = ["MCS","Jlf","mu","PP","rep","Cluster UID","Centroid_X","Centroid_Y"]
    missing = [c for c in need if c not in comp.columns]
    if missing:
        raise ValueError(f"Missing required columns in {COMPOSITION_CSV}: {missing}")

    # Coerce numerics and drop rows missing essentials
    for c in ["MCS","Jlf","mu","PP","rep","Cluster UID","Centroid_X","Centroid_Y"]:
        comp[c] = pd.to_numeric(comp[c], errors="coerce")
    comp = comp.dropna(subset=["MCS","Jlf","mu","PP","rep","Cluster UID"]).copy()

    # Read parameter list or fall back to all unique triples in comp
    plist = read_param_list_or_fallback(comp, PARAM_LIST_CSV)
    if plist.empty:
        print("No parameter combinations to plot.")
        return

    plist = plist.sort_values(["Jlf", "mu", "PP"]).reset_index(drop=True)
    
    
    outputs = []
    for idx, pr in enumerate(plist.itertuples(index=False), start=1):
        jlf, mu, pp = float(pr.Jlf), float(pr.mu), float(pr.PP)
        figpath = plot_param_all_reps(comp, jlf, mu, pp, OUT_DIR, idx=idx)
        if figpath:
            outputs.append(figpath)

    print(f"\nDone. Figures in: {OUT_DIR.resolve()}")
    if not outputs:
        print("(No figures produced — check that selected params exist in the composition file.)")

if __name__ == "__main__":
    main()



### Interactive plots


In [15]:
import os, math, io
from ast import literal_eval
from pathlib import Path
from typing import List, Iterable, Optional, Tuple, Dict

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from mpl_toolkits.axes_grid1 import make_axes_locatable  # for colorbar axis
from matplotlib import animation

from PIL import Image

# imageio is optional; if unavailable we'll try Matplotlib-FFmpeg, else PNG sequence + ffmpeg cmd
try:
    import imageio.v2 as imageio
    HAS_IMAGEIO = True
except Exception:
    imageio = None
    HAS_IMAGEIO = False

import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------
# Matplotlib settings
# --------------------------------
plt.ioff()
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

# =============================
# CONFIG
# =============================
COMPOSITION_CSV = "../../Results/Data/Clusters/ClusterComposition_1.csv"
SAVE_DIR        = "../../Results/Data/Clusters/Trajectory"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

APP_TITLE       = "Cluster Trajectories (TrackID & UID side-by-side)"

# Fixed scale per selection so full trajectory stays visible while animating
MIN_X_SPAN = 50.0
MIN_Y_SPAN = 5.0
PAD_FRAC   = 0.02

# Tracker thresholds (only needed for TrackID view’s linker)
JACCARD_MIN       = 0.15
CENTROID_MAX_DIST = 10.0

# Styling
LINE_WIDTH   = 2
DOT_SIZE     = 28      # points^2
START_SIZE   = 180
END_SIZE     = 220     # mapped to markersize for ax.plot
TEXT_FONT_SZ = 9

# Legend placement (outside, after colorbar)
LEGEND_ANCHOR_X = 1.20
LEGEND_ANCHOR_Y = 0.5

# ===== Font knobs =====
AXIS_LABEL_FONTSIZE       = 14
AXIS_LABEL_FONTWEIGHT     = "bold"

TICK_LABEL_FONTSIZE       = 12
TICK_LABEL_FONTWEIGHT     = "bold"

COLORBAR_LABEL_FONTSIZE   = 13
COLORBAR_LABEL_FONTWEIGHT = "bold"
COLORBAR_TICK_FONTSIZE    = 11
COLORBAR_TICK_FONTWEIGHT  = "bold"

LEGEND_FONTSIZE           = 12
LEGEND_FONTWEIGHT         = "bold"

DPI_EXPORT = 300  # for PNG/PDF/TIF

# =============================
# Custom legend handler for dotted "X"
# =============================
class HandlerDottedCross(HandlerBase):
    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        x0 = width / 2
        y0 = height / 2
        dx = width / 2
        dy = height / 2
        line1 = plt.Line2D([x0 - dx, x0 + dx],
                           [y0 - dy, y0 + dy],
                           linestyle=":", color=orig_handle.get_color(),
                           linewidth=orig_handle.get_linewidth(), transform=trans)
        line2 = plt.Line2D([x0 - dx, x0 + dx],
                           [y0 + dy, y0 - dy],
                           linestyle=":", color=orig_handle.get_color(),
                           linewidth=orig_handle.get_linewidth(), transform=trans)
        return [line1, line2]

# Dummy handle to carry style for legend entry
dotted_cross_handle = Line2D([], [], color="black", lw=2, label="End (700 MCS)")

# =============================
# Helpers
# =============================
def parse_list(s):
    if pd.isna(s):
        return []
    try:
        v = literal_eval(str(s))
        if isinstance(v, (list, tuple)):
            return [float(x) for x in v]
    except Exception:
        pass
    return []

def row_points(row, round_dec=1):
    lx = parse_list(row.get("Member_Leader_X"))
    ly = parse_list(row.get("Member_Leader_Y"))
    fx = parse_list(row.get("Member_Follower_X"))
    fy = parse_list(row.get("Member_Follower_Y"))
    pts = [(x, y) for x, y in zip(lx, ly)] + [(x, y) for x, y in zip(fx, fy)]
    if not pts:
        return np.empty((0, 2), dtype=float)
    pts = np.asarray(pts, dtype=float)
    return np.round(pts, round_dec) if round_dec is not None else pts

def jaccard_from_points(A, B):
    if A.size == 0 or B.size == 0:
        return 0.0
    sA, sB = set(map(tuple, A)), set(map(tuple, B))
    inter, uni = len(sA & sB), len(sA | sB)
    return 0.0 if uni == 0 else inter / uni

def centroid_from_points(pts):
    if pts.size == 0:
        return np.nan, np.nan
    return float(np.mean(pts[:, 0])), float(np.mean(pts[:, 1]))

def color_cycle(n):
    cmaps = ["tab10", "tab20", "tab20b", "tab20c", "Set3", "Accent", "Dark2", "Paired"]
    cols, idx = [], 0
    while len(cols) < n:
        cmap = plt.get_cmap(cmaps[idx % len(cmaps)])
        steps = 20 if "tab20" in cmap.name else 10
        for i in range(steps):
            if len(cols) >= n:
                break
            cols.append(cmap(i / max(1, steps - 1)))
        idx += 1
    return cols[:n]

def _sorted_unique(series):
    vals = list({float(x) for x in series.dropna().tolist()})
    vals.sort()
    return vals

def _sanitize(s: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-=" else "_" for ch in str(s))

def _ffmpeg_available() -> bool:
    try:
        return animation.writers.is_available("ffmpeg")
    except Exception:
        return False

# -------- SUPER-ROBUST dropdown setter ----------
def _set_dropdown(drop: widgets.Dropdown, options, prefer_keep=True):
    prev = drop.value
    values = [v for (_, v) in options]
    try:
        drop.value = None
    except Exception:
        pass
    drop.options = options
    if prefer_keep and (prev in values):
        drop.value = prev
    else:
        drop.value = (values[0] if values else None)
# ------------------------------------------------

# Safely set slider+play ranges in order (max → min → step → value)
def _set_range_widgets(slider: widgets.IntSlider, play: widgets.Play,
                       vmin: int, vmax: int, step: int, start_value: int = None):
    if vmin > vmax:
        vmin, vmax = vmax, vmin
    val = vmin if start_value is None else max(vmin, min(start_value, vmax))

    slider.unobserve(_on_slider_change, names="value")
    try:
        slider.max = int(vmax)
        slider.min = int(vmin)
        slider.step = int(step)
        slider.value = int(val)

        play.max = int(vmax)
        play.min = int(vmin)
        play.step = int(step)
        play.value = int(val)
    finally:
        slider.observe(_on_slider_change, names="value")

# =============================
# TrackID trajectory builder
# =============================
def build_tracks_df(df, jlf, mu, pp, rep,
                    jaccard_min=JACCARD_MIN, centroid_max_dist=CENTROID_MAX_DIST):
    sub = df[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp) & (df["rep"] == rep)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["MCS","Cluster ID","TrackID","centroid_x_fix","centroid_y_fix","TotalCells"])
    sub = sub.sort_values(["MCS", "Cluster ID"]).reset_index(drop=True)

    time_points = sorted(sub["MCS"].unique())
    track_id_counter = 1
    active_tracks = []  # [{'track': int, 'points': Nx2, 'centroid': (x,y)}]
    assignments = []

    for t in time_points:
        cur = sub[sub["MCS"] == t].copy()
        cur["pts"] = cur.apply(row_points, axis=1)
        cur["centroid_from_pts"] = cur["pts"].apply(centroid_from_points)

        unmatched = set(cur.index.tolist())
        used_tracks = set()

        # 1) Jaccard match
        for ci in list(unmatched):
            row_pts = cur.at[ci, "pts"]
            best_track, best_j = None, -1.0
            for tr in active_tracks:
                if tr["track"] in used_tracks:
                    continue
                j = jaccard_from_points(row_pts, tr["points"])
                if j > best_j:
                    best_j, best_track = j, tr
            if best_track is not None and best_j >= jaccard_min:
                cx, cy = cur.at[ci, "centroid_from_pts"]
                if np.isnan(cx) or np.isnan(cy):
                    cx, cy = float(cur.at[ci, "Centroid_X"]), float(cur.at[ci, "Centroid_Y"])
                assignments.append({
                    "MCS": t,
                    "Cluster ID": cur.at[ci, "Cluster ID"],
                    "TrackID": best_track["track"],
                    "centroid_x_fix": cx,
                    "centroid_y_fix": cy,
                    "TotalCells": float(cur.at[ci, "Total Cells"]) if "Total Cells" in cur.columns else np.nan
                })
                best_track["points"] = row_pts
                best_track["centroid"] = (cx, cy)
                unmatched.remove(ci)
                used_tracks.add(best_track["track"])

        # 2) Centroid distance match
        for ci in list(unmatched):
            cx, cy = cur.at[ci, "centroid_from_pts"]
            if np.isnan(cx) or np.isnan(cy):
                cx, cy = float(cur.at[ci, "Centroid_X"]), float(cur.at[ci, "Centroid_Y"])
            best_track, best_dist = None, float("inf")
            for tr in active_tracks:
                if tr["track"] in used_tracks:
                    continue
                tx, ty = tr["centroid"]
                if tx is None or np.isnan(tx) or ty is None or np.isnan(ty):
                    continue
                d = math.hypot(cx - tx, cy - ty)
                if d < best_dist:
                    best_dist, best_track = d, tr
            if best_track is not None and best_dist <= centroid_max_dist:
                assignments.append({
                    "MCS": t,
                    "Cluster ID": cur.at[ci, "Cluster ID"],
                    "TrackID": best_track["track"],
                    "centroid_x_fix": cx,
                    "centroid_y_fix": cy,
                    "TotalCells": float(cur.at[ci, "Total Cells"]) if "Total Cells" in cur.columns else np.nan
                })
                best_track["points"] = cur.at[ci, "pts"]
                best_track["centroid"] = (cx, cy)
                unmatched.remove(ci)
                used_tracks.add(best_track["track"])

        # 3) New tracks
        for ci in list(unmatched):
            cx, cy = cur.at[ci, "centroid_from_pts"]
            if np.isnan(cx) or np.isnan(cy):
                cx, cy = float(cur.at[ci, "Centroid_X"]), float(cur.at[ci, "Centroid_Y"])
            tr = {"track": track_id_counter, "points": cur.at[ci, "pts"], "centroid": (cx, cy)}
            active_tracks.append(tr)
            assignments.append({
                "MCS": t,
                "Cluster ID": cur.at[ci, "Cluster ID"],
                "TrackID": track_id_counter,
                "centroid_x_fix": cx,
                "centroid_y_fix": cy,
                "TotalCells": float(cur.at[ci, "Total Cells"]) if "Total Cells" in cur.columns else np.nan
            })
            track_id_counter += 1

    tracks_df = pd.DataFrame(assignments).sort_values(["TrackID","MCS"]).reset_index(drop=True)
    return tracks_df

# =============================
# UID trajectory builder
# =============================
def build_tracks_df_uid(df, jlf, mu, pp, rep):
    sub = df[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp) & (df["rep"] == rep)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["MCS", "Cluster UID", "centroid_x_fix", "centroid_y_fix", "TotalCells"])

    sub["Cluster UID"] = sub["Cluster UID"].astype(str)

    pts = sub.apply(row_points, axis=1)
    cents = pts.apply(centroid_from_points)
    sub["centroid_x_fix"] = [cx if not np.isnan(cx) else float(x) for (cx, _), x in zip(cents, sub["Centroid_X"])]
    sub["centroid_y_fix"] = [cy if not np.isnan(cy) else float(y) for (_, cy), y in zip(cents, sub["Centroid_Y"])]

    out_cols = ["MCS", "Cluster UID", "centroid_x_fix", "centroid_y_fix", "Total Cells"]
    tracks = sub[out_cols].rename(columns={"Total Cells": "TotalCells"}).copy()
    tracks = tracks.sort_values(["Cluster UID", "MCS"]).reset_index(drop=True)
    return tracks

# =============================
# UID ↔ TrackID color mapping helper
# =============================
def _uid_to_tid_map_for_params(df, jlf, mu, pp, rep, tracks_df):
    sub = df[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp) & (df["rep"] == rep)].copy()
    if sub.empty or tracks_df is None or tracks_df.empty:
        return {}

    sub = sub[["MCS", "Cluster ID", "Cluster UID"]].copy()
    sub["Cluster UID"] = sub["Cluster UID"].astype(str)

    merged = pd.merge(
        tracks_df[["MCS", "Cluster ID", "TrackID"]],
        sub,
        on=["MCS", "Cluster ID"],
        how="left"
    )

    uid_to_tid = {}
    for uid, g in merged.dropna(subset=["Cluster UID"]).groupby("Cluster UID"):
        mode_vals = g["TrackID"].mode()
        if not mode_vals.empty:
            uid_to_tid[uid] = int(mode_vals.iloc[0])
    return uid_to_tid

# =============================
# Load data once
# =============================
assert os.path.exists(COMPOSITION_CSV), f"Missing: {COMPOSITION_CSV}"
df = pd.read_csv(COMPOSITION_CSV)

need_tracks = ["MCS","Jlf","mu","PP","rep","Cluster ID",
               "Centroid_X","Centroid_Y",
               "Member_Leader_X","Member_Leader_Y",
               "Member_Follower_X","Member_Follower_Y",
               "Total Cells"]
need_uid    = ["MCS","Jlf","mu","PP","rep","Cluster UID",
               "Centroid_X","Centroid_Y",
               "Member_Leader_X","Member_Leader_Y",
               "Member_Follower_X","Member_Follower_Y",
               "Total Cells"]

missing_tracks = [c for c in need_tracks if c not in df.columns]
missing_uid    = [c for c in need_uid    if c not in df.columns]
if missing_tracks:
    raise ValueError(f"Missing columns for TrackID view in {COMPOSITION_CSV}: {missing_tracks}")
if missing_uid:
    raise ValueError(f"Missing columns for UID view in {COMPOSITION_CSV}: {missing_uid}")

for c in ["MCS","Jlf","mu","PP","rep","Cluster ID","Centroid_X","Centroid_Y","Total Cells"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "Cluster UID" in df.columns:
    df["Cluster UID"] = df["Cluster UID"].astype(str)

df = df.dropna(subset=["MCS","Jlf","mu","PP","rep"]).copy()

# =============================
# Widgets (shared)
# =============================
jlf_options = _sorted_unique(df["Jlf"])
dd_jlf = widgets.Dropdown(
    options=[(f"{v:g}", v) for v in jlf_options],
    value=(jlf_options[0] if jlf_options else None),
    description="Jlf:",
    layout=widgets.Layout(width="180px"),
)
dd_mu  = widgets.Dropdown(description="μ:", layout=widgets.Layout(width="180px"))
dd_pp  = widgets.Dropdown(description="PP:", layout=widgets.Layout(width="180px"))
dd_rep = widgets.Dropdown(description="rep:", layout=widgets.Layout(width="140px"))

# time slider + play
play       = widgets.Play(interval=300, value=0, min=0, max=1, step=1)
slider     = widgets.IntSlider(value=0, min=0, max=1, step=1, description="MCS", readout=True)
widgets.jslink((play, 'value'), (slider, 'value'))

# Save buttons + status
lbl_status        = widgets.HTML(value="")
btn_save          = widgets.Button(description="Save PNG/PDF/TIF", icon="save", layout=widgets.Layout(width="150px"))
btn_save_all      = widgets.Button(description="Save ALL MCS", icon="save", layout=widgets.Layout(width="140px"))
btn_video_mp4_tr  = widgets.Button(description="MP4 (TrackID)", icon="film", layout=widgets.Layout(width="130px"))
btn_video_mp4_uid = widgets.Button(description="MP4 (UID)",     icon="film", layout=widgets.Layout(width="130px"))
bar_progress      = widgets.IntProgress(value=0, min=0, max=100, bar_style='',
                                        orientation='horizontal',
                                        layout=widgets.Layout(width="260px", visibility="hidden"))
lbl_batch         = widgets.HTML(value="")

# Two outputs (side-by-side canvases)
out_tracks = widgets.Output()   # left: TrackID view
out_uid    = widgets.Output()   # right: UID view

# =============================
# Figure states
# =============================
_state_tracks = dict(
    fig=None, ax=None, cbar=None,
    jlf=None, mu=None, pp=None, rep=None,
    trdf=None, mcs_vals=[], per_tid={}, tids=[],
    colors=[],
    tid_color_map={}   # TrackID -> color
)

_state_uid = dict(
    fig=None, ax=None, cbar=None,
    jlf=None, mu=None, pp=None, rep=None,
    trdf=None, mcs_vals=[], per_uid={}, uids=[],
    colors=[]
)

# =============================
# Limits helper
# =============================
def _compute_limits(trdf):
    xmin = float(trdf["centroid_x_fix"].min()); xmax = float(trdf["centroid_x_fix"].max())
    ymin = float(trdf["centroid_y_fix"].min()); ymax = float(trdf["centroid_y_fix"].max())
    dx = max(xmax - xmin, 0.0); dy = max(ymax - ymin, 0.0)
    eff_x_span = max(MIN_X_SPAN, dx * (1.0 + PAD_FRAC))
    eff_y_span = max(MIN_Y_SPAN, dy * (1.0 + PAD_FRAC))
    x_center = 0.5 * (xmax + xmin) if dx > 0 else xmin
    y_center = 0.5 * (ymax + ymin) if dy > 0 else ymin
    return (x_center - eff_x_span / 2.0, x_center + eff_x_span / 2.0), \
           (y_center - eff_y_span / 2.0, y_center + eff_y_span / 2.0)

# =============================
# INIT + DRAW: TrackID view
# =============================
def _init_tracks():
    with out_tracks:
        clear_output(wait=True)
        if _state_tracks["fig"] is not None:
            try: plt.close(_state_tracks["fig"])
            except: pass

        fig, ax = plt.subplots(figsize=(9.2, 7.6))
        _state_tracks.update(fig=fig, ax=ax, cbar=None)

        jlf, mu, pp, rep = dd_jlf.value, dd_mu.value, dd_pp.value, dd_rep.value
        if None in (jlf, mu, pp, rep):
            ax.set_axis_off(); display(fig); return

        trdf = build_tracks_df(df, jlf, mu, pp, rep)
        if trdf.empty:
            ax.set_axis_off()
            display(fig)
            return

        mcs_vals = sorted(trdf["MCS"].unique())
        tids     = list(trdf["TrackID"].unique())
        colors   = color_cycle(len(tids))
        per_tid  = {tid: trdf[trdf["TrackID"] == tid].sort_values("MCS") for tid in tids}

        _state_tracks.update(jlf=jlf, mu=mu, pp=pp, rep=rep,
                             trdf=trdf, mcs_vals=mcs_vals, per_tid=per_tid,
                             tids=tids, colors=colors)

        _state_tracks["tid_color_map"] = {tid: col for tid, col in zip(tids, colors)}

        (x0, x1), (y0, y1) = _compute_limits(trdf)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_xlabel("Centroid X", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_FONTWEIGHT)
        ax.set_ylabel("Centroid Y", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_FONTWEIGHT)
        ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight(TICK_LABEL_FONTWEIGHT)

        norm = Normalize(vmin=float(mcs_vals[0]), vmax=float(mcs_vals[-1]))
        sm = ScalarMappable(norm=norm, cmap="viridis")
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4.5%", pad=0.12)
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label("Time (MCS)", fontsize=COLORBAR_LABEL_FONTSIZE, fontweight=COLORBAR_LABEL_FONTWEIGHT)
        cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)
        for t in cbar.ax.get_yticklabels():
            t.set_fontweight(COLORBAR_TICK_FONTWEIGHT)
        _state_tracks["cbar"] = cbar

        legend_elems = [Line2D([0],[0], color=col, lw=LINE_WIDTH, label=f"Cluster {tid}")
                        for tid, col in zip(tids, colors)]
        leg1 = ax.legend(
            handles=legend_elems,
            loc="center left",
            bbox_to_anchor=(LEGEND_ANCHOR_X, LEGEND_ANCHOR_Y),
            borderaxespad=0.0,
            frameon=True,
            prop={"size": LEGEND_FONTSIZE, "weight": LEGEND_FONTWEIGHT},
        )

        marker_handles = [
            Line2D([0], [0], marker='o', color='black', markersize=8,
                   markerfacecolor='none', lw=0, label="Start"),
            Line2D([0], [0], marker='x', color='black', markersize=8,
                   linestyle='None', lw=0, label="End"),
            dotted_cross_handle
        ]
        leg2 = ax.legend(
            handles=marker_handles,
            loc="center left",
            bbox_to_anchor=(LEGEND_ANCHOR_X, LEGEND_ANCHOR_Y - 0.45),
            frameon=True,
            handler_map={dotted_cross_handle: HandlerDottedCross()},
            prop={"size": LEGEND_FONTSIZE, "weight": LEGEND_FONTWEIGHT},
            title_fontsize=LEGEND_FONTSIZE,
        )
        ax.add_artist(leg1)

        _set_range_widgets(slider, play,
                           vmin=int(mcs_vals[0]),
                           vmax=int(mcs_vals[-1]),
                           step=10,
                           start_value=int(mcs_vals[0]))

        _draw_tracks(slider.value)
        display(fig)

def _draw_tracks(mcs_k: int):
    ax = _state_tracks["ax"]; fig = _state_tracks["fig"]
    if ax is None or fig is None:
        return

    # Clear artists (not legends/colorbar)
    for art in list(ax.collections):
        try:
            if getattr(art, 'axes', None) is ax:
                art.remove()
        except Exception: pass
    for line in list(ax.lines):
        try: line.remove()
        except Exception: pass
    for txt in list(ax.texts):
        try: txt.remove()
        except Exception: pass

    mcs_vals = _state_tracks["mcs_vals"]
    if not mcs_vals:
        return
    mcs_k = min(mcs_vals, key=lambda v: abs(v - mcs_k))

    mmin, mmax = float(mcs_vals[0]), float(mcs_vals[-1])
    norm = Normalize(vmin=mmin, vmax=mmax)
    cmap = plt.get_cmap("viridis")

    for tid, col in zip(_state_tracks["tids"], _state_tracks["colors"]):
        g  = _state_tracks["per_tid"][tid]
        gk = g[g["MCS"] <= mcs_k]
        if gk.empty:
            continue

        xs = gk["centroid_x_fix"].to_numpy()
        ys = gk["centroid_y_fix"].to_numpy()
        mcs = gk["MCS"].to_numpy().astype(float)

        # path
        ax.plot(xs, ys, '-', lw=LINE_WIDTH, color=col, alpha=0.9)
        # dots colored by MCS
        colors = cmap(norm(mcs))
        ax.scatter(xs, ys, s=DOT_SIZE, c=colors, edgecolors='none')
        # bold labels
        if "TotalCells" in gk.columns:
            for xi, yi, tc in zip(xs, ys, gk["TotalCells"].tolist()):
                if not (isinstance(tc, float) and math.isnan(tc)) and tc is not None:
                    ax.text(xi, yi + 0.5, f"{int(tc)}",
                            ha="center", va="bottom",
                            fontsize=TEXT_FONT_SZ, fontweight="bold")

        # start marker (open circle)
        ax.scatter([xs[0]], [ys[0]], s=START_SIZE, facecolors='none',
                   edgecolors=col, linewidths=1.8)

        # end marker: dotted X if last MCS==700 else solid 'x'
        end_x, end_y = xs[-1], ys[-1]
        end_mcs = float(mcs[-1])
        if np.isclose(end_mcs, 700.0, atol=1e-6):
            frac = 0.02
            xspan = ax.get_xlim()[1] - ax.get_xlim()[0]
            yspan = ax.get_ylim()[1] - ax.get_ylim()[0]
            dx = frac * xspan
            dy = frac * yspan
            ax.plot([end_x - dx, end_x + dx], [end_y - dy, end_y + dy],
                    linestyle=":", color=col, linewidth=2.0)
            ax.plot([end_x - dx, end_x + dx], [end_y + dy, end_y - dy],
                    linestyle=":", color=col, linewidth=2.0)
        else:
            ax.plot([end_x], [end_y], marker='x', linestyle='None',
                    color=col, markersize=max(6, int(math.sqrt(END_SIZE))),
                    markeredgewidth=2.0)

    ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight(TICK_LABEL_FONTWEIGHT)
    fig.canvas.draw_idle()

# =============================
# INIT + DRAW: UID view
# =============================
def _init_uid():
    with out_uid:
        clear_output(wait=True)
        if _state_uid["fig"] is not None:
            try: plt.close(_state_uid["fig"])
            except: pass

        fig, ax = plt.subplots(figsize=(9.2, 7.6))
        _state_uid.update(fig=fig, ax=ax, cbar=None)

        jlf, mu, pp, rep = dd_jlf.value, dd_mu.value, dd_pp.value, dd_rep.value
        if None in (jlf, mu, pp, rep):
            ax.set_axis_off(); display(fig); return

        trdf = build_tracks_df_uid(df, jlf, mu, pp, rep)
        if trdf.empty:
            ax.set_axis_off()
            display(fig); return

        mcs_vals = sorted(trdf["MCS"].unique())
        uids     = list(trdf["Cluster UID"].unique())
        per_uid  = {uid: trdf[trdf["Cluster UID"] == uid].sort_values("MCS") for uid in uids}

        uid_to_tid = _uid_to_tid_map_for_params(df, jlf, mu, pp, rep, _state_tracks["trdf"])
        tid_color_map = _state_tracks.get("tid_color_map", {})

        uid_colors = [None] * len(uids)
        used_colors = set()
        for i, uid in enumerate(uids):
            tid = uid_to_tid.get(str(uid))
            if tid in tid_color_map:
                uid_colors[i] = tid_color_map[tid]
                used_colors.add(tid_color_map[tid])

        palette   = color_cycle(len(uids) + len(used_colors))
        remaining = [c for c in palette if c not in used_colors]
        rem_iter  = iter(remaining)
        for i, col in enumerate(uid_colors):
            if col is None:
                try:
                    uid_colors[i] = next(rem_iter)
                except StopIteration:
                    uid_colors[i] = color_cycle(1)[0]

        _state_uid.update(jlf=jlf, mu=mu, pp=pp, rep=rep,
                          trdf=trdf, mcs_vals=mcs_vals, per_uid=per_uid,
                          uids=uids, colors=uid_colors)

        (x0, x1), (y0, y1) = _compute_limits(trdf)
        ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
        ax.set_xlabel("Centroid X", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_FONTWEIGHT)
        ax.set_ylabel("Centroid Y", fontsize=AXIS_LABEL_FONTSIZE, fontweight=AXIS_LABEL_FONTWEIGHT)
        ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_fontweight(TICK_LABEL_FONTWEIGHT)

        norm = Normalize(vmin=float(mcs_vals[0]), vmax=float(mcs_vals[-1]))
        sm = ScalarMappable(norm=norm, cmap="viridis")
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="4.5%", pad=0.12)
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label("Time (MCS)", fontsize=COLORBAR_LABEL_FONTSIZE, fontweight=COLORBAR_LABEL_FONTWEIGHT)
        cbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)
        for t in cbar.ax.get_yticklabels():
            t.set_fontweight(COLORBAR_TICK_FONTWEIGHT)
        _state_uid["cbar"] = cbar

        legend_elems = [Line2D([0],[0], color=col, lw=LINE_WIDTH, label=f"UID {uid}")
                        for uid, col in zip(uids, uid_colors)]
        leg1 = ax.legend(
            handles=legend_elems,
            loc="center left",
            bbox_to_anchor=(LEGEND_ANCHOR_X, LEGEND_ANCHOR_Y),
            borderaxespad=0.0,
            frameon=True,
            prop={"size": LEGEND_FONTSIZE, "weight": LEGEND_FONTWEIGHT},
        )

        marker_handles = [
            Line2D([0], [0], marker='o', color='black', markersize=8,
                   markerfacecolor='none', lw=0, label="Start"),
            Line2D([0], [0], marker='x', color='black', markersize=8,
                   linestyle='None', lw=0, label="End"),
            dotted_cross_handle
        ]
        leg2 = ax.legend(
            handles=marker_handles,
            loc="center left",
            bbox_to_anchor=(LEGEND_ANCHOR_X, LEGEND_ANCHOR_Y - 0.45),
            frameon=True,
            handler_map={dotted_cross_handle: HandlerDottedCross()},
            prop={"size": LEGEND_FONTSIZE, "weight": LEGEND_FONTWEIGHT},
            title_fontsize=LEGEND_FONTSIZE,
        )
        ax.add_artist(leg1)

        _set_range_widgets(slider, play,
                           vmin=int(mcs_vals[0]),
                           vmax=int(mcs_vals[-1]),
                           step=10,
                           start_value=int(mcs_vals[0]))
        _draw_uid(slider.value)
        display(fig)

def _draw_uid(mcs_k: int):
    ax = _state_uid["ax"]; fig = _state_uid["fig"]
    if ax is None or fig is None:
        return

    for art in list(ax.collections):
        try:
            if getattr(art, 'axes', None) is ax:
                art.remove()
        except Exception: pass
    for line in list(ax.lines):
        try: line.remove()
        except Exception: pass
    for txt in list(ax.texts):
        try: txt.remove()
        except Exception: pass

    mcs_vals = _state_uid["mcs_vals"]
    if not mcs_vals:
        return
    mcs_k = min(mcs_vals, key=lambda v: abs(v - mcs_k))

    mmin, mmax = float(mcs_vals[0]), float(mcs_vals[-1])
    norm = Normalize(vmin=mmin, vmax=mmax)
    cmap = plt.get_cmap("viridis")

    for uid, col in zip(_state_uid["uids"], _state_uid["colors"]):
        g  = _state_uid["per_uid"][uid]
        gk = g[g["MCS"] <= mcs_k]
        if gk.empty:
            continue

        xs = gk["centroid_x_fix"].to_numpy()
        ys = gk["centroid_y_fix"].to_numpy()
        mcs = gk["MCS"].to_numpy().astype(float)

        ax.plot(xs, ys, '-', lw=LINE_WIDTH, color=col, alpha=0.9)
        colors = cmap(norm(mcs))
        ax.scatter(xs, ys, s=DOT_SIZE, c=colors, edgecolors='none')
        if "TotalCells" in gk.columns:
            for xi, yi, tc in zip(xs, ys, gk["TotalCells"].tolist()):
                if not (isinstance(tc, float) and math.isnan(tc)) and tc is not None:
                    ax.text(xi, yi + 0.5, f"{int(tc)}",
                            ha="center", va="bottom",
                            fontsize=TEXT_FONT_SZ, fontweight="bold")

        ax.scatter([xs[0]], [ys[0]], s=START_SIZE, facecolors='none',
                   edgecolors=col, linewidths=1.8)

        end_x, end_y = xs[-1], ys[-1]
        end_mcs = float(mcs[-1])
        if np.isclose(end_mcs, 700.0, atol=1e-6):
            frac = 0.02
            xspan = ax.get_xlim()[1] - ax.get_xlim()[0]
            yspan = ax.get_ylim()[1] - ax.get_ylim()[0]
            dx = frac * xspan
            dy = frac * yspan
            ax.plot([end_x - dx, end_x + dx], [end_y - dy, end_y + dy],
                    linestyle=":", color=col, linewidth=2.0)
            ax.plot([end_x - dx, end_x + dx], [end_y + dy, end_y - dy],
                    linestyle=":", color=col, linewidth=2.0)
        else:
            ax.plot([end_x], [end_y], marker='x', linestyle='None',
                    color=col, markersize=max(6, int(math.sqrt(END_SIZE))),
                    markeredgewidth=2.0)

    ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight(TICK_LABEL_FONTWEIGHT)
    fig.canvas.draw_idle()

# =============================
# Export helpers
# =============================
def _save_all_formats(fig: plt.Figure, base_path_no_ext: str, dpi: int = DPI_EXPORT):
    """Save fig as PNG, PDF, and TIF (LZW). base_path_no_ext like '/path/to/name'."""
    png = f"{base_path_no_ext}.png"
    pdf = f"{base_path_no_ext}.pdf"
    tif = f"{base_path_no_ext}.tif"

    fig.savefig(png, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf, dpi=dpi, bbox_inches="tight")

    # robust TIF LZW: render to PNG buffer then Pillow->TIFF
    buf = io.BytesIO()
    fig.savefig(buf, dpi=dpi, bbox_inches="tight", format="png", facecolor="white")
    buf.seek(0)
    im = Image.open(buf).convert("RGB")
    im.save(tif, compression="tiff_lzw", dpi=(dpi, dpi))
    buf.close()
    return png, pdf, tif

def _fig_to_rgb(fig: plt.Figure) -> np.ndarray:
    """Return an (H,W,3) uint8 array of the entire figure (including legends/colorbars)."""
    fig.canvas.draw()
    # Try buffer_rgba first
    try:
        rgba = np.asarray(fig.canvas.buffer_rgba())
        if rgba.ndim == 3 and rgba.shape[2] == 4:
            return rgba[:, :, :3].copy()
    except Exception:
        pass
    # Try tostring_rgb
    try:
        w, h = fig.canvas.get_width_height()
        buf = fig.canvas.tostring_rgb()
        return np.frombuffer(buf, dtype=np.uint8).reshape(h, w, 3)
    except Exception:
        pass
    # Try print_to_buffer (Agg backends)
    try:
        try:
            buf, (w, h) = fig.canvas.print_to_buffer()
        except Exception:
            (w, h), buf = fig.canvas.print_to_buffer()
        rgba = np.frombuffer(buf, dtype=np.uint8).reshape(h, w, 4)
        return rgba[:, :, :3].copy()
    except Exception as e:
        raise RuntimeError("Could not extract RGB buffer from the Matplotlib figure.") from e

def _encode_video_from_frames(frames, out_path, fps=4):
    """Try imageio MP4 first, then Matplotlib-FFmpeg, else PNG sequence + ffmpeg cmd (also writes a GIF)."""
    # A) imageio (preferred)
    if HAS_IMAGEIO:
        try:
            imageio.mimsave(out_path, frames, fps=fps, codec="libx264", quality=8)
            return f"Saved MP4 to <code>{out_path}</code> (imageio)"
        except Exception:
            try:
                imageio.mimsave(out_path, frames, fps=fps)  # generic fallback
                return f"Saved MP4 to <code>{out_path}</code> (imageio generic)"
            except Exception:
                pass

    # B) Matplotlib FFmpeg writer
    if _ffmpeg_available():
        try:
            fig_tmp, ax_tmp = plt.subplots(figsize=(9.2, 7.6))
            ax_tmp.axis("off")
            im_artist = ax_tmp.imshow(frames[0])

            Writer = animation.writers["ffmpeg"]
            writer = Writer(fps=fps, metadata=dict(artist="trajectory"), codec="libx264", bitrate=-1)
            with writer.saving(fig_tmp, out_path, dpi=150):
                for arr in frames:
                    im_artist.set_data(arr)
                    writer.grab_frame()
            plt.close(fig_tmp)
            return f"Saved MP4 to <code>{out_path}</code> (Matplotlib FFmpeg writer)"
        except Exception:
            pass

    # C) No MP4 backend: dump PNGs + advise ffmpeg; also create a GIF so you still get a video artifact
    seq_dir = os.path.splitext(out_path)[0] + "_frames"
    os.makedirs(seq_dir, exist_ok=True)
    for i, fr in enumerate(frames):
        Image.fromarray(fr).save(os.path.join(seq_dir, f"frame_{i:04d}.png"))

    gif_path = os.path.splitext(out_path)[0] + ".gif"
    try:
        if HAS_IMAGEIO:
            imageio.mimsave(gif_path, frames, duration=0.25)  # ~4 fps
        else:
            pil_frames = [Image.fromarray(fr) for fr in frames]
            pil_frames[0].save(gif_path, save_all=True, append_images=pil_frames[1:],
                               optimize=True, duration=250, loop=0)
    except Exception:
        pass

    ff_cmd = (
        f"ffmpeg -y -r {fps} -i '{os.path.join(seq_dir, 'frame_%04d.png')}' "
        f"-vf 'format=yuv420p' -vcodec libx264 -pix_fmt yuv420p '{out_path}'"
    )
    return (
        "No MP4 encoder available. Saved PNG frames and a GIF.\n"
        f"Frames: <code>{seq_dir}</code><br>"
        f"GIF: <code>{gif_path}</code><br>"
        f"To make MP4 locally, run:<br><code>{ff_cmd}</code>"
    )

# =============================
# Shared control wiring
# =============================
def _refresh_both(*_):
    _init_tracks()
    _init_uid()
    if _state_tracks["mcs_vals"]:
        x0,x1 = _state_tracks["ax"].get_xlim()
        y0,y1 = _state_tracks["ax"].get_ylim()
        lbl_status.value = (f"{len(_state_tracks['tids'])} clusters (TrackID) • "
                            f"{len(_state_uid['uids']) if _state_uid['uids'] else 0} clusters (UID) • "
                            f"x=[{x0:.2f},{x1:.2f}] y=[{y0:.2f},{y1:.2f}]")

def _on_slider_change(change):
    if change["name"] == "value":
        with out_tracks:
            clear_output(wait=True)
            _draw_tracks(change["new"])
            display(_state_tracks["fig"])
        with out_uid:
            clear_output(wait=True)
            _draw_uid(change["new"])
            display(_state_uid["fig"])

# ---------- Save current MCS (PNG/PDF/TIF LZW for both panes) ----------
def _on_save_clicked(_btn):
    if _state_tracks["fig"] is None or _state_uid["fig"] is None:
        lbl_status.value = "<b>Nothing to save:</b> pick parameters first."
        return

    jlf, mu, pp, rep = dd_jlf.value, dd_mu.value, dd_pp.value, dd_rep.value
    mcs = slider.value

    base1 = os.path.join(SAVE_DIR, _sanitize(f"Traj_TrackID_Jlf{jlf:g}_mu{mu:g}_PP{pp:g}_rep{rep:g}_MCS{int(mcs)}"))
    base2 = os.path.join(SAVE_DIR, _sanitize(f"Traj_UID_Jlf{jlf:g}_mu{mu:g}_PP{pp:g}_rep{rep:g}_MCS{int(mcs)}"))

    p1 = _save_all_formats(_state_tracks["fig"], base1)
    p2 = _save_all_formats(_state_uid["fig"], base2)

    lbl_status.value = f"Saved:<br><code>{p1}</code><br><code>{p2}</code>"

# ---------- Save ALL MCS (PNG/PDF/TIF LZW for both panes) ----------
def _on_save_all_clicked(_btn):
    if (_state_tracks["fig"] is None or not _state_tracks["mcs_vals"] or
        _state_uid["fig"]    is None or not _state_uid["mcs_vals"]):
        lbl_status.value = "<b>Nothing to save:</b> pick parameters first."
        return

    jlf, mu, pp, rep = dd_jlf.value, dd_mu.value, dd_pp.value, dd_rep.value
    mcs_vals = list(_state_tracks["mcs_vals"])

    subdir = _sanitize(f"Traj_Jlf{jlf:g}_mu{mu:g}_PP{pp:g}_rep{rep:g}_BOTH")
    target_dir = os.path.join(SAVE_DIR, subdir)
    os.makedirs(target_dir, exist_ok=True)

    bar_progress.min = 0
    bar_progress.max = len(mcs_vals)
    bar_progress.value = 0
    bar_progress.layout.visibility = "visible"
    lbl_batch.value  = f"Saving {len(mcs_vals)} frames to <code>{target_dir}</code> ..."

    for k, m in enumerate(mcs_vals, 1):
        _draw_tracks(int(m))
        _draw_uid(int(m))

        base1 = os.path.join(target_dir, _sanitize(f"{subdir}_TrackID_MCS{int(m)}"))
        base2 = os.path.join(target_dir, _sanitize(f"{subdir}_UID_MCS{int(m)}"))
        _save_all_formats(_state_tracks["fig"], base1)
        _save_all_formats(_state_uid["fig"], base2)

        bar_progress.value = k

    bar_progress.layout.visibility = "hidden"
    lbl_batch.value  = f"Done: saved {2*len(mcs_vals)} images (PNG/PDF/TIF for each) in <code>{target_dir}</code>"
    lbl_status.value = lbl_batch.value

# ---------- Single-pane video (MP4), legends included ----------
def _save_video_single(which="track", fps=4):
    """
    which: 'track' or 'uid'
    Produces an MP4 for the chosen pane, including its legend/colorbar.
    """
    if which.lower() == "track":
        state = _state_tracks
        drawer = _draw_tracks
        tag = "TrackID"
    elif which.lower() == "uid":
        state = _state_uid
        drawer = _draw_uid
        tag = "UID"
    else:
        lbl_status.value = "<b>Unknown pane:</b> choose 'track' or 'uid'."
        return

    if state["fig"] is None or not state["mcs_vals"]:
        lbl_status.value = f"<b>Nothing to save:</b> open {tag} first."
        return

    jlf, mu, pp, rep = dd_jlf.value, dd_mu.value, dd_pp.value, dd_rep.value
    out_base = _sanitize(f"Traj_{tag}_Jlf{jlf:g}_mu{mu:g}_PP{pp:g}_rep{rep:g}_anim")
    out_path = os.path.join(SAVE_DIR, f"{out_base}.mp4")

    mcs_vals = list(state["mcs_vals"])
    frames = []

    bar_progress.min = 0
    bar_progress.max = len(mcs_vals)
    bar_progress.value = 0
    bar_progress.layout.visibility = "visible"
    lbl_batch.value  = f"Rendering {tag} frames..."

    for k, m in enumerate(mcs_vals, 1):
        drawer(int(m))
        state["fig"].canvas.draw()          # legend + colorbar are baked in
        frame = _fig_to_rgb(state["fig"])   # full figure pixels
        frames.append(frame.astype(np.uint8))
        bar_progress.value = k

    bar_progress.layout.visibility = "hidden"
    lbl_batch.value = "Encoding MP4..."
    msg = _encode_video_from_frames(frames, out_path, fps=fps)
    lbl_batch.value  = msg
    lbl_status.value = msg

def _on_save_video_mp4_track(_btn):
    _save_video_single(which="track", fps=4)

def _on_save_video_mp4_uid(_btn):
    _save_video_single(which="uid", fps=4)

btn_save.on_click(_on_save_clicked)
btn_save_all.on_click(_on_save_all_clicked)
btn_video_mp4_tr.on_click(_on_save_video_mp4_track)
btn_video_mp4_uid.on_click(_on_save_video_mp4_uid)

# =============================
# Cascaded dropdown updates
# =============================
def _update_mu(*_):
    jlf = dd_jlf.value
    mus = _sorted_unique(df.loc[df["Jlf"] == jlf, "mu"])
    _set_dropdown(dd_mu, [(f"{v:g}", v) for v in mus], prefer_keep=False)
    _update_pp()

def _update_pp(*_):
    jlf, mu = dd_jlf.value, dd_mu.value
    if jlf is None or mu is None:
        _set_dropdown(dd_pp, [], prefer_keep=False)
        _set_dropdown(dd_rep, [], prefer_keep=False)
        return
    pps = _sorted_unique(df.loc[(df["Jlf"] == jlf) & (df["mu"] == mu), "PP"])
    _set_dropdown(dd_pp, [(f"{v:g}", v) for v in pps], prefer_keep=False)
    _update_rep()

def _update_rep(*_):
    jlf, mu, pp = dd_jlf.value, dd_mu.value, dd_pp.value
    if None in (jlf, mu, pp):
        _set_dropdown(dd_rep, [], prefer_keep=False)
        return
    reps = _sorted_unique(df.loc[(df["Jlf"] == jlf) & (df["mu"] == mu) & (df["PP"] == pp), "rep"])
    labels = [(str(int(r)) if float(r).is_integer() else f"{r:g}", r) for r in reps]
    _set_dropdown(dd_rep, labels, prefer_keep=False)
    _refresh_both()  # rebuild both panes on rep change too

def _on_rep_change(change):
    if change["name"] == "value":
        _refresh_both()

# Hook observers
dd_jlf.observe(_update_mu, names="value")
dd_mu.observe(_update_pp, names="value")
dd_pp.observe(_update_rep, names="value")
dd_rep.observe(_on_rep_change, names="value")
slider.observe(_on_slider_change, names="value")

# Seed options and first render
_update_mu()

# =============================
# UI layout
# =============================
top = widgets.HTML(f"<h3 style='margin:4px 0'>{APP_TITLE}</h3>")
controls = widgets.HBox([
    dd_jlf, dd_mu, dd_pp, dd_rep,
    widgets.HBox([play, slider], layout=widgets.Layout(margin='0 0 0 16px')),
    btn_save, btn_save_all,
    btn_video_mp4_tr, btn_video_mp4_uid,
    bar_progress
])

display(top, controls)
display(widgets.HBox([out_tracks, out_uid], layout=widgets.Layout(gap="16px")))
display(lbl_status, lbl_batch)


HTML(value="<h3 style='margin:4px 0'>Cluster Trajectories (TrackID & UID side-by-side)</h3>")

HTML(value='1 clusters (TrackID) • 1 clusters (UID) • x=[118.27,168.27] y=[70.14,75.86]')

HTML(value='')